# Stage 30 — Fair CPU hyperparameter search and architecture transfer

**Research goal:** compare two promising hard-2-bit optimizers against a stronger, independently tuned **Muon-STE**, using **full training loss** and **CPU end-to-end step time**. Do not assume a discrete method is better until it wins held-out tests.

**Candidate shortlist:** `joint8` (common discrete/codebook proximal objective, one main LR) and `fast8` (split codebook-gradient objective, separate codebook LR). Both maintain 2-bit forward weights and an 8-bit-per-weight persistent residual; no FP32 shadow weights in these methods. Muon-STE uses FP32 master weights and momentum with the same quantized forward codebooks.

**What changed from Stage 28:**

- Genuine **logarithmic coarse learning-rate sweep** (0.003–0.7, optionally 1.3) **per optimizer and task**, then a **fine LR neighborhood and one-factor sensitivity search** across other material parameters. Muon is given *at least as many* refinement trials as each discrete candidate.
- Select by **development full-training-loss AUC**, using a longer refinement horizon and two development seeds; freeze configurations before evaluation on untouched seeds 901+.
- More architectures and loss families: MSE, pseudo-Huber, smoothed quantile, Poisson NLL, BCE, multiclass CE, transformer next-token CE, and optional focal loss / pairwise ranking / RNN multiclass.
- Row-parallel **Numba CPU kernels**, tested against the Stage 28 serial implementation. NumPy arrays are views over actual Torch persistent buffers—no per-step FP32 shadow weights.
- **Matched codebook bounds** for Muon and discrete arms; optional controlled full re-run with outer bound 8 instead of 2. Bound saturation diagnostics help determine whether it can matter.
- Resumable deterministic run JSON/checkpoints, stage-by-stage search CSVs, full loss-versus-updates/time plots, time-to-shared-threshold, and automatic ZIP download. A session-time cap stops at a method boundary, saves partial results, and allows continuing from the same output directory.

**Scientific limitations:** This is a CPU FP32 reference for Muon, not official GPU Muon; INT2 symbols are stored as uint8 (logical INT2 storage would be smaller); no guarantee of global nonlinear-loss descent. One-factor refinement is broader than Stage 28 but is not a complete factorial search. Training speed should be evaluated on the actual Colab CPU.


## The two discrete objectives

Both use actual deployed weights \(W=Q(K,a,b)\) with row codewords \((-a,-b,b,a)\) and residual grid \(\mathcal E_8\). Define a normalized first-order target \(T=W+\beta E-\eta_t a\operatorname{clip}(G/(r v),-4,4)\).

**Joint INT8:** approximately block-coordinate minimize a common surrogate

\[
\min_{K',a',b',E'}\; \frac12\sum_{ij}v_j(Q_{ij}(K',a',b')+E'_{ij}-T_{ij})^2
+\frac{\lambda}{2}\sum_{ij}v_j(E'_{ij})^2+\frac{\gamma n}{2}\sum_i((a'_i-a_i)^2+(b'_i-b_i)^2).
\]

**Split INT8:** choose legal indices/residuals using the target-fitting term, but update row codebooks from an independent first-order training-gradient proximal objective. Its second codebook learning rate is `code_eta`. Both methods use exact **conditional** finite/rowwise solves, not a global optimum guarantee.

Both candidate solvers use the same 8-bit residual precision so this run tests **objective/solver differences**, not an arbitrary precision change. A zero-residual method is *not* promoted because prior tests showed large convergence degradation. Benchmarks record persistent optimizer state and temporary training-step timing separately.


In [ ]:
# CPU-only environment. In Colab, Runtime -> Change runtime type -> CPU.
from pathlib import Path
import os, sys, json, importlib, subprocess, time
os.environ.setdefault('STAGE27_THREADS','1')         # PyTorch CPU thread count
os.environ.setdefault('STAGE30_NUMBA_THREADS','2')   # independent matrix rows
for module,package in [('numpy','numpy'),('pandas','pandas'),('numba','numba'),('matplotlib','matplotlib'),('torch','torch')]:
    try:importlib.import_module(module)
    except ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import numpy as np, pandas as pd, torch
Path('stage30_src').mkdir(exist_ok=True)
print('PyTorch',torch.__version__,'CPU threads',os.environ['STAGE27_THREADS'],'Numba threads',os.environ['STAGE30_NUMBA_THREADS'])


PyTorch 2.14.1+cpu CPU threads 1 Numba threads 2


## Embedded source (self-contained, no checkout required)

The next cells write the historical common model/data modules and the two new Stage 30 CPU kernels and search harness into `stage30_src/`.


In [ ]:
%%writefile stage30_src/discrete_optim.py
"""Stage 16: fully trainable 2-bit discrete optimizer prototypes.

The proposed solvers have NO FP32 master weights. An autograd FP32 realized
weight tensor is recreated on every forward, rather than maintained as state.
Each row has two learned positive ordered magnitudes {-a,-b,b,a}.

Finite grid subproblem at fixed codebooks:
 min_(k') sum_j [g_j(q(k')-w_j) + .5*h_j(q(k')-w_j)^2],
 subject to up to s index transitions per row.
A separable gain + top-s is an *exact global solution* to this CONDITIONAL
surrogate. Magnitudes are then re-fit to the same surrogate (a 2D QP).
Alternating steps do not jointly solve it globally. No global descent guarantee.
"""
import math,time
import torch
from torch import nn
import torch.nn.functional as F

def decode(k,a,b):
    av=a[:,None]; bv=b[:,None]
    return torch.where(k==0,-av,torch.where(k==1,-bv,torch.where(k==2,bv,av)))

def quantize_shadow(w,a,b):
    t=((a+b)/2)[:,None]
    k=(w>=-t).to(torch.uint8)+(w>=0).to(torch.uint8)+(w>=t).to(torch.uint8)
    return decode(k,a,b),k

class DiscreteLinear(nn.Module):
    def __init__(self,init_k,a,b,device='cpu'):
        super().__init__()
        self.register_buffer('k',init_k.clone().to(device=device,dtype=torch.uint8))
        self.register_buffer('a',a.clone().to(device))
        self.register_buffer('b',b.clone().to(device))
        self._w=None;self._input=None;self._outgrad=None
    def forward(self,x):
        self._input=x.detach()
        w=decode(self.k,self.a,self.b).detach().requires_grad_(True)
        self._w=w
        return F.linear(x,w)

class STELinear(nn.Module):
    def __init__(self,init_k,a,b,device='cpu'):
        super().__init__()
        W=decode(init_k,a,b)
        self.shadow=nn.Parameter(W.clone().to(device))
        self.a=nn.Parameter(a.clone().to(device))
        self.b=nn.Parameter(b.clone().to(device))
    def forward(self,x):
        q,k=quantize_shadow(self.shadow,self.a,self.b)
        w=self.shadow+(q-self.shadow).detach()+(q-q.detach())
        return F.linear(x,w)
    @torch.no_grad()
    def clamp_codebook(self):
        self.a.clamp_(min=.006,max=2.0)
        self.b.clamp_(min=.002,max=1.95)
        self.a.copy_(torch.maximum(self.a,self.b+.004))
        self.a.clamp_(max=2.0)
        self.b.copy_(torch.minimum(self.b,self.a-.004))

class QuantMLP(nn.Module):
    def __init__(self,sizes,seed=0,method='direct',device='cpu',activation='tanh'):
        super().__init__();self.sizes=sizes;self.method=method;self.activation=activation
        g=torch.Generator().manual_seed(seed)
        ls=[]
        for di,do in zip(sizes[:-1],sizes[1:]):
            # Equal initialization for all methods (including initial realized weights)
            k=torch.randint(0,4,(do,di),generator=g,dtype=torch.uint8)
            a=torch.full((do,),min(.7,1.2/math.sqrt(di)),dtype=torch.float32)
            b=a/3
            ls.append((DiscreteLinear if method=='direct' else STELinear)(k,a,b,device=device))
        self.layers=nn.ModuleList(ls)
    def forward(self,x):
        for i,layer in enumerate(self.layers):
            x=layer(x)
            if i<len(self.layers)-1:
                x=torch.tanh(x) if self.activation=='tanh' else F.relu(x)
        return x
    def direct_layers(self):return list(self.layers)
    def ste_groups(self):
        return [m.shadow for m in self.layers],[p for m in self.layers for p in (m.a,m.b)]

@torch.no_grad()
def constrained_two_level_fit(ua,ub,Ha,Hb,old_a,old_b,gap=.004,amin=.006,amax=2.0,bmin=.002):
    """Exact weighted 2D projection onto amin<=a<=amax,bmin<=b<=a-gap.
    Enumerates the free interior and all four possible active faces, including
    vertices through face clipping. This is a convex 2D quadratic program.
    """
    wa=Ha.clamp(min=1e-12);wb=Hb.clamp(min=1e-12)
    ua=torch.where(Ha>1e-12,ua,old_a);ub=torch.where(Hb>1e-12,ub,old_b)
    # Feasible interior, otherwise give it an infinite score.
    feasible=(ua>=amin)&(ua<=amax)&(ub>=bmin)&(ub<=ua-gap)
    cands=[]
    cands.append((ua,ub,feasible))
    aa=torch.full_like(ua,amin);bb=ub.clamp(min=bmin,max=amin-gap)
    cands.append((aa,bb,torch.ones_like(feasible)))
    aa=torch.full_like(ua,amax);bb=ub.clamp(min=bmin,max=amax-gap)
    cands.append((aa,bb,torch.ones_like(feasible)))
    bb=torch.full_like(ub,bmin);aa=ua.clamp(min=max(amin,bmin+gap),max=amax)
    cands.append((aa,bb,torch.ones_like(feasible)))
    bb=((wa*(ua-gap)+wb*ub)/(wa+wb)).clamp(min=max(bmin,amin-gap),max=amax-gap)
    aa=bb+gap
    cands.append((aa,bb,torch.ones_like(feasible)))
    scores=torch.stack([torch.where(valid,.5*wa*(aa-ua).square()+.5*wb*(bb-ub).square(),torch.full_like(ua,float('inf'))) for aa,bb,valid in cands])
    choose=scores.argmin(0)
    aa_out=torch.stack([c[0] for c in cands]).gather(0,choose[None,:]).squeeze(0)
    bb_out=torch.stack([c[1] for c in cands]).gather(0,choose[None,:]).squeeze(0)
    return aa_out,bb_out

class DiscreteSurrogateOptimizer:
    """Three mathematical objectives share one conditional exact discrete solver:

    'budget': uniform proximal metric with rowwise sparse exchange budget.
    'diagonal': self-scaled diagonal metric (adaptive per feature).
    'full': uniform metric, full unrestricted grid transitions (no budget).

    Hyperparameter lr (= inverse metric strength), optional budget_frac fixed .15;
    row scale profile step is exact conditional weighted least squares.
    """
    def __init__(self,model,lr=1.0,kind='budget',budget_frac=.15,codebook=True,metric_floor=1e-9):
        self.model=model;self.lr=lr;self.kind=kind;self.budget_frac=budget_frac
        self.codebook=codebook;self.metric_floor=metric_floor
        self.transitions=0;self.calls=0
        self.ema_beta=.9
        self.state={}
        for layer in self.model.direct_layers():
            if kind in ('mprox','dualavg','feedback'):
                self.state[layer]={'buffer':torch.zeros_like(decode(layer.k,layer.a,layer.b))}
                if kind=='dualavg':
                    self.state[layer]['anchor']=decode(layer.k,layer.a,layer.b).detach().clone()
                    self.state[layer]['fixed_metric']=None
    @torch.no_grad()
    def step(self):
        self.calls+=1
        self.transitions=0
        for layer in self.model.direct_layers():
            w=layer._w;g=w.grad
            if g is None:continue
            k=layer.k; a=layer.a;b=layer.b
            old=w.detach()
            center=old
            if self.kind=='mprox':
                st=self.state[layer]
                st['buffer'].mul_(self.ema_beta).add_(g,alpha=1-self.ema_beta)
                g=st['buffer']/(1-self.ema_beta**self.calls)
            elif self.kind=='dualavg':
                st=self.state[layer]
                if st['fixed_metric'] is None:
                    st['fixed_metric']=g.square().mean(dim=1,keepdim=True).sqrt().clamp(min=1e-7)/a[:,None].clamp(min=.005)
                st['buffer'].add_(g)
                center=st['anchor']
                g=st['buffer']
            elif self.kind=='feedback':
                center=old+self.state[layer]['buffer']
            # Scale-invariant proximal curvature in units of the local grid size.
            # gamma has units |gradient|/|weight|. The optional diagonal is
            # row_rms(g) * feature_energy_j / (mean_energy * a_i * lr).
            rms=g.square().mean(dim=1,keepdim=True).sqrt().clamp(min=self.metric_floor)
            h=rms/(a[:,None].clamp(min=.005)*self.lr)
            if self.kind=='dualavg':h=self.state[layer]['fixed_metric']*math.sqrt(self.calls)/self.lr
            if self.kind=='diagonal':
                xx=layer._input.reshape(-1,layer._input.shape[-1])
                ex=xx.detach().square().mean(dim=0,keepdim=True).clamp(min=1e-8)
                h=h*(ex/ex.mean()).clamp(min=.1,max=10.)
            h=h.expand_as(g)
            levels=torch.stack((-a,-b,b,a),dim=-1)
            delta=levels[:,:,None]-center[:,None,:]
            cost=g[:,None,:]*delta+.5*h[:,None,:]*delta.square()
            vals,idx=cost.min(dim=1)
            idx=idx.to(torch.uint8)
            # Compare with the CURRENT CODEWORD cost, not zero: for dual
            # averaging and residual-corrected centers, keeping the current
            # symbol is not a zero-cost action. This makes the fixed-codebook
            # conditional integer minimization exact.
            current_cost=cost.gather(1,k.long()[:,None,:]).squeeze(1)
            # Nonnegative switching penalty (hysteresis): if the incumbent
            # surrogate cost at the corrected center is positive, require a
            # proposed flip to overcome it. This is a fixed-at-step penalty
            # lambda_ij=max(0,c_ij(k_t)); the finite-cardinality subproblem
            # remains exactly solvable coordinatewise / via top-k selection.
            switch_penalty=current_cost.clamp(min=0) if self.kind in ('feedback','dualavg') else torch.zeros_like(current_cost)
            benefit=current_cost-vals-switch_penalty
            improve=(benefit>1e-13)&(idx!=k)
            if self.kind in ('budget','diagonal'):
                count=max(1,math.ceil(self.budget_frac*k.shape[1]))
                gain=torch.where(improve,benefit,torch.zeros_like(vals))
                top=torch.topk(gain,k=min(count,k.shape[1]),dim=1).indices
                allow=torch.zeros_like(improve).scatter_(1,top,True)
                improve &=allow
            k_new=torch.where(improve,idx,k)
            self.transitions+=int(improve.sum().item())
            # Conditional variable projection: each magnitude has a disjoint
            # signed indicator and an exact 1D quadratic minimizer. If ordering
            # fails use a weighted isotonic projection onto a>=b+gap.
            if self.codebook:
                sg_a=torch.where(k_new==0,-1.,torch.where(k_new==3,1.,0.))
                sg_b=torch.where(k_new==1,-1.,torch.where(k_new==2,1.,0.))
                target=center-g/h
                Ha=(h*sg_a.square()).sum(1)
                Hb=(h*sg_b.square()).sum(1)
                Sa=(h*sg_a*target).sum(1)
                Sb=(h*sg_b*target).sum(1)
                ua=Sa/Ha.clamp(min=1e-12)
                ub=Sb/Hb.clamp(min=1e-12)
                na,nb=constrained_two_level_fit(ua,ub,Ha,Hb,a,b)
                layer.a.copy_(na);layer.b.copy_(nb)
            layer.k.copy_(k_new)
            if self.kind=='feedback':
                realized=decode(layer.k,layer.a,layer.b)
                residual=(target-realized).clamp(min=-2*a[:,None],max=2*a[:,None])
                self.state[layer]['buffer'].copy_(residual)
    def state_bytes(self):
        stored=sum(m.k.numel()+8*m.a.numel() for m in self.model.direct_layers())
        return stored+sum(v.numel()*v.element_size() for d in self.state.values() for v in d.values() if torch.is_tensor(v))

class BaselineOptimizer:
    def __init__(self,model,method='adam',lr=.003,code_lr=.003,weight_decay=0.,momentum=.95,muon_adjust='match_rms_adamw',head_lr_factor=1.0,beta1=.9,beta2=.999):
        from pathlib import Path
        mat,scales=model.ste_groups()
        param_groups=[{'params':mat[:-1],'lr':lr},{'params':[mat[-1]],'lr':lr*head_lr_factor}]
        if method=='adam':self.opt=torch.optim.AdamW(param_groups,weight_decay=weight_decay,betas=(beta1,beta2))
        elif method=='muon':
            # CPU BF16 NS can be slow; float32 NS is recorded in metadata.
            # Official Muon is used on GPU if supported; CPU uses an exact
            # polynomial FP32 reproduction, not the official BF16 mode.
            if mat[0].device.type=='cpu':
                from muon_cpu import CPUFloat32Muon
                self.opt=CPUFloat32Muon(param_groups,momentum=momentum,weight_decay=weight_decay,adjust_lr_fn=muon_adjust)
            else:
                self.opt=torch.optim.Muon(param_groups,momentum=momentum,weight_decay=weight_decay,adjust_lr_fn=muon_adjust)
        else:raise ValueError(method)
        self.sc=torch.optim.Adam(scales,lr=code_lr)
        self.model=model
    def zero_grad(self):self.opt.zero_grad(set_to_none=True);self.sc.zero_grad(set_to_none=True)
    def step(self):
        self.opt.step();self.sc.step()
        for layer in self.model.layers:layer.clamp_codebook()
    def state_bytes(self):
        return sum(p.numel()*4 for p in self.model.parameters())+sum(sum(v.numel()*v.element_size() for v in state.values() if torch.is_tensor(v)) for opt in (self.opt,self.sc) for state in opt.state.values())


Overwriting stage30_src/discrete_optim.py


In [ ]:
%%writefile stage30_src/muon_cpu.py
"""FP32 CPU Muon reference following PyTorch's momentum recurrence; GPU uses official torch.optim.Muon."""
import math
import torch
class CPUFloat32Muon(torch.optim.Optimizer):
    def __init__(self,params,lr=.02,momentum=.95,nesterov=True,weight_decay=0.,adjust_lr_fn='original',ns_steps=5,eps=1e-7):
        super().__init__(params,dict(lr=lr,momentum=momentum,nesterov=nesterov,weight_decay=weight_decay,adjust_lr_fn=adjust_lr_fn,ns_steps=ns_steps,eps=eps))
    @torch.no_grad()
    def step(self,closure=None):
        loss=closure() if closure is not None else None
        for group in self.param_groups:
            lr=group['lr'];mu=group['momentum'];wd=group['weight_decay']
            for p in group['params']:
                if p.grad is None:continue
                if p.ndim!=2:raise ValueError('Muon supports 2D matrix parameters')
                g=p.grad;st=self.state[p]
                if 'momentum_buffer' not in st:st['momentum_buffer']=torch.zeros_like(g)
                buf=st['momentum_buffer']
                buf.lerp_(g,1-mu) # PyTorch official recurrence
                update=torch.lerp(g,buf,mu) if group['nesterov'] else buf
                x=update.float().clone()
                transpose=x.shape[0]>x.shape[1]
                if transpose:x=x.T
                x=x/x.norm().clamp(min=group['eps'])
                a,b,c=3.4445,-4.775,2.0315
                for _ in range(group['ns_steps']):
                    gram=x@x.T
                    x=a*x+(b*gram+c*(gram@gram))@x
                if transpose:x=x.T
                A,B=p.shape;mode=group['adjust_lr_fn']
                if mode in ('original',None):factor=math.sqrt(max(1,A/B))
                elif mode=='match_rms_adamw':factor=.2*math.sqrt(max(A,B))
                elif mode=='spectral_unclamped':factor=math.sqrt(A/B)
                else:raise ValueError(mode)
                p.mul_(1-lr*wd).add_(x.to(p.dtype),alpha=-lr*factor)
        return loss


Overwriting stage30_src/muon_cpu.py


In [ ]:
%%writefile stage30_src/advanced_models.py
"""Fully trainable residual MLP and CNN, all matrix/conv weights legal 2-bit.
Fused/packed INT2 operations are NOT implemented; realized temporary FP32
weights are constructed for PyTorch autograd and thus dominate peak memory.
"""
import math,torch
from torch import nn
from torch.nn import functional as F
from discrete_optim import QuantMLP,DiscreteLinear,STELinear,decode,quantize_shadow

def _initial(do,di,g,device='cpu'):
    k=torch.randint(0,4,(do,di),dtype=torch.uint8,generator=g)
    a=torch.full((do,),min(.7,1.2/math.sqrt(di)),dtype=torch.float32)
    b=a/3
    return k,a,b

class ResidualQuantMLP(nn.Module):
    def __init__(self,input_dim=64,width=128,depth=3,out=4,seed=1,method='direct',device='cpu'):
        super().__init__();self.base=QuantMLP([input_dim]+[width]*depth+[out],seed=seed,method=method,device=device)
        self.layers=self.base.layers
    def forward(self,x):
        h=torch.tanh(self.layers[0](x))
        for l in self.layers[1:-1]:
            h=(h+0.7*torch.tanh(l(h)))/math.sqrt(1+0.7**2)
        return self.layers[-1](h)
    def direct_layers(self):return list(self.layers)
    def ste_groups(self):return self.base.ste_groups()

class DiscreteConv(nn.Module):
    def __init__(self,cin,cout,kernel=3,stride=1,seed=1,device='cpu'):
        super().__init__();g=torch.Generator().manual_seed(seed)
        k,a,b=_initial(cout,cin*kernel*kernel,g)
        self.inner=DiscreteLinear(k,a,b,device=device)
        self.k=self.inner.k;self.a=self.inner.a;self.b=self.inner.b
        self.cin=cin;self.cout=cout;self.kernel=kernel;self.stride=stride
        self._w=None;self._input=None
    def forward(self,x):
        self._input=x.detach()
        # The 2D leaf receives gradients via the convolution's 4D view.
        w=decode(self.k,self.a,self.b).detach().requires_grad_(True)
        self._w=w
        return F.conv2d(x,w.reshape(self.cout,self.cin,self.kernel,self.kernel),stride=self.stride,padding=self.kernel//2)

class STEConv(nn.Module):
    def __init__(self,cin,cout,kernel=3,stride=1,seed=1,device='cpu'):
        super().__init__();g=torch.Generator().manual_seed(seed)
        k,a,b=_initial(cout,cin*kernel*kernel,g)
        self.inner=STELinear(k,a,b,device=device)
        self.shadow=self.inner.shadow;self.a=self.inner.a;self.b=self.inner.b
        self.cin=cin;self.cout=cout;self.kernel=kernel;self.stride=stride
    def forward(self,x):
        q,_=quantize_shadow(self.shadow,self.a,self.b)
        w=self.shadow+(q-self.shadow).detach()+(q-q.detach())
        return F.conv2d(x,w.reshape(self.cout,self.cin,self.kernel,self.kernel),stride=self.stride,padding=self.kernel//2)
    def clamp_codebook(self):return self.inner.clamp_codebook()

class QuantCNN(nn.Module):
    def __init__(self,channels=1,num_classes=4,width=16,seed=0,method='direct',device='cpu'):
        super().__init__()
        direct=(method=='direct')
        Conv=DiscreteConv if direct else STEConv
        g=torch.Generator().manual_seed(seed)
        d1=width;d2=width*2
        self.conv1=Conv(channels,d1,3,1,seed+1,device)
        self.conv2=Conv(d1,d2,3,2,seed+2,device)
        self.conv3=Conv(d2,d2,3,2,seed+3,device)
        k,a,b=_initial(num_classes,d2,g)
        self.head=(DiscreteLinear if direct else STELinear)(k,a,b,device=device)
        self.layers=nn.ModuleList([self.conv1,self.conv2,self.conv3,self.head])
    def forward(self,x):
        x=F.relu(self.conv1(x))
        x=F.relu(self.conv2(x))
        x=F.relu(self.conv3(x))
        x=x.mean((-2,-1))
        return self.head(x)
    def direct_layers(self):return list(self.layers)
    def ste_groups(self):
        return [layer.shadow for layer in self.layers],[p for layer in self.layers for p in (layer.a,layer.b)]


Overwriting stage30_src/advanced_models.py


In [ ]:
%%writefile stage30_src/rnn_arch.py
"""Fully quantized recurrent benchmark; reuses one weight leaf across timesteps.
RNN is intentionally a simple tanh-RNN so that all hidden/input/recurrent matrices are 2-bit.
"""
import math,torch
from torch import nn
from torch.nn import functional as F
from discrete_optim import DiscreteLinear,STELinear,decode,quantize_shadow

class QuantRNN(nn.Module):
 def __init__(self,d=8,h=64,out=4,seed=0,method='direct',device='cpu'):
  super().__init__();gen=torch.Generator().manual_seed(seed)
  L=DiscreteLinear if method=='direct' else STELinear
  self.layers=nn.ModuleList()
  for di,do in ((d,h),(h,h),(h,out)):
   k=torch.randint(0,4,(do,di),dtype=torch.uint8,generator=gen)
   a=torch.full((do,),min(.7,1.2/math.sqrt(di)))
   self.layers.append(L(k,a,a/3,device=device))
  self.direct=method=='direct'
 def _weight(self,layer):
  if self.direct:
   w=decode(layer.k,layer.a,layer.b).detach().requires_grad_(True)
   layer._w=w;return w
  q,_=quantize_shadow(layer.shadow,layer.a,layer.b)
  return layer.shadow+(q-layer.shadow).detach()+(q-q.detach())
 def forward(self,x):
  wx,wh,wo=[self._weight(l) for l in self.layers]
  h=torch.zeros(x.shape[0],wh.shape[0],device=x.device)
  for t in range(x.shape[1]):
   h=torch.tanh(F.linear(x[:,t],wx)+F.linear(h,wh))
  return F.linear(h,wo)
 def direct_layers(self):return list(self.layers)
 def ste_groups(self):return [m.shadow for m in self.layers],[p for m in self.layers for p in (m.a,m.b)]

def sequence_data(seed,task='robust',n=2000,T=14,dim=8,out=4):
 gen=torch.Generator().manual_seed(seed+50401)
 X=torch.randn(n,T,dim,generator=gen)
 a=torch.randn(dim,32,generator=gen)/math.sqrt(dim)
 b=torch.randn(32,32,generator=gen)/math.sqrt(32)
 c=torch.randn(32,out,generator=gen)/math.sqrt(32)
 h=torch.zeros(n,32)
 for t in range(T):h=torch.tanh(X[:,t]@a+h@b)
 z=(h@c)*3+.4*torch.sin(X[:,-1,:out])
 if task=='bce':Y=torch.bernoulli(torch.sigmoid(z),generator=gen)
 elif task=='multiclass':
  Y=torch.multinomial(F.softmax(z,dim=-1),1,generator=gen).squeeze(-1)
  mask=torch.rand(n,generator=gen)<.1
  Y[mask]=torch.randint(out,(int(mask.sum()),),generator=gen)
 else:
  Y=z+.4*torch.randn(n,out,generator=gen)
  noise=(torch.rand(n,out,generator=gen)<.15).float()
  Y+=noise*torch.randn(n,out,generator=gen)*2
 return X,Y


Overwriting stage30_src/rnn_arch.py


In [ ]:
%%writefile stage30_src/packed_prox.py
"""Stage 18 discrete-proximal training with packed low-precision co-state.
For fixed row levels, solves exactly per entry:
  min_(k in {0,1,2,3}, e in E_b(a_i))
   .5*h_ij*(q_i(k)+e-T_ij)^2 + .5*lambda*h_ij*e^2.
Eliminate e by deterministic nearest-neighbor to (T-q)/(1+lambda), then
choose the lowest of four profiled costs; update row codebooks conditionally.
No FP32 parameter-sized persistent tensors. Residual state is packed INT4
(two signed codewords/byte), or INT8; materialization is temporary.
"""
import torch
import math
from discrete_optim import decode,constrained_two_level_fit

def _decode_4(packed, length):
    lo=(packed & 15).to(torch.int16)-7
    hi=((packed>>4)&15).to(torch.int16)-7
    return torch.stack((lo,hi),dim=-1).flatten(-2)[...,:length].to(torch.float32)

def _encode_4(code):
    code=(code.to(torch.int16)+7).clamp(0,14).to(torch.uint8)
    n=code.shape[1]
    if n%2:code=torch.cat([code,torch.full((len(code),1),7,device=code.device,dtype=torch.uint8)],dim=1)
    return (code[:,0::2]|(code[:,1::2]<<4)).contiguous()

class PackedDiscreteProx:
    def __init__(self,model,lr=.12,bits=4,lam=.0,metric='rms',passes=1,
                 residual_span=1.,momentum=False,momentum_beta=.9,flip_fraction=1.0):
        self.model=model;self.lr=float(lr);self.bits=int(bits);self.lam=float(lam)
        self.metric=metric;self.passes=int(passes);self.residual_span=residual_span
        self.momentum=momentum;self.beta=momentum_beta;self.flip_fraction=float(flip_fraction)
        self.state={};self.calls=0;self.transitions=0
        for layer in model.direct_layers():
            m,n=layer.k.shape
            if bits==4:buf=torch.full((m,(n+1)//2),0x77,device=layer.k.device,dtype=torch.uint8)
            elif bits==8:buf=torch.zeros((m,n),device=layer.k.device,dtype=torch.int8)
            else:raise ValueError(bits)
            self.state[layer]={'error':buf,'res_scale':layer.a.detach().clone()*self.residual_span}
            if momentum:
                self.state[layer]['momentum']=torch.zeros((m,n),device=layer.k.device,dtype=torch.int8)
                self.state[layer]['g_scale']=torch.ones((m,1),device=layer.k.device)
    def _unpack(self,layer):
        e=self.state[layer]['error']
        if self.bits==4:return _decode_4(e,layer.k.shape[1]) / 7.
        return e.float()/127.
    def _quantize(self,e,scale):
        maxval=7 if self.bits==4 else 127
        code=torch.round((e/scale[:,None].clamp_min(1e-8))*maxval).clamp(-maxval,maxval)
        return _encode_4(code) if self.bits==4 else code.to(torch.int8)
    @torch.no_grad()
    def step(self):
        self.calls+=1;self.transitions=0
        for layer in self.model.direct_layers():
            if layer._w is None or layer._w.grad is None:continue
            k=layer.k;a=layer.a;b=layer.b;g=layer._w.grad.detach()
            old=decode(k,a,b)
            if self.momentum:
                s=self.state[layer];gs=g.square().mean(1,keepdim=True).sqrt().clamp_min(1e-8)
                last=s['momentum'].float()/127*s['g_scale']
                m=self.beta*last+(1-self.beta)*g
                ms=m.abs().amax(1,keepdim=True).clamp_min(1e-8)
                s['momentum'].copy_(torch.round(m/ms*127).clamp(-127,127).to(torch.int8));s['g_scale'].copy_(ms)
                g=m
            rms=g.square().mean(1,keepdim=True).sqrt().clamp_min(1.e-8)
            if self.metric=='factored':
                colrms=g.square().mean(0,keepdim=True).sqrt().clamp_min(1.e-8)
                globalrms=g.square().mean().sqrt().clamp_min(1.e-8)
                h=(rms*colrms/globalrms).clamp_min(1.e-8)
            else:h=rms
            step=-self.lr*a[:,None]*(g/h).clamp(-4,4)
            scale=self.state[layer]['res_scale']
            error=self._unpack(layer)*scale[:,None]
            target=old+error+step
            # minimize fixed-codebook joint e,k objective; exact finite profile
            # all shapes [m, 4, n]
            q=torch.stack((-a,-b,b,a),dim=1)[:,:,None]
            emax=scale[:,None,None]
            denom=(1+self.lam)
            ev=torch.round((target[:,None,:]-q)/denom/emax.clamp_min(1e-8)*(7 if self.bits==4 else 127))
            ev=ev.clamp(-(7 if self.bits==4 else 127),(7 if self.bits==4 else 127))
            er=ev*emax/(7 if self.bits==4 else 127)
            cost=(q+er-target[:,None,:]).square()+self.lam*er.square()
            idx=cost.argmin(1).to(torch.uint8)
            if self.flip_fraction<1.0:
                # Exact top-k trust-region selection for fixed (a,b); changes in
                # codebook magnitudes during the next step are not capped here.
                current_cost=cost.gather(1,k.long()[:,None,:]).squeeze(1)
                best_cost=cost.gather(1,idx.long()[:,None,:]).squeeze(1)
                benefit=torch.where(idx!=k,h*(current_cost-best_cost),torch.zeros_like(best_cost))
                budget=max(1,math.ceil(self.flip_fraction*k.shape[1]))
                keep=torch.zeros_like(benefit,dtype=torch.bool)
                if budget>0:
                    top=benefit.topk(k=min(budget,k.shape[1]),dim=1).indices
                    keep.scatter_(1,top,True)
                idx=torch.where(keep&(benefit>1e-12),idx,k)
            ee=er.gather(1,idx.long()[:,None,:]).squeeze(1)
            # conditional row codebook optimization for fixed index / residual
            for _ in range(self.passes):
                aa_sign=torch.where(idx==0,-1.,torch.where(idx==3,1.,0.))
                bb_sign=torch.where(idx==1,-1.,torch.where(idx==2,1.,0.))
                rem=target-ee
                Ha=(h*aa_sign.square()).sum(1);Hb=(h*bb_sign.square()).sum(1)
                ua=(h*aa_sign*rem).sum(1)/Ha.clamp_min(1e-8)
                ub=(h*bb_sign*rem).sum(1)/Hb.clamp_min(1e-8)
                na,nb=constrained_two_level_fit(ua,ub,Ha,Hb,a,b)
                a.copy_(na);b.copy_(nb)
                qn=decode(idx,a,b)
                ee=torch.round(((target-qn)/(1+self.lam))/scale[:,None].clamp_min(1e-8)*(7 if self.bits==4 else 127))
                ee=ee.clamp(-(7 if self.bits==4 else 127),(7 if self.bits==4 else 127))*scale[:,None]/(7 if self.bits==4 else 127)
            self.transitions+=int((idx!=k).sum().item())
            layer.k.copy_(idx)
            self.state[layer]['error'].copy_(self._quantize(ee,scale))
    def aux_bytes(self):
        return sum(v.numel()*v.element_size() for st in self.state.values() for v in st.values())
    def state_bytes(self):
        return sum(l.k.numel()+8*l.a.numel() for l in self.model.direct_layers())+self.aux_bytes()


Overwriting stage30_src/packed_prox.py


In [ ]:
%%writefile stage30_src/packed_prox_numba.py
"""Fused CPU reference for a deterministic, profiled INT2+INT4 proximal QP.
Numba consumes/updates CPU torch tensor .numpy() views in place. No allocation
of parameter-sized optimizer state beyond existing nibble-packed residual bytes.
The fixed per-row residual grid allows exact conditional minimizers.
GPU path is PackedDiscreteProx (Torch) until a CUDA/Triton kernel exists.
"""
import math
import numpy as np
import torch
from numba import njit
from packed_prox import PackedDiscreteProx

@njit(cache=True)
def _clip(x,lo,hi):
    return min(hi,max(lo,x))

@njit(cache=True)
def _round_grid(x,maximum):
    # nearest integer, even ties; NumPy/Torch-compatible
    return int(_clip(np.rint(x),-maximum,maximum))

@njit(cache=True)
def _fitter(ua,ub,Ha,Hb,aa0,bb0,amin=0.006,amax=2.,bmin=0.002,gap=0.004):
    if Ha<1e-12:ua=aa0
    if Hb<1e-12:ub=bb0
    wa=max(Ha,1e-12);wb=max(Hb,1e-12)
    best=1e30;ans_a=aa0;ans_b=bb0
    # unconstrained
    if (ua>=amin and ua<=amax and ub>=bmin and ub<=ua-gap):
        best=0.;ans_a=ua;ans_b=ub
    # a=amin
    ax=amin;bx=_clip(ub,bmin,amin-gap)
    val=wa*(ax-ua)**2+wb*(bx-ub)**2
    if val<best:best=val;ans_a=ax;ans_b=bx
    # a=amax
    ax=amax;bx=_clip(ub,bmin,amax-gap)
    val=wa*(ax-ua)**2+wb*(bx-ub)**2
    if val<best:best=val;ans_a=ax;ans_b=bx
    # b=bmin
    bx=bmin;ax=_clip(ua,max(amin,bmin+gap),amax)
    val=wa*(ax-ua)**2+wb*(bx-ub)**2
    if val<best:best=val;ans_a=ax;ans_b=bx
    # a=b+gap
    bx=_clip((wa*(ua-gap)+wb*ub)/(wa+wb),max(bmin,amin-gap),amax-gap);ax=bx+gap
    val=wa*(ax-ua)**2+wb*(bx-ub)**2
    if val<best:best=val;ans_a=ax;ans_b=bx
    return ans_a,ans_b

@njit(cache=True)
def _fused(k,a,b,packed,scale,g,lr,lam,bitwidth,flip_fraction,metric):
    m,n=k.shape
    maximum=7 if bitwidth==4 else 127
    nflip=0
    col_rms=np.ones(n,dtype=np.float64)
    global_rms=1.
    if metric==1:
        globalsum=0.
        for j in range(n):
            vv=0.
            for i in range(m):vv+=float(g[i,j])*float(g[i,j])
            col_rms[j]=max(math.sqrt(vv/m),1.e-8)
            globalsum+=vv
        global_rms=max(math.sqrt(globalsum/(m*n)),1.e-8)
    for i in range(m):
        old_a=float(a[i]);old_b=float(b[i]);ss=float(scale[i]);
        rms=0.
        for j in range(n):rms+=float(g[i,j])*float(g[i,j])
        rms=max(math.sqrt(rms/n),1.e-8)
        target=np.empty(n,dtype=np.float64)
        ix=np.empty(n,dtype=np.uint8)
        ee=np.empty(n,dtype=np.float64)
        benefits=np.zeros(n,dtype=np.float64)
        incerr=np.empty(n,dtype=np.float64)
        hz=np.empty(n,dtype=np.float64)
        for j in range(n):
            code=int(k[i,j]);old=(-old_a if code==0 else -old_b if code==1 else old_b if code==2 else old_a)
            if bitwidth==4:
                bv=int(packed[i,j//2]);signcode=((bv&15) if j%2==0 else ((bv>>4)&15))-7
            else:signcode=int(packed[i,j])
            err=signcode*ss/maximum
            hh=rms if metric==0 else max(rms*col_rms[j]/global_rms,1.e-8)
            hz[j]=hh
            gg=float(g[i,j])/hh
            step=-lr*old_a*_clip(gg,-4.,4.)
            t=old+err+step;target[j]=t
            best=1e30;best_k=code;best_e=0.
            for kk in range(4):
                q=(-old_a if kk==0 else -old_b if kk==1 else old_b if kk==2 else old_a)
                v=_round_grid((t-q)/(1.+lam)/ss*maximum,maximum)
                et=v*ss/maximum
                cost=(q+et-t)**2+lam*et**2
                if cost<best:best=cost;best_k=kk;best_e=et
            ix[j]=best_k;ee[j]=best_e
            olderr=_round_grid((t-old)/(1.+lam)/ss*maximum,maximum)*ss/maximum
            oldcost=(old+olderr-t)**2+lam*olderr**2
            benefits[j]=hz[j]*(oldcost-best) if best_k!=code else 0.
            incerr[j]=olderr
        if flip_fraction<1.:
            budget=max(1,int(math.ceil(flip_fraction*n)))
            sorted_j=np.argsort(benefits)
            choose=np.zeros(n,dtype=np.uint8)
            for pp in range(n-budget,n):
                choose[sorted_j[pp]]=1
            for j in range(n):
                if choose[j]==0 or benefits[j]<=1.e-12:
                    ix[j]=k[i,j];ee[j]=incerr[j]
        Ha=0.;Hb=0.;Sa=0.;Sb=0.
        for j in range(n):
            ixj=int(ix[j]);t=target[j]-ee[j];w=hz[j]
            if ixj==0:Ha+=w;Sa-=w*t
            elif ixj==3:Ha+=w;Sa+=w*t
            elif ixj==1:Hb+=w;Sb-=w*t
            else:Hb+=w;Sb+=w*t
        ua=Sa/max(Ha,1e-8);ub=Sb/max(Hb,1e-8)
        new_a,new_b=_fitter(ua,ub,Ha,Hb,old_a,old_b)
        a[i]=new_a;b[i]=new_b
        for j in range(n):
            kk=int(ix[j]);q=(-new_a if kk==0 else -new_b if kk==1 else new_b if kk==2 else new_a)
            erq=_round_grid(((target[j]-q)/(1.+lam))/ss*maximum,maximum)
            if bitwidth==4:
                oldbyte=int(packed[i,j//2]);code=erq+7
                if j%2==0:packed[i,j//2]=(oldbyte&240)|code
                else:packed[i,j//2]=(oldbyte&15)|(code<<4)
            else:packed[i,j]=erq
            if kk!=int(k[i,j]):nflip+=1
            k[i,j]=kk
    return nflip

class FusedPackedDiscreteProx(PackedDiscreteProx):
    @torch.no_grad()
    def step(self):
        if self.momentum or self.passes!=1 or self.metric not in ('rms','factored'):
            return super().step()
        self.calls+=1;self.transitions=0
        for l in self.model.direct_layers():
            if l._w is None or l._w.grad is None:continue
            if l.k.device.type!='cpu':
                # PyTorch tensor reference path on GPU, no pre-existing gradient release.
                self.calls-=1
                return super().step()
            st=self.state[l]
            self.transitions+=_fused(l.k.numpy(),l.a.numpy(),l.b.numpy(),st['error'].numpy(),
                st['res_scale'].numpy(),l._w.grad.detach().contiguous().numpy(),self.lr,self.lam,self.bits,self.flip_fraction,1 if self.metric=='factored' else 0)


Overwriting stage30_src/packed_prox_numba.py


In [ ]:
%%writefile stage30_src/large_suite.py
"""Stage 16 scalable experiment runner: loss-agnostic all-layer 2-bit training.
Runs MLP, residual MLP, CNN on noisy/misspecified synthetic data and optional
FashionMNIST/CIFAR-10. Full TRAINING loss vs updates and training seconds.
No actual INT2 storage or kernels: PyTorch uses temporary FP32 realized weights.
GPU benchmarking needs a CUDA runtime and is not validated on this CPU host.
"""
import argparse,math,os,json,time,random
from pathlib import Path
from functools import lru_cache
import numpy as np,pandas as pd
import torch
from torch.nn import functional as F
from discrete_optim import QuantMLP,DiscreteSurrogateOptimizer,BaselineOptimizer,decode
from advanced_models import ResidualQuantMLP,QuantCNN
from packed_prox import PackedDiscreteProx
try:
 from packed_prox_numba import FusedPackedDiscreteProx
except ImportError:
 FusedPackedDiscreteProx=PackedDiscreteProx
from rnn_arch import QuantRNN,sequence_data

PRESETS={
 'mlp_robust':dict(arch='mlp',task='robust',n=2400,input_dim=64,width=128,depth=4,out=8,batch=192,steps=250),
 'resmlp_quantile':dict(arch='resmlp',task='quantile',n=2400,input_dim=64,width=128,depth=4,out=8,batch=192,steps=250),
 'resmlp_multiclass':dict(arch='resmlp',task='multiclass',n=3000,input_dim=64,width=128,depth=4,out=8,batch=192,steps=280),
 'resmlp_focal':dict(arch='resmlp',task='focal',n=3000,input_dim=64,width=128,depth=4,out=8,batch=192,steps=280),
 'mlp_bce':dict(arch='mlp',task='bce',n=2400,input_dim=64,width=128,depth=4,out=8,batch=192,steps=250),
 'resmlp_poisson':dict(arch='resmlp',task='poisson',n=2400,input_dim=64,width=128,depth=4,out=8,batch=192,steps=250),
 'mlp_ranking':dict(arch='mlp',task='ranking',n=2400,input_dim=64,width=128,depth=4,out=1,batch=192,steps=250),
 'rnn_robust':dict(arch='rnn',task='robust',n=2000,input_dim=8,width=64,depth=1,out=4,batch=96,steps=260,sequence_length=14),
 'rnn_bce':dict(arch='rnn',task='bce',n=2000,input_dim=8,width=64,depth=1,out=4,batch=96,steps=260,sequence_length=14),
 'rnn_multiclass':dict(arch='rnn',task='multiclass',n=2000,input_dim=8,width=64,depth=1,out=4,batch=96,steps=260,sequence_length=14),
 'cnn_shapes':dict(arch='cnn',task='multiclass',n=3000,channels=1,width=24,out=4,batch=96,steps=200),
 'cnn_fashion':dict(arch='cnn',task='multiclass',n=6000,channels=1,width=24,out=10,batch=96,steps=250),
 'cnn_cifar':dict(arch='cnn',task='multiclass',n=8000,channels=3,width=24,out=10,batch=96,steps=250),
}

@lru_cache(maxsize=30)
def make_dataset(preset,seed,n=None,root='./datasets'):
    c=PRESETS[preset];n=n or c['n'];g=torch.Generator().manual_seed(seed+71001)
    if c['arch']=='rnn':return sequence_data(seed,c['task'],n=n,T=c['sequence_length'],dim=c['input_dim'],out=c['out'])
    if preset in ('cnn_fashion','cnn_cifar'):
        try:
            from torchvision import datasets,transforms
        except Exception as exc:
            raise RuntimeError('torchvision is required for FashionMNIST/CIFAR10 presets. Install a torch-compatible torchvision.') from exc
        ds=(datasets.FashionMNIST(root,train=True,transform=transforms.ToTensor(),download=True)
            if preset=='cnn_fashion' else datasets.CIFAR10(root,train=True,transform=transforms.ToTensor(),download=True))
        inds=torch.randperm(len(ds),generator=g)[:n]
        X=torch.stack([ds[int(i)][0] for i in inds]);Y=torch.tensor([int(ds[int(i)][1]) for i in inds])
        X=(X-.5)/.5
        noisy=torch.rand(n,generator=g)<.12
        Y[noisy]=torch.randint(0,c['out'],(int(noisy.sum()),),generator=g)
        return X,Y
    if preset=='cnn_shapes':
        Y=torch.randint(0,4,(n,),generator=g)
        xcoord=torch.arange(28)[None,None,:].float();ycoord=torch.arange(28)[None,:,None].float()
        cx=torch.randint(7,22,(n,),generator=g).float()[:,None,None]
        cy=torch.randint(7,22,(n,),generator=g).float()[:,None,None]
        th=1.0+torch.rand(n,1,1,generator=g)*2.0
        vertical=(xcoord-cx).abs()<th
        horizontal=(ycoord-cy).abs()<th
        diag=((xcoord-cx)-(ycoord-cy)).abs()<th*1.4
        circle=(((xcoord-cx)**2+(ycoord-cy)**2).sqrt()-(4+th)).abs()<th
        canvas=torch.where((Y==0)[:,None,None],vertical,torch.where((Y==1)[:,None,None],horizontal,torch.where((Y==2)[:,None,None],diag,circle))).float()
        canvas=canvas+torch.randn(n,28,28,generator=g)*.35
        X=canvas[:,None,:,:]
        noisy=torch.rand(n,generator=g)<.15
        Y[noisy]=torch.randint(0,4,(int(noisy.sum()),),generator=g)
        return X,Y
    d=c['input_dim'];o=c['out'];X=torch.randn(n,d,generator=g)
    W1=torch.randn(d,96,generator=g)/math.sqrt(d)
    W2=torch.randn(96,80,generator=g)/math.sqrt(96)
    W3=torch.randn(80,o,generator=g)/math.sqrt(80)
    Z=(torch.tanh(1.5*X@W1)@W2).tanh()@W3*3.6
    Z=Z+.5*torch.sin(2*X[:,:o])+(.25*X[:,:o]**2-.25)
    task=c['task']
    if task in ('robust','quantile'):
        Y=Z+.3*torch.randn(n,o,generator=g)
        spikes=torch.rand(n,o,generator=g)<.12
        Y=Y+spikes*torch.randn(n,o,generator=g)*2.5
    elif task in ('multiclass','focal'):
        probs=torch.softmax(1.5*Z,dim=1)
        Y=torch.multinomial(probs,1,generator=g).squeeze(1)
        flips=torch.rand(n,generator=g)<.20
        Y[flips]=torch.randint(o,(int(flips.sum()),),generator=g)
    elif task=='bce':
        Y=torch.bernoulli(torch.sigmoid(Z),generator=g)
    elif task=='poisson':
        Y=torch.poisson(torch.exp(Z.clamp(-3,3)),generator=g)
    elif task=='ranking':
        A=torch.randn(n,d,generator=g);B=torch.randn(n,d,generator=g)
        def teacher(v):
            zz=(torch.tanh(1.5*v@W1)@W2).tanh()@W3*3.6
            return zz[:,0]
        da=teacher(A)-teacher(B)+.7*torch.randn(n,generator=g)
        Y=torch.where(da>0,1.,-1.)
        flips=torch.rand(n,generator=g)<.12
        Y[flips]=-Y[flips]
        X=torch.stack([A,B],dim=1)
    else:raise ValueError(task)
    return X.float(),Y

def objective(pred,y,task):
    if task=='robust':
        r=pred-y;delta=.55;return (delta**2*(torch.sqrt(1+(r/delta)**2)-1)).mean()
    if task=='quantile':
        r=y-pred;return (.8*r+.3*F.softplus(-r/.3)).mean()
    if task=='multiclass':return F.cross_entropy(pred,y.long())
    if task=='focal':
        lp=F.log_softmax(pred,dim=1);py=lp[torch.arange(len(y),device=y.device),y.long()]
        return (-((1-py.exp())**2)*py).mean()
    if task=='bce':return F.binary_cross_entropy_with_logits(pred,y)
    if task=='poisson':return (torch.exp(pred.clamp(-6,6))-y*pred.clamp(-6,6)).mean()
    if task=='ranking':return F.softplus(-y*pred.reshape(-1)).mean()
    raise ValueError(task)

def make_model(preset,method,seed,device):
    c=PRESETS[preset];mtype='direct' if method.startswith('d') else 'ste'
    if c['arch']=='mlp':model=QuantMLP([c['input_dim']]+[c['width']]*c['depth']+[c['out']],seed=seed,method=mtype,device=device)
    elif c['arch']=='resmlp':model=ResidualQuantMLP(c['input_dim'],c['width'],c['depth'],c['out'],seed=seed,method=mtype,device=device)
    elif c['arch']=='rnn':model=QuantRNN(d=c['input_dim'],h=c['width'],out=c['out'],seed=seed,method=mtype,device=device)
    else:model=QuantCNN(c['channels'],c['out'],c['width'],seed=seed,method=mtype,device=device)
    return model.to(device)

def predict(model,x,task):
    if task=='ranking':return (model(x[:,0])-model(x[:,1])).squeeze(-1)
    return model(x)

def cuda_sync(device):
    if str(device).startswith('cuda'):torch.cuda.synchronize(device)

def evaluate(model,X,Y,task,device,chunk=512):
    out=0.;cost=0.
    with torch.no_grad():
        for a in range(0,len(X),chunk):
            xx=X[a:a+chunk].to(device);yy=Y[a:a+chunk].to(device)
            t=time.perf_counter();v=objective(predict(model,xx,task),yy,task)
            cuda_sync(device);cost+=time.perf_counter()-t
            out+=float(v.item())*len(xx)
    return out/len(X),cost

def state_persist(opt,model,direct):
    if direct:
        st=[]
        for layer in model.direct_layers():
            d={k:(v.detach().cpu().clone() if torch.is_tensor(v) else v) for k,v in opt.state.get(layer,{}).items()}
            st.append(d)
        return {'state':st,'calls':opt.calls}
    return {'opt':opt.opt.state_dict(),'code_opt':opt.sc.state_dict()}

def restore_optimizer(opt,model,direct,obj,device):
    if direct:
        opt.calls=obj['calls']
        for layer,d in zip(model.direct_layers(),obj['state']):
            opt.state[layer]={k:(v.to(device) if torch.is_tensor(v) else v) for k,v in d.items()}
    else:
        opt.opt.load_state_dict(obj['opt']);opt.sc.load_state_dict(obj['code_opt'])

def train(preset,method,cfg,seed=101,steps=None,n=None,batch=None,device='cpu',eval_every=15,checkpoint_dir=None,resume=False,checkpoint_every=50):
    c=PRESETS[preset];steps=steps or c['steps'];batch=batch or c['batch']
    X,Y=make_dataset(preset,seed,n=n);task=c['task']
    torch.manual_seed(seed+17000)
    model=make_model(preset,method,seed+9000,device)
    direct=method.startswith('d')
    if method in ('dq4','dq4f','dq4trust'):
        opt_cfg=dict(cfg)
        if method=='dq4f':opt_cfg.setdefault('metric','factored')
        if method=='dq4trust':opt_cfg.setdefault('flip_fraction',.05)
        opt=(FusedPackedDiscreteProx if str(device)=='cpu' else PackedDiscreteProx)(model,bits=4,**opt_cfg)
    elif method=='dq8':opt=(FusedPackedDiscreteProx if str(device)=='cpu' else PackedDiscreteProx)(model,bits=8,**cfg)
    elif direct:opt=DiscreteSurrogateOptimizer(model,kind=method[1:],**cfg)
    else:opt=BaselineOptimizer(model,method=method,**cfg)
    rng=torch.Generator().manual_seed(seed+9999)
    order=torch.randint(0,len(X),(steps,batch),generator=rng)
    tag=f'{preset}_{method}_{seed}';checkpoint=None
    if checkpoint_dir:
        checkpoint=Path(checkpoint_dir)/(tag+'.pt');checkpoint.parent.mkdir(exist_ok=True,parents=True)
    t=0.;opt_t=0.;eval_t=0.;checkpoint_t=0.;curves=[];start_step=0;total_transitions=0
    if resume and checkpoint is not None and checkpoint.exists():
        v=torch.load(checkpoint,map_location='cpu',weights_only=False)
        if v['cfg']!=cfg or v['steps']!=steps or v['batch']!=batch:raise ValueError('Checkpoint configuration differs from requested run')
        model.load_state_dict(v['model']);restore_optimizer(opt,model,direct,v['optimizer'],device)
        if 'torch_rng' in v:torch.set_rng_state(v['torch_rng'])
        if str(device).startswith('cuda') and 'cuda_rng' in v:torch.cuda.set_rng_state_all(v['cuda_rng'])
        t=v['train_s'];opt_t=v['opt_s'];eval_t=v['eval_s'];curves=v['curve'];start_step=v['step'];total_transitions=v['transitions']
    if str(device).startswith('cuda'):torch.cuda.reset_peak_memory_stats(device)
    for step in range(start_step,steps+1):
        if (step%eval_every==0 or step==steps) and (not curves or curves[-1]['step']!=step):
            tr_loss,evsec=evaluate(model,X,Y,task,device)
            eval_t+=evsec
            curves.append(dict(preset=preset,task=task,method=method,seed=seed,step=step,full_training_loss=tr_loss,training_seconds=t))
        if step==steps:break
        ids=order[step]
        xx=X[ids].to(device);yy=Y[ids].to(device)
        cuda_sync(device);t0=time.perf_counter()
        if direct:model.zero_grad(set_to_none=True)
        else:opt.zero_grad()
        v=objective(predict(model,xx,task),yy,task)
        v.backward()
        cuda_sync(device);t1=time.perf_counter()
        opt.step()
        cuda_sync(device);t2=time.perf_counter()
        t+=t2-t0;opt_t+=t2-t1
        if direct:total_transitions+=opt.transitions
        if checkpoint is not None and (step+1)%checkpoint_every==0:
            p0=time.perf_counter()
            cp={'cfg':cfg,'batch':batch,'steps':steps,'step':step+1,'model':model.state_dict(),
                'optimizer':state_persist(opt,model,direct),'torch_rng':torch.get_rng_state(),
                'cuda_rng':torch.cuda.get_rng_state_all() if str(device).startswith('cuda') else None,'train_s':t,'opt_s':opt_t,'eval_s':eval_t,'curve':curves,'transitions':total_transitions}
            torch.save(cp,checkpoint);checkpoint_t+=time.perf_counter()-p0
    vals=np.array([h['full_training_loss'] for h in curves]);epochs=np.array([h['step'] for h in curves]);secs=np.array([h['training_seconds'] for h in curves])
    gpu_peak=int(torch.cuda.max_memory_allocated(device)) if str(device).startswith('cuda') else None
    result=dict(preset=preset,task=task,method=method,seed=seed,cfg=json.dumps(cfg,sort_keys=True),
                n=len(X),steps=steps,batch=batch,device=str(device),initial=vals[0],final=vals[-1],
                loss_auc=float(np.trapezoid(vals,epochs)/steps),loss_time_auc=float(np.trapezoid(vals,secs)/max(t,1e-12)),
                training_s=t,optimizer_only_s=opt_t,evaluation_s=eval_t,checkpoint_s=checkpoint_t,
                optimizer_plus_parameter_bytes=int(opt.state_bytes()),persistent_auxiliary_bytes=int(opt.aux_bytes()) if hasattr(opt,'aux_bytes') else None,gpu_peak_bytes=gpu_peak,
                total_index_transitions=total_transitions)
    return result,curves

# Tune independently on development seeds; never choose based on held-out runs.
SEARCH_GRIDS={
 'dq4':[dict(lr=lr,lam=lam) for lr in (.04,.10,.20,.40) for lam in (0.,.10)],
 'dq8':[dict(lr=lr,lam=lam) for lr in (.04,.10,.20,.40) for lam in (0.,.10)],
 'dq4f':[dict(lr=lr,lam=lam) for lr in (.04,.10,.20,.40) for lam in (0.,.10)],
 'dq4trust':[dict(lr=lr,lam=.10,flip_fraction=frac) for lr in (.04,.10,.20) for frac in (.025,.05,.2)],
 'dsr':[{'lr':v} for v in (.025,.05,.10,.20)],
 'dqmc':[{'lr':v} for v in (.025,.05,.10,.20)],
 'dsketch':[{'lr':v} for v in (.025,.05,.10,.20)],
 'drank1':[{'lr':v} for v in (.025,.05,.10,.20)],
 'drowcol':[{'lr':v} for v in (.025,.05,.10,.20)],
 'dstrat':[{'lr':v} for v in (.025,.05,.10,.20)],
 'ddet':[{'lr':v} for v in (.05,.15,.30)],
 'ddualavg':[{'lr':v} for v in (.12,.24,.4,.7)],
 'dfeedback':[{'lr':v} for v in (.04,.08,.12,.2)],
 'dfull':[{'lr':v} for v in (.075,.15,.3)],
 'dbudget':[{'lr':v} for v in (.1,.2,.4)],
 'ddiagonal':[{'lr':v} for v in (.1,.2,.4)],
 'adam':[{'lr':lr,'code_lr':cr,'head_lr_factor':f} for lr in (.01,.03,.07) for cr in (.003,.02) for f in (.5,2.)],
 'muon':[{'lr':lr,'code_lr':cr,'head_lr_factor':f} for lr in (.03,.1,.3) for cr in (.003,.02) for f in (.5,2.)],
}

def experiment(presets,methods=('ddualavg','dfeedback','dfull','adam','muon'),out='stage16_outputs',device='cpu',
               devseeds=(11,12),evalseeds=(101,102,103),devsteps=60,steps_override=None,n_override=None,
               batch_override=None,resume=False):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    dev=[];held=[];curves=[];locked={}
    for preset in presets:
        locked[preset]={}
        for method in methods:
            best=None
            for cfg in SEARCH_GRIDS[method]:
                rs=[]
                for seed in devseeds:
                    r,_=train(preset,method,cfg,seed=seed,steps=devsteps,n=n_override,batch=batch_override,device=device,eval_every=max(5,devsteps//6))
                    rs.append(r);dev.append(r)
                score=float(np.mean([r['loss_auc'] for r in rs]))
                if best is None or score<best[0]:best=(score,cfg)
            locked[preset][method]=best[1]
            print('LOCKED',preset,method,best,flush=True)
    pd.DataFrame(dev).to_csv(out/'development.csv',index=False)
    (out/'locked_configs.json').write_text(json.dumps(locked,indent=2))
    (out/'protocol.json').write_text(json.dumps(dict(presets=list(presets),methods=list(methods),devseeds=list(devseeds),evalseeds=list(evalseeds),devsteps=devsteps,steps_override=steps_override,n_override=n_override,batch_override=batch_override,device=device),indent=2))
    for preset in presets:
        for seed in evalseeds:
            for method in methods:
                r,h=train(preset,method,locked[preset][method],seed=seed,steps=steps_override,n=n_override,batch=batch_override,device=device,
                          checkpoint_dir=out/'checkpoints',resume=resume)
                held.append(r);curves.extend(h)
                pd.DataFrame(held).to_csv(out/'heldout.csv',index=False)
                pd.DataFrame(curves).to_csv(out/'curves.csv',index=False)
                print('EVAL',preset,seed,method,'AUC',round(r['loss_auc'],5),'final',round(r['final'],5),'sec',round(r['training_s'],3),flush=True)
    return pd.DataFrame(held),pd.DataFrame(curves)

def make_plots(results,curves,out='stage16_outputs'):
    import matplotlib.pyplot as plt
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    if len(curves)==0:return
    for preset in curves['preset'].unique():
        sub=curves[curves.preset==preset]
        for x,col in [('step','updates'),('training_seconds','time')]:
            fig,ax=plt.subplots(figsize=(8,5))
            for method in sub['method'].unique():
                p=sub[sub.method==method]
                groups=[]
                for seed in p.seed.unique():
                    z=p[p.seed==seed].sort_values(x)
                    groups.append((z[x].to_numpy(),z.full_training_loss.to_numpy()))
                if x=='step' and len({len(k[0]) for k in groups})==1:
                    yy=np.array([k[1] for k in groups]);xx=groups[0][0]
                    ax.plot(xx,yy.mean(0),label=method)
                else:
                    for i,(xx,yy) in enumerate(groups):ax.plot(xx,yy,label=method if i==0 else None,alpha=.45)
            ax.set(xlabel='Optimizer updates' if x=='step' else 'Training seconds (evaluation excluded)',ylabel='Full training loss',title=preset)
            ax.legend();fig.tight_layout();fig.savefig(out/f'{preset}_loss_vs_{col}.png',dpi=150);plt.close(fig)

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--presets',default='resmlp_multiclass,mlp_robust,cnn_shapes')
    p.add_argument('--methods',default='ddualavg,dfeedback,dfull,adam,muon')
    p.add_argument('--device',default='cuda' if torch.cuda.is_available() else 'cpu')
    p.add_argument('--output',default='stage16_outputs');p.add_argument('--devsteps',type=int,default=60)
    p.add_argument('--steps',type=int,default=None);p.add_argument('--n',type=int,default=None)
    p.add_argument('--batch',type=int,default=None);p.add_argument('--resume',action='store_true')
    p.add_argument('--devseeds',default='11,12');p.add_argument('--evalseeds',default='101,102,103')
    z=p.parse_args();results,curves=experiment(z.presets.split(','),z.methods.split(','),out=z.output,device=z.device,
             devseeds=[int(v) for v in z.devseeds.split(',')],evalseeds=[int(v) for v in z.evalseeds.split(',')],devsteps=z.devsteps,
             steps_override=z.steps,n_override=z.n,batch_override=z.batch,resume=z.resume)
    make_plots(results,curves,z.output)


Overwriting stage30_src/large_suite.py


In [ ]:
%%writefile stage30_src/stage22_optim.py
"""Stage22 experimental solvers: activation-diagonal discretely optimized INT2+packed INT4.

At step t, choose q[k], e on a fixed per-row INT4 grid to minimize
  sum_ij (h_ij/2) (q_ij + e_ij - T_ij)^2 + (lam*h_ij/2)e_ij^2
  + kappa*n/2 sum_i [(a_i-a_old)^2+(b_i-b_old)^2],
 T_ij = W_ij + beta*E_ij - eta*a_i*g_ij/(RMS(g_i)*v_j).
 v_j approximates E[x_j^2]/mean E[x^2], clipped.
 For fixed a,b, the joint (k,e) minimizer is exact (4 choices, closed-form e).
 For fixed k,e, row a,b is a convex 2-variable constrained quadratic.
 Alternating is NOT a global minimizer. State INT4 per weight, scales per row.
"""
import math
import numpy as np
import torch
from numba import njit
from packed_prox import PackedDiscreteProx
from packed_prox_numba import _fitter

@njit(cache=True)
def _activation_step(k,a,b,packed,scale,g,v,lr,lam,beta,kappa,cap):
    m,n=k.shape
    numflips=0
    for i in range(m):
        ai=float(a[i]);bi=float(b[i]);ss=float(scale[i]);gr=1.e-8
        for j in range(n):gr+=float(g[i,j])**2
        gr=math.sqrt(gr/n)
        target=np.empty(n,dtype=np.float64)
        ids=np.empty(n,dtype=np.uint8)
        ee=np.empty(n,dtype=np.float64)
        for j in range(n):
            ko=int(k[i,j]);old=(-ai if ko==0 else -bi if ko==1 else bi if ko==2 else ai)
            packedbyte=int(packed[i,j//2]);ec=((packedbyte&15) if j%2==0 else ((packedbyte>>4)&15))-7
            olderr=ec*ss/7.
            t=old+beta*olderr-lr*ai*max(-cap,min(cap,float(g[i,j])/gr/max(1.e-6,float(v[j]))))
            target[j]=t
            best=1.e50;bestk=ko;beste=0.
            for z in range(4):
                q=(-ai if z==0 else -bi if z==1 else bi if z==2 else ai)
                v2=int(max(-7,min(7,np.rint((t-q)/(1.+lam)/ss*7.))))
                e=v2*ss/7.
                f=(q+e-t)**2+lam*e*e
                if f<best:best=f;bestk=z;beste=e
            ids[j]=bestk;ee[j]=beste
        Ha=kappa*n;Hb=kappa*n;Sa=kappa*n*ai;Sb=kappa*n*bi
        for j in range(n):
            ko=int(ids[j]);t=target[j]-ee[j];vij=float(v[j])
            if ko==0:Ha+=vij;Sa-=vij*t
            elif ko==3:Ha+=vij;Sa+=vij*t
            elif ko==1:Hb+=vij;Sb-=vij*t
            else:Hb+=vij;Sb+=vij*t
        na,nb=_fitter(Sa/max(1.e-12,Ha),Sb/max(1.e-12,Hb),Ha,Hb,ai,bi)
        a[i]=na;b[i]=nb
        for j in range(n):
            ko=int(ids[j]);q=(-na if ko==0 else -nb if ko==1 else nb if ko==2 else na)
            ev=int(max(-7,min(7,np.rint((target[j]-q)/(1.+lam)/ss*7.))))+7
            oldbyte=int(packed[i,j//2]);packed[i,j//2]=((oldbyte&240)|ev) if j%2==0 else ((oldbyte&15)|(ev<<4))
            if ko!=int(k[i,j]):numflips+=1
            k[i,j]=ko
    return numflips


def activation_diag(layer,power=1.):
    """One-step, temporary input-energy diagonal. Approximate conv 3x3 patch stats.
    Critically: do not cache across steps (features move during training)."""
    n=layer.k.shape[1]
    x=getattr(layer,'_input',None)
    if x is None:return torch.ones(n,dtype=torch.float32,device=layer.k.device)
    with torch.no_grad():
        if x.ndim==2:v=x.float().square().mean(0)
        elif x.ndim==3:v=x.float().square().mean((0,1))
        elif x.ndim==4:
            v=x.float().square().mean((0,2,3))
            v=v.repeat_interleave(n//len(v))
        else:return torch.ones(n,dtype=torch.float32,device=layer.k.device)
        if v.numel()!=n:return torch.ones(n,dtype=torch.float32,device=layer.k.device)
        v=(v/v.mean().clamp_min(1e-8)).clamp(min=.2,max=5.).pow(power)
    return v.contiguous()

class ActivationPacked(PackedDiscreteProx):
    def __init__(self,model,lr=.1,lam=.05,beta=.9,kappa=.01,cap=4.,use_activation=True,activation_power=1.,**kw):
        kw.pop('bits',None)
        super().__init__(model,lr=lr,lam=lam,bits=4,**kw)
        self.beta=float(beta);self.kappa=float(kappa);self.cap=float(cap);self.use_activation=bool(use_activation);self.activation_power=float(activation_power)
    @torch.no_grad()
    def step(self):
        if self.model.direct_layers()[0].k.device.type!='cpu':
            # generic torch path with identical index/residual profiled choices
            return self._gpu_step()
        self.calls+=1;self.transitions=0
        for l in self.model.direct_layers():
            if l._w is None or l._w.grad is None:continue
            v=activation_diag(l,self.activation_power) if self.use_activation else torch.ones(l.k.shape[1],device=l.k.device)
            st=self.state[l]
            self.transitions+=_activation_step(l.k.numpy(),l.a.numpy(),l.b.numpy(),st['error'].numpy(),st['res_scale'].numpy(),l._w.grad.detach().contiguous().numpy(),v.cpu().numpy(),self.lr,self.lam,self.beta,self.kappa,self.cap)
    @torch.no_grad()
    def _gpu_step(self):
        # Torch execution, gradients never retained between steps, extra FP32 temps are temporary.
        from discrete_optim import decode, constrained_two_level_fit
        from packed_prox import _decode_4
        self.calls+=1; self.transitions=0
        for l in self.model.direct_layers():
            if l._w is None or l._w.grad is None:continue
            g=l._w.grad.detach();a=l.a;b=l.b;k=l.k
            v=activation_diag(l,self.activation_power) if self.use_activation else torch.ones(k.shape[1],device=k.device)
            rms=g.square().mean(1,keepdim=True).sqrt().clamp_min(1e-8)
            s=self.state[l]['res_scale'];old=decode(k,a,b)
            e0=_decode_4(self.state[l]['error'],k.shape[1])*s[:,None]/7
            t=old+self.beta*e0-self.lr*a[:,None]*(g/rms/v[None,:]).clamp(-self.cap,self.cap)
            q=torch.stack((-a,-b,b,a),dim=1)[:,:,None]
            eg=torch.round((t[:,None,:]-q)/(1+self.lam)/s[:,None,None]*7).clamp(-7,7)*s[:,None,None]/7
            co=(q+eg-t[:,None,:]).square()+self.lam*eg.square()
            idx=co.argmin(1).to(torch.uint8)
            ee=eg.gather(1,idx.long()[:,None,:]).squeeze(1)
            sa=torch.where(idx==0,-1.,torch.where(idx==3,1.,0.))
            sb=torch.where(idx==1,-1.,torch.where(idx==2,1.,0.))
            ha=(v[None,:]*sa.square()).sum(1)+self.kappa*k.shape[1]
            hb=(v[None,:]*sb.square()).sum(1)+self.kappa*k.shape[1]
            ua=((v[None,:]*sa*(t-ee)).sum(1)+self.kappa*k.shape[1]*a)/ha
            ub=((v[None,:]*sb*(t-ee)).sum(1)+self.kappa*k.shape[1]*b)/hb
            na,nb=constrained_two_level_fit(ua,ub,ha,hb,a,b)
            a.copy_(na);b.copy_(nb)
            re=torch.round((t-decode(idx,a,b))/(1+self.lam)/s[:,None]*7).clamp(-7,7)
            from packed_prox import _encode_4
            self.transitions+=int((idx!=k).sum().item());k.copy_(idx)
            self.state[l]['error'].copy_(_encode_4(re))

@njit(cache=True)
def _fitter_rank1(Ha,Hb,Sa,Sb,ua,ub,U,V,c,gam,old_a,old_b):
    A=Ha+gam*U*U;B=Hb+gam*V*V;C=gam*U*V
    p=Sa+gam*U*c;q=Sb+gam*V*c
    det=A*B-C*C
    mn=.006;mx=2.;low=.002;gap=.004
    best=1.e100;xa=old_a;xb=old_b
    for mode in range(5):
        if mode==0:
            if det<=1e-16:continue
            a=(p*B-q*C)/det;b=(q*A-p*C)/det
        elif mode==1:
            a=mn;b=max(low,min(mn-gap,(q-C*a)/max(B,1e-16)))
        elif mode==2:
            a=mx;b=max(low,min(mx-gap,(q-C*a)/max(B,1e-16)))
        elif mode==3:
            b=low;a=max(max(mn,b+gap),min(mx,(p-C*b)/max(A,1e-16)))
        else:
            b=(p+q-(A+C)*gap)/max(1e-16,(A+B+2*C))
            b=max(max(low,mn-gap),min(mx-gap,b));a=b+gap
        if a<mn-1e-10 or a>mx+1e-10 or b<low-1e-10 or b>a-gap+1e-10:continue
        val=.5*A*a*a+.5*B*b*b+C*a*b-p*a-q*b
        if val<best:best=val;xa=a;xb=b
    return xa,xb

@njit(cache=True)
def _meanbalance_update(k,a,b,packed,scale,g,mu,lr,lam,beta,kappa,gamma,dc_fraction):
    m,n=k.shape; nflip=0
    # μ normalized to unit Euclidean norm. penalty γ*n/2 (μᵀ(qnew-qanchor))².
    gg=gamma*n
    for i in range(m):
        ai=float(a[i]);bi=float(b[i]);ss=float(scale[i]);rms=1e-8
        for j in range(n):rms+=float(g[i,j])**2
        rms=math.sqrt(rms/n)
        target=np.empty(n,dtype=np.float64)
        ix=np.empty(n,dtype=np.uint8)
        er=np.empty(n,dtype=np.float64)
        orig=np.empty(n,dtype=np.float64)
        oldmean=0.;desiredmean=0.
        for j in range(n):
            ko=int(k[i,j]);w=(-ai if ko==0 else -bi if ko==1 else bi if ko==2 else ai)
            ecode=(int(packed[i,j//2])&15) if j%2==0 else (int(packed[i,j//2])>>4)
            e=(ecode-7)*ss/7
            tar=w+beta*e-lr*ai*max(-4.,min(4.,float(g[i,j])/rms))
            target[j]=tar;orig[j]=w;ix[j]=ko;er[j]=e
            oldmean+=float(mu[j])*w;desiredmean+=float(mu[j])*tar
        dest=(1-dc_fraction)*oldmean+dc_fraction*desiredmean
        curmean=oldmean
        for j in range(n):
            co=int(ix[j]);wold=orig[j];muj=float(mu[j])
            curminus=curmean-muj*wold
            best=1e100;kkbest=co;ebest=0.;qbest=wold
            for kk in range(4):
                q=(-ai if kk==0 else -bi if kk==1 else bi if kk==2 else ai)
                ec=int(max(-7,min(7,np.rint((target[j]-q)/(1+lam)/ss*7.))))
                e=ec*ss/7.
                cost=(q+e-target[j])**2+lam*e*e+gg*(curminus+muj*q-dest)**2
                if cost<best:best=cost;kkbest=kk;ebest=e;qbest=q
            curmean=curminus+muj*qbest
            ix[j]=kkbest;er[j]=ebest
            orig[j]=qbest
        Ha=kappa*n;Hb=kappa*n;Sa=kappa*n*ai;Sb=kappa*n*bi;U=0.;V=0.
        for j in range(n):
            ij=int(ix[j]);t=target[j]-er[j];u=float(mu[j])
            if ij==0:Ha+=1;Sa-=t;U-=u
            elif ij==3:Ha+=1;Sa+=t;U+=u
            elif ij==1:Hb+=1;Sb-=t;V-=u
            else:Hb+=1;Sb+=t;V+=u
        na,nb=_fitter_rank1(Ha,Hb,Sa,Sb,0.,0.,U,V,dest,gg,ai,bi)
        a[i]=na;b[i]=nb
        for j in range(n):
            ij=int(ix[j]);q=(-na if ij==0 else -nb if ij==1 else nb if ij==2 else na)
            ev=int(max(-7,min(7,np.rint((target[j]-q)/(1+lam)/ss*7.))))+7
            bb=int(packed[i,j//2]);packed[i,j//2]=((bb&240)|ev) if j%2==0 else ((bb&15)|(ev<<4))
            if ij!=int(k[i,j]):nflip+=1
            k[i,j]=ij
    return nflip

class MeanBalancedPacked(PackedDiscreteProx):
    """Rank-one activation-mean response penalty on legal discrete transitions.
    CPU reference; GPU fallback raises to prevent silently running a different method.
    State = packed INT4 residual + codebook scales, no full curvature state.
    """
    def __init__(self,model,lr=.15,lam=.05,beta=.85,kappa=.0,gamma=.5,dc_fraction=0.,**kw):
        kw.pop('bits',None)
        super().__init__(model,lr=lr,lam=lam,bits=4,**kw)
        self.beta=beta;self.kappa=kappa;self.gamma=gamma;self.dc_fraction=dc_fraction
    @torch.no_grad()
    def step(self):
        if self.model.direct_layers()[0].k.device.type!='cpu':
            raise RuntimeError('MeanBalancedPacked currently has CPU-only exact conditional implementation')
        self.calls+=1;self.transitions=0
        for l in self.model.direct_layers():
            if l._w is None or l._w.grad is None:continue
            x=getattr(l,'_input',None);n=l.k.shape[1]
            if x is not None:
                if x.ndim==2:mu=x.mean(0)
                elif x.ndim==4:mu=x.mean((0,2,3)).repeat_interleave(n//x.shape[1])
                else:mu=torch.zeros(n)
                if mu.numel()!=n:mu=torch.zeros(n)
                mu=mu/(mu.norm().clamp_min(1e-6))
            else:mu=torch.zeros(n)
            st=self.state[l]
            self.transitions+=_meanbalance_update(l.k.numpy(),l.a.numpy(),l.b.numpy(),st['error'].numpy(),st['res_scale'].numpy(),l._w.grad.detach().contiguous().numpy(),mu.contiguous().numpy(),self.lr,self.lam,self.beta,self.kappa,self.gamma,self.dc_fraction)

@njit(cache=True)
def _fitter_mat2(A,B,C,p,q,olda,oldb):
    mn=.006;mx=2.;low=.002;gap=.004
    det=A*B-C*C;best=1.e100;xa=olda;xb=oldb
    for mode in range(5):
        if mode==0:
            if det<=1e-16:continue
            a=(p*B-q*C)/det;b=(q*A-p*C)/det
        elif mode==1:
            a=mn;b=max(low,min(mn-gap,(q-C*a)/max(B,1e-16)))
        elif mode==2:
            a=mx;b=max(low,min(mx-gap,(q-C*a)/max(B,1e-16)))
        elif mode==3:
            b=low;a=max(max(mn,b+gap),min(mx,(p-C*b)/max(A,1e-16)))
        else:
            b=(p+q-(A+C)*gap)/max(1e-16,(A+B+2*C))
            b=max(max(low,mn-gap),min(mx-gap,b));a=b+gap
        if a<mn-1e-10 or a>mx+1e-10 or b<low-1e-10 or b>a-gap+1e-10:continue
        val=.5*A*a*a+.5*B*b*b+C*a*b-p*a-q*b
        if val<best:best=val;xa=a;xb=b
    return xa,xb

@njit(cache=True)
def _outchannel_step(k,a,b,packed,scale,g,lr,lam,beta,kappa,gamma):
    """Finite 4-choice Gauss-Seidel for sum_ij proximity + gamma*n/2 * ||u^T(Q-Q_old)||².
    Coupled in i for fixed input coordinate j, with u_i=1/sqrt(m).
    Codebook a/b 2D exact conditional solves full quadratic for one row at a time.
    """
    m,n=k.shape;norm=1./math.sqrt(m);coup=gamma*m;nf=0
    oldw=np.empty((m,n),dtype=np.float64);target=np.empty((m,n),dtype=np.float64)
    e=np.empty((m,n),dtype=np.float64);qq=np.empty((m,n),dtype=np.float64)
    knew=k.copy();dc=np.zeros(n,dtype=np.float64)
    for i in range(m):
        ai=float(a[i]);bi=float(b[i]);ss=float(scale[i]);rms=1e-8
        for j in range(n):rms+=float(g[i,j])**2
        rms=math.sqrt(rms/n)
        for j in range(n):
            kk=int(k[i,j]);w=(-ai if kk==0 else -bi if kk==1 else bi if kk==2 else ai)
            ec=(int(packed[i,j//2])&15) if j%2==0 else (int(packed[i,j//2])>>4)
            er=(ec-7)*ss/7
            oldw[i,j]=w;qq[i,j]=w
            target[i,j]=w+beta*er-lr*ai*max(-4.,min(4.,float(g[i,j])/rms))
            e[i,j]=er
    # One exact coordinate sweep on joint discrete (fixed a,b) objective.
    for j in range(n):
        s=0.
        for i in range(m):s+=norm*(qq[i,j]-oldw[i,j])
        for i in range(m):
            ai=float(a[i]);bi=float(b[i]);ss=float(scale[i]);t=target[i,j]
            rem=s-norm*(qq[i,j]-oldw[i,j]);best=1.e100;kn=int(knew[i,j]);qb=qq[i,j];eb=e[i,j]
            for z in range(4):
                v=(-ai if z==0 else -bi if z==1 else bi if z==2 else ai)
                ev=int(max(-7,min(7,np.rint((t-v)/(1+lam)/ss*7.))))*ss/7.
                co=(v+ev-t)**2+lam*ev*ev+coup*(rem+norm*(v-oldw[i,j]))**2
                if co<best:best=co;kn=z;qb=v;eb=ev
            s=rem+norm*(qb-oldw[i,j]);qq[i,j]=qb;e[i,j]=eb;knew[i,j]=kn
    # Exact conditional rowwise 2x2 codebook subproblem including every output coupling term.
    for i in range(m):
        ai=float(a[i]);bi=float(b[i]);A=kappa*n;B=kappa*n;C=0.;p=kappa*n*ai;q=kappa*n*bi
        for j in range(n):
            z=int(knew[i,j]);u=-1. if z==0 else 1. if z==3 else 0.
            v=-1. if z==1 else 1. if z==2 else 0.
            t=target[i,j]-e[i,j]
            A+=u*u;B+=v*v;p+=u*t;q+=v*t
            s=0.
            for ii in range(m):
                if ii!=i:s+=norm*(qq[ii,j]-oldw[ii,j])
            r=s-norm*oldw[i,j]
            # gamma*m/2*(r + norm*(u*a+v*b))^2
            A+=coup*norm*norm*u*u;B+=coup*norm*norm*v*v
            C+=coup*norm*norm*u*v
            p-=coup*norm*u*r;q-=coup*norm*v*r
        na,nb=_fitter_mat2(A,B,C,p,q,ai,bi);a[i]=na;b[i]=nb
        for j in range(n):
            z=int(knew[i,j]);qq[i,j]=(-na if z==0 else -nb if z==1 else nb if z==2 else na)
    for i in range(m):
        ss=float(scale[i])
        for j in range(n):
            ev=int(max(-7,min(7,np.rint((target[i,j]-qq[i,j])/(1+lam)/ss*7.))))+7
            bb=int(packed[i,j//2]);packed[i,j//2]=((bb&240)|ev) if j%2==0 else ((bb&15)|(ev<<4))
            if int(knew[i,j])!=int(k[i,j]):nf+=1
            k[i,j]=knew[i,j]
    return nf

class OutputBalancedPacked(PackedDiscreteProx):
    def __init__(self,model,lr=.15,lam=.05,beta=.85,kappa=.015,gamma=.1,**kw):
        kw.pop('bits',None);super().__init__(model,lr=lr,lam=lam,bits=4,**kw)
        self.beta=beta;self.kappa=kappa;self.gamma=gamma
    @torch.no_grad()
    def step(self):
        if self.model.direct_layers()[0].k.device.type!='cpu':
            raise RuntimeError('CPU conditional solver only; no silently different GPU fallback')
        self.calls+=1;self.transitions=0
        for l in self.model.direct_layers():
            if l._w is None or l._w.grad is None:continue
            st=self.state[l]
            self.transitions+=_outchannel_step(l.k.numpy(),l.a.numpy(),l.b.numpy(),st['error'].numpy(),st['res_scale'].numpy(),l._w.grad.detach().contiguous().numpy(),self.lr,self.lam,self.beta,self.kappa,self.gamma)


Overwriting stage30_src/stage22_optim.py


In [ ]:
%%writefile stage30_src/stage26_optim.py
"""Stage26: deterministic hard-INT2 + packed-INT4 optimizer candidates.

Conditional solves are exact for fixed codebook magnitudes.  The alternation
with the row codebook is NOT a global minimization of neural-network loss.
These are research prototypes: uint8 symbol buffers and temporary FP32 weights
are NOT a fused hardware INT2 implementation.
"""
import math
import torch
from stage22_optim import ActivationPacked, activation_diag
from discrete_optim import decode, constrained_two_level_fit
from packed_prox import _decode_4, _encode_4

class ScheduledPacked(ActivationPacked):
    """Activation-aware, proximal index + INT4 residual with cosine relaxation.

    For fixed row magnitudes, minimize over k in {0,1,2,3} and e in INT4:
      (q(k)+e-T)^2 + lam*e^2,
    where T = W + beta*E - lr(t)*a*clip(g/(row_RMS * activation_energy)).
    Conditional magnitudes solve weighted 2-variable least-squares, with
    proximal anchor fixed to the pre-update magnitudes.
    """
    def __init__(self, model, total_steps=100, floor=0.20, **kwargs):
        super().__init__(model, **kwargs)
        self.base_lr=float(self.lr)
        self.total_steps=max(1,int(total_steps))
        self.floor=float(floor)
    @torch.no_grad()
    def step(self):
        x=min(self.calls,self.total_steps-1)/max(1,self.total_steps-1)
        self.lr=self.base_lr*(self.floor+(1-self.floor)*.5*(1+math.cos(math.pi*x)))
        return super().step()
    @torch.no_grad()
    def _gpu_step(self):
        """Vectorized tensor implementation with no per-layer .item() synchronization."""
        self.calls+=1
        self.transitions=0
        for layer in self.model.direct_layers():
            if layer._w is None or layer._w.grad is None:continue
            g=layer._w.grad.detach();a=layer.a;b=layer.b;k=layer.k
            v=(activation_diag(layer,self.activation_power) if self.use_activation
               else torch.ones(k.shape[1],device=k.device))
            rms=g.square().mean(1,keepdim=True).sqrt().clamp_min(1e-8)
            state=self.state[layer]
            scale=state['res_scale']
            e0=_decode_4(state['error'],k.shape[1])*scale[:,None]/7.
            target=decode(k,a,b)+self.beta*e0-self.lr*a[:,None]*(g/rms/v[None,:]).clamp(-self.cap,self.cap)
            levels=torch.stack((-a,-b,b,a),dim=1)[:,:,None]
            ecode=torch.round((target[:,None,:]-levels)/(1+self.lam)/scale[:,None,None]*7).clamp(-7,7)
            ecand=ecode*scale[:,None,None]/7
            cost=(levels+ecand-target[:,None,:]).square()+self.lam*ecand.square()
            idx=cost.argmin(dim=1).to(torch.uint8)
            ee=ecand.gather(1,idx.long()[:,None,:]).squeeze(1)
            sa=torch.where(idx==0,-1.,torch.where(idx==3,1.,0.))
            sb=torch.where(idx==1,-1.,torch.where(idx==2,1.,0.))
            h_a=(v[None,:]*sa.square()).sum(1)+self.kappa*k.shape[1]
            h_b=(v[None,:]*sb.square()).sum(1)+self.kappa*k.shape[1]
            ua=((v[None,:]*sa*(target-ee)).sum(1)+self.kappa*k.shape[1]*a)/h_a
            ub=((v[None,:]*sb*(target-ee)).sum(1)+self.kappa*k.shape[1]*b)/h_b
            new_a,new_b=constrained_two_level_fit(ua,ub,h_a,h_b,a,b)
            a.copy_(new_a);b.copy_(new_b)
            new_e=torch.round((target-decode(idx,a,b))/(1+self.lam)/scale[:,None]*7).clamp(-7,7)
            k.copy_(idx)
            state['error'].copy_(_encode_4(new_e))

class SplitCodebookPacked(ScheduledPacked):
    """Alternating discrete and row-codebook-gradient proximal blocks.

    Block A: exact conditional k,e minimization of the packed proximal target.
    Block B: exact 2D row-constrained minimization of a separate local loss
      linearization <g, da*sa + db*sb>+ha/2*da^2+hb/2*db^2,
      where sa,sb are the index-specific signed indicators and ha,hb>0.
    These two blocks DO NOT share one globally optimized joint objective; the
    method is an alternating-surrogate research candidate (Stage25 design).
    """
    def __init__(self,model,code_eta=.04,scale_mode='count',**kw):
        super().__init__(model,**kw)
        self.code_eta=float(code_eta)
        self.scale_mode=str(scale_mode)
    @torch.no_grad()
    def step(self):
        x=min(self.calls,self.total_steps-1)/max(1,self.total_steps-1)
        self.lr=self.base_lr*(self.floor+(1-self.floor)*.5*(1+math.cos(math.pi*x)))
        self.calls+=1
        self.transitions=0
        for layer in self.model.direct_layers():
            if layer._w is None or layer._w.grad is None:continue
            g=layer._w.grad.detach();a=layer.a;b=layer.b;k=layer.k
            v=(activation_diag(layer,self.activation_power) if self.use_activation
               else torch.ones(k.shape[1],device=k.device))
            rms=g.square().mean(1,keepdim=True).sqrt().clamp_min(1e-8)
            state=self.state[layer];scale=state['res_scale']
            e0=_decode_4(state['error'],k.shape[1])*scale[:,None]/7
            target=decode(k,a,b)+self.beta*e0-self.lr*a[:,None]*(g/rms/v[None,:]).clamp(-self.cap,self.cap)
            levels=torch.stack((-a,-b,b,a),dim=1)[:,:,None]
            ecode=torch.round((target[:,None,:]-levels)/(1+self.lam)/scale[:,None,None]*7).clamp(-7,7)
            ecand=ecode*scale[:,None,None]/7
            cost=(levels+ecand-target[:,None,:]).square()+self.lam*ecand.square()
            idx=cost.argmin(1).to(torch.uint8)
            sa=torch.where(idx==0,-1.,torch.where(idx==3,1.,0.))
            sb=torch.where(idx==1,-1.,torch.where(idx==2,1.,0.))
            ga=(sa*g).sum(1);gb=(sb*g).sum(1)
            n_a=sa.square().sum(1);n_b=sb.square().sum(1)
            if self.scale_mode=='sqrt':
                n_a=n_a.sqrt().clamp_min(1);n_b=n_b.sqrt().clamp_min(1)
            else:
                n_a=n_a.clamp_min(1);n_b=n_b.clamp_min(1)
            ha=rms.squeeze(1)*n_a/(self.code_eta*a.clamp_min(.006))
            hb=rms.squeeze(1)*n_b/(self.code_eta*a.clamp_min(.006))
            newa,newb=constrained_two_level_fit(a-ga/ha,b-gb/hb,ha,hb,a,b)
            a.copy_(newa);b.copy_(newb)
            newe=torch.round((target-decode(idx,a,b))/(1+self.lam)/scale[:,None]*7).clamp(-7,7)
            k.copy_(idx)
            state['error'].copy_(_encode_4(newe))


Overwriting stage30_src/stage26_optim.py


In [ ]:
%%writefile stage30_src/stage26_suite.py
"""Stage26 reproducible loss-vs-updates and step-runtime experiment harness.

Every task uses legal 2-bit forward weights in BOTH arms; Muon-STE uses FP32
master matrices, while discrete methods use packed INT4 residual state.
Method selection uses development-only datasets/seeds. Held-out seeds are
independent problem instances with the same sample/dataset/initialization for
all compared methods. No validation loss is involved anywhere.
"""
import os,math,time,json,random,hashlib,zipfile
from pathlib import Path
from itertools import product
from functools import lru_cache
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
import large_suite as old
from discrete_optim import DiscreteLinear,STELinear,decode,quantize_shadow
from stage26_optim import ScheduledPacked,SplitCodebookPacked

TASKS={
 'mlp_robust':dict(arch='mlp',task='robust',n=1200,input_dim=48,width=128,depth=3,out=6,batch=96,steps=120),
 'resmlp_quantile':dict(arch='resmlp',task='quantile',n=1200,input_dim=48,width=128,depth=4,out=6,batch=96,steps=120),
 'resmlp_multiclass':dict(arch='resmlp',task='multiclass',n=1200,input_dim=48,width=128,depth=4,out=6,batch=96,steps=120),
 'resmlp_focal':dict(arch='resmlp',task='focal',n=1200,input_dim=48,width=128,depth=4,out=6,batch=96,steps=120),
 'resmlp_poisson':dict(arch='resmlp',task='poisson',n=1200,input_dim=48,width=128,depth=4,out=6,batch=96,steps=120),
 'rnn_bce':dict(arch='rnn',task='bce',n=1000,input_dim=12,width=72,depth=1,out=6,batch=64,steps=120,sequence_length=14),
 'rnn_robust':dict(arch='rnn',task='robust',n=1000,input_dim=12,width=72,depth=1,out=6,batch=64,steps=120,sequence_length=14),
 'cnn_shapes':dict(arch='cnn',task='multiclass',n=1200,channels=1,width=16,out=4,batch=64,steps=120),
 'transformer_128':dict(arch='transformer',task='nexttoken',n=1280,vocab=40,width=128,depth=4,heads=4,seq=32,batch=24,steps=120),
 'transformer_192':dict(arch='transformer',task='nexttoken',n=1536,vocab=48,width=192,depth=4,heads=6,seq=48,batch=16,steps=120),
 'transformer_256_stress':dict(arch='transformer',task='nexttoken',n=2048,vocab=64,width=256,depth=6,heads=8,seq=64,batch=12,steps=140),
}
old.PRESETS.update({k:v for k,v in TASKS.items() if v['arch']!='transformer'})
old.make_dataset.cache_clear()

class DiscreteEmbedding(DiscreteLinear):
    def forward(self,ids):
        w=decode(self.k,self.a,self.b).detach().requires_grad_(True)
        self._w=w
        return F.embedding(ids,w)
class STEEmbedding(STELinear):
    def forward(self,ids):
        q,_=quantize_shadow(self.shadow,self.a,self.b)
        w=self.shadow+(q-self.shadow).detach()+(q-q.detach())
        return F.embedding(ids,w)

class CausalBlock(nn.Module):
    def __init__(self,width,heads,add):
        super().__init__();self.width=width;self.heads=heads
        self.q=add(width,width);self.k=add(width,width);self.v=add(width,width);self.o=add(width,width)
        self.ff1=add(width,width*2);self.ff2=add(width*2,width)
    def forward(self,x):
        B,T,D=x.shape
        z=F.layer_norm(x,(D,))
        def heads(layer):return layer(z).reshape(B,T,self.heads,D//self.heads).transpose(1,2)
        attn=F.scaled_dot_product_attention(heads(self.q),heads(self.k),heads(self.v),is_causal=True,dropout_p=0.)
        x=x+self.o(attn.transpose(1,2).reshape(B,T,D))/math.sqrt(2)
        z=F.layer_norm(x,(D,))
        return x+self.ff2(F.gelu(self.ff1(z)))/math.sqrt(2)

class QuantTransformer(nn.Module):
    def __init__(self,cfg,seed,method,device):
        super().__init__();self.is_transformer=True
        vocab=cfg['vocab'];width=cfg['width'];depth=cfg['depth'];heads=cfg['heads']
        g=torch.Generator().manual_seed(seed)
        L=DiscreteLinear if method!='muon' else STELinear
        E=DiscreteEmbedding if method!='muon' else STEEmbedding
        self.layers=nn.ModuleList()
        def add(nin,nout,embedding=False):
            k=torch.randint(0,4,(nout,nin),generator=g,dtype=torch.uint8)
            a=torch.full((nout,),min(.7,1.2/math.sqrt(nin)),dtype=torch.float32)
            l=(E if embedding else L)(k,a,a/3,device=device)
            self.layers.append(l)
            return l
        self.emb=add(width,vocab,True)
        self.blocks=nn.ModuleList([CausalBlock(width,heads,add) for _ in range(depth)])
        self.head=add(width,vocab)
        pos=torch.arange(512).float()[:,None]
        freqs=torch.exp(-math.log(10000)*torch.arange(0,width,2).float()/width)
        pe=torch.zeros(512,width)
        pe[:,0::2]=torch.sin(pos*freqs);pe[:,1::2]=torch.cos(pos*freqs)
        self.register_buffer('posenc',pe)
        self.width=width;self.vocab=vocab
    def forward(self,tokens):
        x=self.emb(tokens)*math.sqrt(self.width)
        x=x+self.posenc[:tokens.shape[1]]
        for block in self.blocks:x=block(x)
        return self.head(F.layer_norm(x,(self.width,)))
    def direct_layers(self):return list(self.layers)
    def ste_groups(self):return [l.shadow for l in self.layers],[p for l in self.layers for p in (l.a,l.b)]

@lru_cache(maxsize=24)
def tokens_data(seed,task_name):
    c=TASKS[task_name];n=c['n'];T=c['seq'];v=c['vocab']
    r=torch.Generator().manual_seed(10019+seed)
    z=torch.zeros(n,T+1,dtype=torch.long)
    z[:,0]=torch.randint(0,v,(n,),generator=r)
    mode=torch.randint(1,5,(n,),generator=r)
    for t in range(T):
        lead=z[:,t];lag=z[:,t-1] if t>0 else z[:,0]
        nxt=(lead+mode+(lag%4))%v
        noisy=torch.rand(n,generator=r)<.22
        nxt[noisy]=torch.randint(0,v,(int(noisy.sum()),),generator=r)
        z[:,t+1]=nxt
    return z[:,:-1],z[:,1:]

@lru_cache(maxsize=30)
def dataset(task,seed):
    if TASKS[task]['arch']=='transformer':return tokens_data(seed,task)
    return old.make_dataset(task,seed)

def model_for(task,method,seed,device):
    c=TASKS[task]
    if c['arch']=='transformer':return QuantTransformer(c,seed,'muon' if method=='muon' else 'direct',device).to(device)
    return old.make_model(task,'muon' if method=='muon' else 'dq4',seed,device)

class MuonSTE:
    """Official torch.optim.Muon on CUDA; FP32-equivalent CPU smoke reference.
    Two settings are swept: Muon for all matrix weights, or Muon for hidden
    matrices with Adam for embedding/head as the PyTorch docs recommend.
    Codebook scalars use Adam regardless of head setting.
    """
    def __init__(self,model,cfg,device):
        layers=list(model.layers)
        master=[layer.shadow for layer in layers]
        head_adam=cfg['head_mode']=='adam'
        is_trans=getattr(model,'is_transformer',False)
        # Always put embedding in Adam for hybrid. All-Muon is also swept.
        excluded=([0,len(layers)-1] if is_trans else [len(layers)-1]) if head_adam else []
        muon_idx=[i for i in range(len(layers)) if i not in excluded]
        assert len(muon_idx)>0
        groups=[]
        mat=[master[i] for i in muon_idx if i!=len(layers)-1]
        if mat:groups.append({'params':mat,'lr':cfg['lr']})
        if len(layers)-1 in muon_idx:
            groups.append({'params':[master[-1]],'lr':cfg['lr']*cfg['head_lr_factor']})
        if str(device).startswith('cuda') and hasattr(torch.optim,'Muon'):
            self.muon=torch.optim.Muon(groups,momentum=cfg['momentum'],weight_decay=0.,adjust_lr_fn=cfg['adjust_lr_fn'])
        else:
            from muon_cpu import CPUFloat32Muon
            if str(device).startswith('cuda'):
                print('WARNING: torch.optim.Muon missing; using FP32 PyTorch-ops Muon reference on GPU. For official Muon timings, upgrade PyTorch.',flush=True)
            self.muon=CPUFloat32Muon(groups,momentum=cfg['momentum'],weight_decay=0.,adjust_lr_fn=cfg['adjust_lr_fn'])
        code=[p for l in layers for p in (l.a,l.b)]
        aux=[{'params':code,'lr':cfg['code_lr']}]
        if excluded:aux.append({'params':[master[i] for i in excluded],'lr':cfg['head_lr']})
        self.adam=torch.optim.Adam(aux)
        self.model=model
    def zero_grad(self):
        self.muon.zero_grad(set_to_none=True);self.adam.zero_grad(set_to_none=True)
    @torch.no_grad()
    def step(self):
        self.muon.step();self.adam.step()
        for l in self.model.layers:l.clamp_codebook()
    def state_dict(self):return {'muon':self.muon.state_dict(),'adam':self.adam.state_dict()}
    def load_state_dict(self,state):self.muon.load_state_dict(state['muon']);self.adam.load_state_dict(state['adam'])


def setup_optimizer(model,method,cfg,device,steps):
    if method=='muon':return MuonSTE(model,cfg,device)
    params=dict(lr=cfg['lr'],lam=cfg['lam'],beta=cfg['beta'],floor=cfg['floor'],
                kappa=cfg.get('kappa',.01),use_activation=cfg.get('use_activation',True),
                activation_power=cfg.get('activation_power',1.),total_steps=steps)
    if method=='scheduled_int4':return ScheduledPacked(model,**params)
    if method=='codebook_block':
        params['code_eta']=cfg['code_eta'];params['scale_mode']=cfg.get('scale_mode','count')
        return SplitCodebookPacked(model,**params)
    raise ValueError(method)

def loss_for(model,x,y,task):
    c=TASKS[task]
    if c['arch']=='transformer':
        logits=model(x)
        return F.cross_entropy(logits.reshape(-1,c['vocab']),y.reshape(-1))
    if c['task']=='ranking':return old.objective((model(x[:,0])-model(x[:,1])).squeeze(-1),y,c['task'])
    return old.objective(model(x),y,c['task'])

def sync(device):
    if str(device).startswith('cuda'):torch.cuda.synchronize(device)

@torch.no_grad()
def full_loss(model,X,Y,task,device):
    c=TASKS[task];chunk=(32 if c['arch']=='transformer' else 256)
    acc=0.
    for start in range(0,len(X),chunk):
        x=X[start:start+chunk].to(device,non_blocking=True)
        y=Y[start:start+chunk].to(device,non_blocking=True)
        val=loss_for(model,x,y,task)
        acc+=float(val.detach().item())*len(x)
    return acc/len(X)

def optimizer_snapshot(opt,model,method):
    if method=='muon':return opt.state_dict()
    # Layer identifiers do not survive model deserialization; save by order.
    return {'calls':opt.calls,'state':[
        {name:v.detach().cpu() for name,v in opt.state[layer].items() if torch.is_tensor(v)}
        for layer in model.direct_layers()]}

def restore_optimizer(opt,model,method,state,device):
    if method=='muon':opt.load_state_dict(state);return
    opt.calls=int(state['calls'])
    for layer,entry in zip(model.direct_layers(),state['state']):
        for name,value in entry.items():
            opt.state[layer][name].copy_(value.to(device))

def run_key(task,method,cfg,seed,steps,phase,device,batch):
    blob=json.dumps(dict(task=task,method=method,cfg=cfg,seed=seed,steps=steps,phase=phase,
                         device=device,batch=batch,version='Stage26.1'),sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()[:24]

def train(task,method,cfg,seed,steps,device,out,phase='heldout',eval_every=16,resume=True,checkpoint_every=32):
    out=Path(out);c=TASKS[task];batch=c['batch']
    folder=out/'runs';folder.mkdir(parents=True,exist_ok=True)
    key=run_key(task,method,cfg,seed,steps,phase,device,batch)
    path=folder/f'{key}.json';checkpoint=(out/'checkpoints'/f'{key}.pt')
    if path.exists():
        content=json.loads(path.read_text());return content['result'],content['curve']
    torch.manual_seed(seed+17000)
    X,Y=dataset(task,seed)
    model=model_for(task,method,seed+9000,device)
    opt=setup_optimizer(model,method,cfg,device,steps)
    rng=torch.Generator().manual_seed(seed+9999)
    order=torch.randint(0,len(X),(steps,batch),generator=rng)
    elapsed=0.;eval_seconds=0.;start_step=0;curve=[]
    if resume and checkpoint.exists():
        state=torch.load(checkpoint,map_location='cpu',weights_only=False)
        if state['run_key']!=key:raise ValueError('Invalid checkpoint key')
        model.load_state_dict(state['model']);restore_optimizer(opt,model,method,state['opt'],device)
        if 'torch_rng' in state:torch.set_rng_state(state['torch_rng'])
        if str(device).startswith('cuda') and state.get('cuda_rng') is not None:
            torch.cuda.set_rng_state_all(state['cuda_rng'])
        start_step=state['step'];elapsed=state['train_s'];eval_seconds=state['eval_s'];curve=state['curve']
    for step in range(start_step,steps+1):
        if (step%eval_every==0 or step==steps or step==0) and (not curve or curve[-1]['step']!=step):
            sync(device);ev0=time.perf_counter()
            val=full_loss(model,X,Y,task,device)
            sync(device);eval_seconds+=time.perf_counter()-ev0
            if not np.isfinite(val):raise FloatingPointError(f'Nonfinite full training loss {task} {method} step {step}')
            curve.append(dict(task=task,method=method,seed=seed,step=step,
                              full_training_loss=val,training_seconds=elapsed,phase=phase))
        if step==steps:break
        ids=order[step]
        x=X[ids].to(device,non_blocking=True);y=Y[ids].to(device,non_blocking=True)
        sync(device);t0=time.perf_counter()
        if method=='muon':opt.zero_grad()
        else:model.zero_grad(set_to_none=True)
        train_loss=loss_for(model,x,y,task)
        if step%max(8,eval_every)==0 and not bool(torch.isfinite(train_loss.detach()).item()):
            raise FloatingPointError(f'Nonfinite minibatch loss {task} {method}')
        train_loss.backward();opt.step()
        sync(device);elapsed+=time.perf_counter()-t0
        if checkpoint_every and (step+1)%checkpoint_every==0:
            checkpoint.parent.mkdir(parents=True,exist_ok=True)
            torch.save(dict(run_key=key,step=step+1,model=model.state_dict(),
                opt=optimizer_snapshot(opt,model,method),train_s=elapsed,eval_s=eval_seconds,curve=curve,
                torch_rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all() if str(device).startswith('cuda') else None),checkpoint)
    y=np.array([r['full_training_loss'] for r in curve]);t=np.array([r['step'] for r in curve]);
    result=dict(task=task,method=method,phase=phase,seed=int(seed),steps=steps,
        cfg=json.dumps(cfg,sort_keys=True),initial=float(y[0]),final=float(y[-1]),
        loss_auc=float(np.trapezoid(y,t)/steps),ms_per_step=float(1000*elapsed/steps),
        train_seconds=float(elapsed),diagnostic_seconds=float(eval_seconds),device=str(device))
    path.write_text(json.dumps(dict(result=result,curve=curve),allow_nan=False))
    if checkpoint.exists():checkpoint.unlink()
    return result,curve

# Independent, intentionally broad method-specific parameter spaces.
MUON_GRID=[dict(lr=lr,code_lr=cr,head_lr=hr,head_lr_factor=hf,momentum=mo,
                adjust_lr_fn=adj,head_mode=mode)
           for lr,cr,hr,hf,mo,adj,mode in product((.025,.08,.20,.45),(.003,.015),(.003,.015),
                     (.5,1.5),(.90,.95),('original','match_rms_adamw'),('adam','muon'))]
PACKED_GRID=[dict(lr=lr,lam=lam,beta=beta,floor=floor,kappa=kappa,
                  use_activation=True,activation_power=ap)
             for lr,lam,beta,floor,kappa,ap in product((.05,.13,.30,.55),(.0,.1),(.75,.9),(.15,.35),(.005,.02),(.5,1.))]
BLOCK_GRID=[dict(lr=lr,lam=lam,beta=beta,floor=floor,code_eta=eta,
                 scale_mode='count',kappa=.01,use_activation=True,activation_power=ap)
            for lr,lam,beta,floor,eta,ap in product((.05,.13,.30,.55),(.0,.1),(.75,.9),(.15,.35),(.02,.07),(.5,1.))]
GRIDS={'muon':MUON_GRID,'scheduled_int4':PACKED_GRID,'codebook_block':BLOCK_GRID}
# Anchor configurations from previous development ranges; still selected ONLY on dev data.
ANCHORS={
'muon':[dict(lr=.15,code_lr=.01,head_lr=.01,head_lr_factor=1.,momentum=.95,
             adjust_lr_fn='match_rms_adamw',head_mode='adam'),
        dict(lr=.3,code_lr=.003,head_lr=.003,head_lr_factor=.5,momentum=.95,
             adjust_lr_fn='original',head_mode='muon')],
'scheduled_int4':[dict(lr=.18,lam=.05,beta=.9,floor=.2,kappa=.01,use_activation=True,activation_power=1.)],
'codebook_block':[dict(lr=.14,lam=.05,beta=.9,floor=.2,code_eta=.04,scale_mode='count',
                        kappa=.01,use_activation=True,activation_power=1.)]}
MODES={
 'smoke':dict(screen=1,screen_steps=2,refine=1,refine_steps=3,eval_steps=4,devseeds=(21,),evalseeds=(101,),eval_every=2),
 'quick':dict(screen=6,screen_steps=10,refine=2,refine_steps=28,eval_steps=80,devseeds=(21,31),evalseeds=(101,),eval_every=16),
 'balanced':dict(screen=10,screen_steps=22,refine=3,refine_steps=55,eval_steps=140,devseeds=(21,31),evalseeds=(101,102),eval_every=14),
 'extended':dict(screen=12,screen_steps=30,refine=4,refine_steps=90,eval_steps=220,devseeds=(21,31,32),evalseeds=(101,102,103),eval_every=20),
}

def configurations(method,n):
    """Stratified deterministic sweep spanning learning-rate ranges.
    One predetermined anchor plus quasi-random grid points; no held-out feedback.
    """
    pool=GRIDS[method]
    rng=random.Random(20261126+sum(map(ord,method)))
    sampled=[]
    used=set()
    def add(c):
        key=json.dumps(c,sort_keys=True)
        if key not in used:sampled.append(c);used.add(key)
    for a in ANCHORS[method]:add(a)
    # Cycle over distinct learning rates before filling with random factorial choices.
    shuffled=pool[:];rng.shuffle(shuffled)
    for lr in (.05,.13,.30,.55) if method!='muon' else (.025,.08,.20,.45):
        for c in shuffled:
            if c['lr']==lr:add(c);break
    for c in shuffled:
        if len(sampled)>=n:break
        add(c)
    return sampled[:max(n,len(ANCHORS[method]))]

def tune(task,method,mode,device,out):
    m=MODES[mode];candidates=configurations(method,m['screen'])
    screen=[]
    for c in candidates:
        r,_=train(task,method,c,m['devseeds'][0],m['screen_steps'],device,out,'screen',
                  eval_every=max(2,m['screen_steps']//3),checkpoint_every=0)
        screen.append((r['loss_auc'],c))
    screen.sort(key=lambda v:v[0])
    refine=[]
    for _,cfg in screen[:m['refine']]:
        losses=[]
        for seed in m['devseeds']:
            r,_=train(task,method,cfg,seed,m['refine_steps'],device,out,'refine',
                      eval_every=max(3,m['refine_steps']//5),checkpoint_every=0)
            losses.append(r['loss_auc'])
        refine.append((float(np.mean(losses)),cfg))
    refine.sort(key=lambda v:v[0])
    return refine[0][1],screen,refine

def _collect_results(out):
    rows=[];curves=[]
    for path in sorted((out/'runs').glob('*.json')):
        d=json.loads(path.read_text());rows.append(d['result']);curves.extend(d['curve'])
    df=pd.DataFrame(rows);cf=pd.DataFrame(curves)
    if len(df):df.to_csv(out/'all_runs.csv',index=False)
    if len(cf):cf.to_csv(out/'all_curves.csv',index=False)
    held=df[df.phase=='heldout'].copy() if len(df) else pd.DataFrame()
    if len(held):
        held.to_csv(out/'heldout.csv',index=False)
        summary=held.groupby(['task','method'],as_index=False)[['loss_auc','final','ms_per_step']].mean()
        summary.to_csv(out/'summary.csv',index=False)
        gates=[]
        for task,g in summary.groupby('task'):
            baseline=g[g.method=='muon']
            if len(baseline)!=1:continue
            mu=baseline.iloc[0]
            for _,r in g[g.method!='muon'].iterrows():
                gates.append(dict(task=task,method=r.method,
                    beats_muon_loss_auc=bool(r.loss_auc<mu.loss_auc),
                    beats_muon_ms_per_step=bool(r.ms_per_step<mu.ms_per_step),
                    passes_both=bool(r.loss_auc<mu.loss_auc and r.ms_per_step<mu.ms_per_step)))
        if gates:pd.DataFrame(gates).to_csv(out/'acceptance_gate.csv',index=False)
    return held,cf

def _plot(out,task,curves):
    import matplotlib.pyplot as plt
    c=curves[(curves.task==task)&(curves.phase=='heldout')]
    if not len(c):return
    fig,ax=plt.subplots(figsize=(7,4))
    for method,d in c.groupby('method'):
        pivot=d.pivot_table(index='step',columns='seed',values='full_training_loss')
        ax.plot(pivot.index,pivot.mean(axis=1),marker='.',label=method)
        if len(pivot.columns)>1:
            ax.fill_between(pivot.index,pivot.min(axis=1),pivot.max(axis=1),alpha=.12)
    ax.set(xlabel='Optimizer updates',ylabel='Full training loss',title=task)
    ax.grid(alpha=.2);ax.legend();fig.tight_layout();fig.savefig(out/f'{task}_loss.png',dpi=135);plt.close(fig)
    results=pd.read_csv(out/'heldout.csv') if (out/'heldout.csv').exists() else pd.DataFrame()
    r=results[results.task==task] if len(results) else pd.DataFrame()
    if len(r):
        means=r.groupby('method',as_index=False).ms_per_step.mean()
        fig,ax=plt.subplots(figsize=(6.5,3.4))
        ax.bar(means.method,means.ms_per_step)
        ax.set(ylabel='Forward + backward + optimizer (ms/step)',title=task)
        ax.tick_params(axis='x',labelrotation=12)
        fig.tight_layout();fig.savefig(out/f'{task}_runtime.png',dpi=135);plt.close(fig)

def run_suite(tasks,methods=('muon','codebook_block'),mode='quick',device='auto',output='stage26_results'):
    if 'muon' not in methods:raise ValueError('Muon baseline is mandatory')
    if mode not in MODES:raise ValueError(mode)
    for t in tasks:
        if t not in TASKS:raise ValueError(t)
    device=('cuda' if torch.cuda.is_available() else 'cpu') if device=='auto' else device
    if device.startswith('cuda') and not torch.cuda.is_available():raise RuntimeError('CUDA not available')
    torch.set_num_threads(min(2,torch.get_num_threads()))
    out=Path(output);out.mkdir(parents=True,exist_ok=True)
    protocol_path=out/'protocol.json'
    if protocol_path.exists():
        previous=json.loads(protocol_path.read_text())
        if previous.get('mode')!=mode or previous.get('device')!=device:
            raise ValueError('Existing output folder has a different MODE/device; use a new output folder')
    if device=='cpu':
        # Precompile the Numba CPU kernel outside the timed training updates.
        from stage22_optim import _activation_step
        k=np.array([[0,1,2,3]],dtype=np.uint8);a=np.array([.4],dtype=np.float32);b=a/3
        packed=np.full((1,2),0x77,dtype=np.uint8);scale=a.copy()
        g=np.array([[.1,-.2,.3,-.4]],dtype=np.float32);v=np.ones(4,dtype=np.float32)
        _activation_step(k,a,b,packed,scale,g,v,.1,.05,.9,.01,4.)
    (out/'protocol.json').write_text(json.dumps(dict(tasks=list(tasks),methods=list(methods),mode=mode,
        device=device,dev_seeds=MODES[mode]['devseeds'],eval_seeds=MODES[mode]['evalseeds'],
        metrics=['full training loss vs updates','forward+backward+optimizer ms/step'],
        timing='excludes diagnostic full-training evaluation, checkpoint saving and dataset generation',
        gpu_muon=('PyTorch official torch.optim.Muon' if hasattr(torch.optim,'Muon') else 'FP32 fallback Muon; upgrade PyTorch for official')+', head_mode independently swept',
        state_note='packed INT4 residual is O(P); symbols are uint8 (not physically INT2 packed); FP32 weights are temporary',
        hardware=torch.cuda.get_device_name(0) if device.startswith('cuda') else 'CPU'),indent=2))
    locked_path=out/'locked_configs.json'
    locked=json.loads(locked_path.read_text()) if locked_path.exists() else {}
    held=[]
    for task in tasks:
        locked.setdefault(task,{})
        for method in methods:
            if method not in locked[task]:
                cfg,screen,refine=tune(task,method,mode,device,out)
                locked[task][method]=cfg
                locked_path.write_text(json.dumps(locked,indent=2))
                print('LOCKED',task,method,cfg,'dev AUC',refine[0][0],flush=True)
            else:print('RESTORED',task,method,flush=True)
        for method in methods:
            for seed in MODES[mode]['evalseeds']:
                r,_=train(task,method,locked[task][method],seed,MODES[mode]['eval_steps'],device,
                           out,'heldout',eval_every=MODES[mode]['eval_every'],resume=True)
                held.append(r)
                print('EVAL',task,method,seed,'AUC',round(r['loss_auc'],5),
                      'ms/step',round(r['ms_per_step'],3),flush=True)
        hd,cr=_collect_results(out)
        if len(cr):_plot(out,task,cr)
    hd,cr=_collect_results(out)
    return hd,locked

def export_zip(folder='stage26_results',target='Stage26_Colab_Results.zip',download=True):
    folder=Path(folder);target=Path(target)
    if not folder.exists() or not any(folder.rglob('*.json')):
        raise FileNotFoundError('No run results. Execute the suite first.')
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in sorted(folder.rglob('*')):
            if f.is_file():z.write(f,arcname=str(Path(folder.name)/f.relative_to(folder)))
    assert target.is_file() and target.stat().st_size>100
    print('Created',str(target.resolve()),'bytes',target.stat().st_size,flush=True)
    if download:
        try:
            from google.colab import files
            files.download(str(target.resolve()))
        except ImportError:
            print('Not inside Colab; archive is saved locally.')
    return target


Overwriting stage30_src/stage26_suite.py


In [ ]:
%%writefile stage30_src/stage27_fast.py
"""Stage 27 CPU-fused conditional 2-bit weight / residual optimizer.

Residual variants under exactly the same per-entry proximal objective:
  fast0: no residual, 0 persistent bits per weight;
  fast2: ternary residual, packed 2 bits (legacy; half-width range);
  fast4: signed 15-level grid, packed 4 bits;
  fast8: signed 255-level grid, 8 bits per weight (torch.int8);
  fast16: IEEE 754 binary16 residual, 16 bits per weight (torch.float16).

The 4-, 8-, and 16-bit variants share the fixed row-specific support [-s_i,s_i],
where s_i is initialized with Stage26's row scale. This isolates precision from
range. An integer's optimal conditional residual is nearest to (target-q)/
(1+lambda); for binary16 the same convex quadratic is projected to a binary16
number in [-s,s]. Integer variants clamp their symmetric quantized grids.

Codebook block separately minimizes Stage25's row gradient-proximal 2D surrogate
conditional on selected indices. These are distinct alternating objectives,
not an exact global minimization of training loss. The persistent FP16 tensor is
used directly: Numba operates on a zero-copy uint16 view of its IEEE bits.

The 2-bit weight symbols remain uint8 (not packed). Temporary FP32 realized
weights and autograd gradients still affect peak RAM. CPU only.
"""
import math
import numpy as np
import torch
from numba import njit
from stage26_optim import SplitCodebookPacked
from stage22_optim import activation_diag
from packed_prox_numba import _fitter

@njit(cache=True)
def _half_to_float(bits):
    """IEEE binary16 -> float32 using a scalar bitcast (no array copying)."""
    v=int(bits)
    sign32=(v & 0x8000)<<16
    exponent=(v >> 10) & 31
    fraction=v & 1023
    if exponent==0:
        if fraction==0:
            return float(np.uint32(sign32).view(np.float32))
        # Half subnormals are rare, and this branch is exactly representable.
        sign=-1.0 if sign32 else 1.0
        return sign*math.ldexp(float(fraction),-24)
    if exponent==31:
        return math.nan if fraction else (-math.inf if sign32 else math.inf)
    bits32=sign32 | ((exponent+112)<<23) | (fraction<<13)
    return float(np.uint32(bits32).view(np.float32))

@njit(cache=True)
def _float_to_half(x):
    """Float -> nearest-even binary16, fast IEEE bit arithmetic."""
    f=np.float32(x)
    bits=int(f.view(np.uint32))
    sign=(bits>>16)&0x8000
    exponent=(bits>>23)&255
    fraction=bits&0x7fffff
    if exponent==255:
        return sign | (0x7e00 if fraction else 0x7c00)
    if exponent>=143:
        return sign | 0x7c00
    if exponent<=112:
        # FP32 fraction is 24 effective bits. Shift to binary16 subnormal.
        shift=126-exponent
        if shift>=32:return sign
        src=fraction | 0x800000
        down=src>>shift
        tail=src & ((1<<shift)-1)
        half=1<<(shift-1)
        if tail==half:
            low=sign | down
            mid=(_half_to_float(low)+_half_to_float(low+1))*0.5
            round_up=(x>mid or (x==mid and (down&1))) if x>=0.0 else (x<mid or (x==mid and (down&1)))
        else:
            round_up=tail>half
        if round_up:
            down+=1
        if down>=1024:return sign | 0x0400
        return sign | down
    q=fraction>>13
    tail=fraction&0x1fff
    expo=exponent-112
    if tail==0x1000:
        low=sign | (expo<<10) | q
        mid=(_half_to_float(low)+_half_to_float(low+1))*0.5
        round_up=(x>mid or (x==mid and (q&1))) if x>=0.0 else (x<mid or (x==mid and (q&1)))
    else:
        round_up=tail>0x1000
    if round_up:
        q+=1
    if q==1024:
        q=0
        expo+=1
    if expo>=31:return sign | 0x7c00
    return sign | (expo<<10) | q

@njit(cache=True)
def _residual_value(arr,i,j,bits,span):
    if bits==0:return 0.
    if bits==16:
        return _half_to_float(arr[i,j])
    if bits==8:return float(arr[i,j])*span/127.
    if bits==4:
        byte=int(arr[i,j//2]);v=((byte&15) if j%2==0 else ((byte>>4)&15))-7
        return v*span/7.
    byte=int(arr[i,j//4]);v=((byte>>(2*(j%4)))&3)-1
    return v*span/1.

@njit(cache=True)
def _project_residual(x,span,bits):
    """Best feasible e for (q+e-target)^2 + lambda*e^2."""
    if bits==0:return 0.
    if bits==16:
        v=max(-span,min(span,x))
        code=_float_to_half(v)
        rounded=_half_to_float(code)
        # The row span is FP32 and may lie between two half values. Keep
        # every stored FP16 residual *inside* its legal [-span,span] support.
        if rounded>span or rounded< -span:
            code-=1   # one half-ULP toward zero for either sign
        return _half_to_float(code)
    num=127 if bits==8 else 7 if bits==4 else 1
    code=int(np.rint(x*num/span)) if span>0 else 0
    code=max(-num,min(num,code))
    return code*span/num

@njit(cache=True)
def _write_residual(arr,i,j,bits,quantized,span):
    if bits==0:return
    if bits==16:
        arr[i,j]=_float_to_half(quantized)
        return
    num=127 if bits==8 else 7 if bits==4 else 1
    code=int(np.rint(quantized*num/span)) if span>0 else 0
    code=max(-num,min(num,code))
    if bits==8:
        arr[i,j]=code
    elif bits==4:
        part=code+7;pos=j//2;shift=4*(j%2)
        old=int(arr[i,pos]);mask=15<<shift
        arr[i,pos]=(old & (~mask & 255)) | (part << shift)
    else:
        part=code+1;pos=j//4;shift=2*(j%4)
        old=int(arr[i,pos]);mask=3<<shift
        arr[i,pos]=(old & (~mask & 255)) | (part << shift)

@njit(cache=True)
def _row_update(k,a,b,stored,scale,g,v,lr,lam,beta,code_eta,cap,bits,scale_sqrt):
    m,n=k.shape
    flips=0
    for i in range(m):
        ai=float(a[i]);bi=float(b[i]);span=max(1.e-12,float(scale[i]));rms=1e-8
        for j in range(n):rms+=float(g[i,j])*float(g[i,j])
        rms=math.sqrt(rms/n)
        targets=np.empty(n,dtype=np.float64)
        indices=np.empty(n,dtype=np.uint8)
        ga=0.;gb=0.;na=0.;nb=0.
        for j in range(n):
            ko=int(k[i,j]);old=(-ai if ko==0 else -bi if ko==1 else bi if ko==2 else ai)
            err0=_residual_value(stored,i,j,bits,span)
            gn=float(g[i,j])/rms/max(1e-6,float(v[j]))
            gn=max(-cap,min(cap,gn))
            t=old+beta*err0-lr*ai*gn
            targets[j]=t
            best=1e100;bestk=ko
            for z in range(4):
                q=(-ai if z==0 else -bi if z==1 else bi if z==2 else ai)
                e=_project_residual((t-q)/(1.+lam),span,bits)
                cost=(q+e-t)**2+lam*e*e
                if cost<best:best=cost;bestk=z
            indices[j]=bestk
            if bestk==0:ga-=float(g[i,j]);na+=1.
            elif bestk==3:ga+=float(g[i,j]);na+=1.
            elif bestk==1:gb-=float(g[i,j]);nb+=1.
            else:gb+=float(g[i,j]);nb+=1.
        nn_a=max(1.,math.sqrt(na) if scale_sqrt else na)
        nn_b=max(1.,math.sqrt(nb) if scale_sqrt else nb)
        ha=rms*nn_a/(code_eta*max(.006,ai))
        hb=rms*nn_b/(code_eta*max(.006,ai))
        newa,newb=_fitter(ai-ga/ha,bi-gb/hb,ha,hb,ai,bi)
        a[i]=newa;b[i]=newb
        for j in range(n):
            z=int(indices[j]);q=(-newa if z==0 else -newb if z==1 else newb if z==2 else newa)
            e=_project_residual((targets[j]-q)/(1.+lam),span,bits)
            _write_residual(stored,i,j,bits,e,span)
            if z!=int(k[i,j]):flips+=1
            k[i,j]=z
    return flips

class FastCodebookBlock(SplitCodebookPacked):
    """CPU-only Numba fused solver with optional INT8 or FP16 residual state."""
    def __init__(self,model,residual_bits=4,**kw):
        super().__init__(model,**kw)
        if residual_bits not in (0,2,4,8,16):
            raise ValueError('Residual bits must be 0,2,4,8,16')
        self.residual_bits=int(residual_bits)
        for layer in model.direct_layers():
            m,n=layer.k.shape
            old=self.state[layer]['error']
            if self.residual_bits==2:
                self.state[layer]['error']=torch.full((m,(n+3)//4),0x55,dtype=torch.uint8,device=layer.k.device)
                # Preserve the legacy Stage27 ternary half-range definition.
                self.state[layer]['res_scale'].mul_(.5)
            elif self.residual_bits==0:
                self.state[layer]['error']=torch.empty((m,0),dtype=torch.uint8,device=layer.k.device)
            elif self.residual_bits==8:
                self.state[layer]['error']=torch.zeros((m,n),dtype=torch.int8,device=layer.k.device)
            elif self.residual_bits==16:
                self.state[layer]['error']=torch.zeros((m,n),dtype=torch.float16,device=layer.k.device)
            del old

    @torch.no_grad()
    def step(self):
        if any(layer.k.device.type!='cpu' for layer in self.model.direct_layers()):
            raise RuntimeError('FastCodebookBlock is CPU-only')
        x=min(self.calls,self.total_steps-1)/max(1,self.total_steps-1)
        self.lr=self.base_lr*(self.floor+(1-self.floor)*.5*(1+math.cos(math.pi*x)))
        self.calls+=1;self.transitions=0
        for layer in self.model.direct_layers():
            if layer._w is None or layer._w.grad is None:continue
            v=activation_diag(layer,self.activation_power) if self.use_activation else torch.ones(layer.k.shape[1])
            st=self.state[layer]
            error=st['error'].numpy()
            if self.residual_bits==16:
                error=error.view(np.uint16)   # zero-copy view of actual FP16 buffer
            self.transitions+=_row_update(layer.k.numpy(),layer.a.numpy(),layer.b.numpy(),
                error,st['res_scale'].numpy(),
                layer._w.grad.detach().contiguous().numpy(),v.contiguous().numpy(),
                self.lr,self.lam,self.beta,self.code_eta,self.cap,self.residual_bits,
                self.scale_mode=='sqrt')

    def aux_bytes(self):
        return sum(v.numel()*v.element_size() for st in self.state.values() for v in st.values())


Overwriting stage30_src/stage27_fast.py


In [ ]:
%%writefile stage30_src/stage27_suite.py
"""Stage 27 CPU-first hard-2bit benchmark, independent tuning, plateau diagnostics.

All methods train on identical realized initial quantized weights and identical
minibatch sequences. The loss always refers to the *entire TRAINING dataset*.
Training-time measures exclude diagnostic full-set evaluations and file I/O.
Full training-set plateau flags are descriptive and never treated as proofs.
"""
import os,json,math,time,random,hashlib,zipfile,sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import stage26_suite as s26
from stage27_fast import FastCodebookBlock,_row_update

# A modest default that works in CPU Colab. Colab users can override THREADS.
THREADS=int(os.getenv('STAGE27_THREADS','1'))
torch.set_num_threads(THREADS)
ORIGINAL_N={k:int(v['n']) for k,v in s26.TASKS.items()}
LOCKED_STAGE26={
 'mlp_robust':dict(lr=.08,code_lr=.003,head_lr=.003,head_lr_factor=.5,momentum=.95,adjust_lr_fn='match_rms_adamw',head_mode='muon'),
 'resmlp_quantile':dict(lr=.08,code_lr=.003,head_lr=.003,head_lr_factor=.5,momentum=.9,adjust_lr_fn='match_rms_adamw',head_mode='muon'),
 'rnn_bce':dict(lr=.2,code_lr=.003,head_lr=.015,head_lr_factor=1.5,momentum=.9,adjust_lr_fn='original',head_mode='adam'),
 'cnn_shapes':dict(lr=.15,code_lr=.01,head_lr=.01,head_lr_factor=1.,momentum=.95,adjust_lr_fn='match_rms_adamw',head_mode='adam'),
 'transformer_128':dict(lr=.2,code_lr=.003,head_lr=.003,head_lr_factor=.5,momentum=.95,adjust_lr_fn='original',head_mode='adam'),
 'transformer_192':dict(lr=.2,code_lr=.003,head_lr=.003,head_lr_factor=.5,momentum=.95,adjust_lr_fn='original',head_mode='adam'),
}
# Each alternative minimizes the same kind of local discrete problem but has a
# different feasible residual set and hence different persistent state cost.
METHODS=('muon','fast0','fast2','fast4','fast8','fast16','torch26')
FIXED=dict(lam=.05,beta=.9,floor=.20,kappa=.01,scale_mode='count',use_activation=True,activation_power=1.)
DEFAULT_TASKS=['mlp_robust','resmlp_quantile','rnn_bce','cnn_shapes','transformer_128','transformer_192']
OPTIONAL_TASKS=['resmlp_poisson','resmlp_focal','resmlp_multiclass','rnn_robust','transformer_256_stress']
MODES={
 'smoke':dict(compact_steps=3,transformer_steps=2,screen=1,screen_steps=2,refine=1,refine_steps=2,devseeds=[21],evalseeds=[101],evals=2),
 'quick':dict(compact_steps=160,transformer_steps=72,screen=3,screen_steps=24,refine=1,refine_steps=70,devseeds=[21],evalseeds=[101],evals=9),
 'balanced':dict(compact_steps=400,transformer_steps=200,screen=5,screen_steps=80,refine=2,refine_steps=180,devseeds=[21,31],evalseeds=[101,102],evals=17),
 'plateau':dict(compact_steps=800,transformer_steps=340,screen=7,screen_steps=100,refine=3,refine_steps=300,devseeds=[21,31],evalseeds=[101,102],evals=25),
}

def prepare_numba():
    k=np.array([[0,1,2,3]],dtype='uint8');a=np.array([.6],dtype='float32');b=np.array([.2],dtype='float32')
    g=np.array([[.1,-.2,.1,-.3]],dtype='float32');v=np.ones(4,dtype='float32');sc=a.copy()
    for bits,cols,fill,dtype in ((0,0,0,'uint8'),(2,1,0x55,'uint8'),(4,2,0x77,'uint8'),(8,4,0,'int8'),(16,4,0,'uint16')):
        er=np.full((1,cols),fill,dtype=dtype)
        _row_update(k.copy(),a.copy(),b.copy(),er,sc.copy(),g,v,.1,.05,.9,.04,4.,bits,False)


def model_and_optimizer(task,method,cfg,seed,steps):
    model=s26.model_for(task,'muon' if method=='muon' else 'codebook_block',seed+9000,'cpu')
    if method=='muon':opt=s26.MuonSTE(model,cfg,'cpu')
    elif method=='torch26':
        # Identical Stage26 tensor solver, useful as a performance reference.
        params=dict(FIXED);params.update(cfg);params['total_steps']=steps
        opt=s26.SplitCodebookPacked(model,**params)
    else:
        params=dict(FIXED);params.update(cfg)
        opt=FastCodebookBlock(model,residual_bits=int(method[4:]),total_steps=steps,**params)
    return model,opt


def aux_state_bytes(model,opt,method):
    if method=='muon':
        # Shadow weights are separate trainable master parameters, rather than
        # an optimizer.state entry. Count shadow and all Muon/Adam buffers.
        master=sum(l.shadow.numel()*l.shadow.element_size() for l in model.layers)
        optim=sum(t.numel()*t.element_size() for obj in (opt.muon,opt.adam)
                  for state in obj.state.values() for t in state.values() if torch.is_tensor(t))
        return master,optim
    return 0, opt.aux_bytes()


def optimizer_state_snapshot(model,opt,method):
    return (opt.state_dict() if method=='muon' else {
        'calls':opt.calls,'state':[
            {name:value.detach().cpu().clone() for name,value in opt.state[layer].items()}
            for layer in model.direct_layers()]})

def restore_state(model,opt,method,state):
    if method=='muon':opt.load_state_dict(state);return
    opt.calls=int(state['calls'])
    for l,v in zip(model.direct_layers(),state['state']):
        for key,tensor in v.items():opt.state[l][key].copy_(tensor.to('cpu'))

def stable_key(task,method,cfg,seed,steps,phase):
    data=json.dumps([task,method,cfg,seed,steps,phase,THREADS,s26.TASKS[task]['n'],'v27.5-precision'],sort_keys=True)
    return hashlib.sha256(data.encode()).hexdigest()[:24]

@torch.no_grad()
def full_train_loss(model,X,Y,task):
    # Larger chunks reduce repeated tiny-CPU-forward overhead. Still EVERY
    # training example is evaluated; no training/validation subsampling here.
    chunk=(128 if task=='transformer_192' or task=='transformer_256_stress' else 256)
    total=0.
    for i in range(0,len(X),chunk):
        n=min(chunk,len(X)-i)
        val=s26.loss_for(model,X[i:i+n],Y[i:i+n],task)
        total+=float(val.detach())*n
    return total/len(X)

def train(task,method,cfg,seed,steps,folder,phase='heldout',eval_every=25,resume=True,checkpoint_every=50):
    folder=Path(folder);(folder/'runs').mkdir(parents=True,exist_ok=True)
    key=stable_key(task,method,cfg,seed,steps,phase)
    final=folder/'runs'/f'{key}.json'; ckpt=folder/'checkpoints'/f'{key}.pt'
    if final.exists():
        out=json.loads(final.read_text());return out['result'],out['curve']
    X,Y=s26.dataset(task,seed)
    model,opt=model_and_optimizer(task,method,cfg,seed,steps)
    cfg_task=s26.TASKS[task];batch=cfg_task['batch']
    order=torch.randint(len(X),(steps,batch),generator=torch.Generator().manual_seed(seed+9999))
    step0=0;elapsed=0.;opt_elapsed=0.;eval_elapsed=0.;curves=[];transitions=[]
    if resume and ckpt.exists():
        d=torch.load(ckpt,map_location='cpu',weights_only=False)
        if d['key']!=key:raise ValueError('Checkpoint mismatch')
        model.load_state_dict(d['model']);restore_state(model,opt,method,d['opt'])
        step0=d['step'];elapsed=d['elapsed'];opt_elapsed=d['opt_elapsed'];eval_elapsed=d['eval_elapsed']
        curves=d['curves'];transitions=d.get('transitions',[])
    for it in range(step0,steps+1):
        if it==0 or it==steps or it%eval_every==0:
            if not curves or curves[-1]['step']!=it:
                start=time.perf_counter();val=full_train_loss(model,X,Y,task)
                eval_elapsed+=time.perf_counter()-start
                if not np.isfinite(val):raise FloatingPointError(f'{task} {method} loss nonfinite at {it}')
                curves.append(dict(task=task,method=method,seed=int(seed),phase=phase,
                    step=it,full_training_loss=float(val),training_seconds=elapsed,
                    optimizer_seconds=opt_elapsed))
        if it==steps:break
        x=X[order[it]];y=Y[order[it]]
        start=time.perf_counter()
        if method=='muon':opt.zero_grad()
        else:model.zero_grad(set_to_none=True)
        loss=s26.loss_for(model,x,y,task)
        if not torch.isfinite(loss.detach()):raise FloatingPointError('Nonfinite batch loss')
        loss.backward();middle=time.perf_counter();opt.step();stop=time.perf_counter()
        elapsed+=stop-start;opt_elapsed+=stop-middle
        if method!='muon' and ((it+1)%eval_every==0 or it+1==steps):
            P=sum(l.k.numel() for l in model.direct_layers())
            transitions.append({'step':it+1,'rate':opt.transitions/max(P,1)})
        if checkpoint_every and (it+1)%checkpoint_every==0:
            ckpt.parent.mkdir(parents=True,exist_ok=True)
            torch.save(dict(key=key,step=it+1,model=model.state_dict(),opt=optimizer_state_snapshot(model,opt,method),
                     elapsed=elapsed,opt_elapsed=opt_elapsed,eval_elapsed=eval_elapsed,
                     curves=curves,transitions=transitions),ckpt)
    values=np.array([x['full_training_loss'] for x in curves]);t=np.array([x['step'] for x in curves])
    p=np.argsort(t); values=values[p];t=t[p]
    # Plateau is a descriptive diagnostic: improvement in the last 20% of the
    # horizon as a fraction of the total observed reduction; not an early stop.
    start_i=np.searchsorted(t,round(.8*steps),side='left');start_i=min(start_i,len(t)-2)
    late_drop=float(values[start_i]-values[-1]);total_drop=float(values[0]-values[-1]);
    plateau_ratio=late_drop/max(abs(total_drop),1e-7)
    master,aux=aux_state_bytes(model,opt,method)
    direct=model.direct_layers() if method!='muon' else []
    index_bytes=sum(l.k.numel()*l.k.element_size() for l in direct)
    codebook_bytes=sum((l.a.numel()*l.a.element_size()+l.b.numel()*l.b.element_size()) for l in model.layers)
    model_params=sum(l.k.numel() for l in direct) if method!='muon' else sum(l.shadow.numel() for l in model.layers)
    logical_weight_bytes=math.ceil(model_params*2/8)
    total_persistent_bytes=master+aux+index_bytes+codebook_bytes
    result=dict(task=task,method=method,phase=phase,seed=int(seed),steps=steps,cfg=json.dumps(cfg,sort_keys=True),
      samples=len(X),initial=float(values[0]),final=float(values[-1]),loss_auc=float(np.trapezoid(values,t)/steps),
      ms_per_step=1000*elapsed/steps,ms_optimizer=1000*opt_elapsed/steps,training_seconds=elapsed,
      diagnostic_seconds=eval_elapsed,master_bytes=master,optimizer_aux_bytes=aux,
      weight_indices_physical_bytes=index_bytes,codebook_bytes=codebook_bytes,
      persistent_total_bytes=total_persistent_bytes,logical_packed_int2_bytes=logical_weight_bytes,
      logical_residual_bits=(4 if method=='torch26' else int(method[5:] if method.startswith('joint') else method[4:])) if method!='muon' else None,
      late_drop=late_drop,late_fraction=plateau_ratio,plateau_flag=bool(total_drop>0 and plateau_ratio<.01),
      positive_loss_reversals=int(np.sum(np.diff(values)>1e-6)),
      total_upward_loss=float(np.clip(np.diff(values),0,None).sum()),
      final_transition_rate=float(np.mean([z['rate'] for z in transitions[-3:]])) if transitions else None,
      torch_threads=THREADS)
    final.write_text(json.dumps(dict(result=result,curve=curves,transitions=transitions),allow_nan=False))
    if ckpt.exists():ckpt.unlink()
    return result,curves


def configs(task,method):
    if method=='muon':
        base=LOCKED_STAGE26.get(task,LOCKED_STAGE26['mlp_robust']);pool=[]
        def add(v):
            if v not in pool:pool.append(v)
        add(dict(base))
        for scale in (.5,1.5,2.,3.):
            x=dict(base);x['lr']=round(base['lr']*scale,8);add(x)
        for code_scale in (.5,3.):
            x=dict(base);x['code_lr']=round(base['code_lr']*code_scale,8);add(x)
        x=dict(base);x['momentum']=.90 if base['momentum']==.95 else .95;add(x)
        x=dict(base);x['head_mode']='adam' if base['head_mode']=='muon' else 'muon';add(x)
        # Head-mode changes Adam head route, all codebooks still learned.
        return pool
    # Only two independently tuned hyperparameters: lr and code_eta.
    return [{'lr':lr,'code_eta':eta} for lr in (.07,.14,.23,.35) for eta in (.02,.04,.08)]


def select_screen(task,method,n):
    pool=configs(task,method)
    if n>=len(pool):return pool
    if method=='muon':return pool[:n]
    # Deterministic max-spread sampling across the 2D grid; includes center.
    order=[4,0,11,7,2,9,5,6,1,10,3,8]
    return [pool[i] for i in order[:n]]


def tune(task,method,mode,folder):
    opts=MODES[mode];screen=select_screen(task,method,opts['screen']);screen_results=[]
    # Stage26 learned that a 10-step screen is inadequate. We use longer
    # refinement and do not treat short-screen ranking as conclusive.
    for cfg in screen:
        r,_=train(task,method,cfg,opts['devseeds'][0],opts['screen_steps'],folder,'screen',
                  eval_every=max(2,opts['screen_steps']//3),checkpoint_every=0)
        screen_results.append((r['loss_auc'],cfg))
    screen_results.sort(key=lambda x:x[0])
    refine_results=[]
    for _,c in screen_results[:opts['refine']]:
        vals=[]
        for seed in opts['devseeds']:
            r,_=train(task,method,c,seed,opts['refine_steps'],folder,'refine',
                      eval_every=max(3,opts['refine_steps']//5),checkpoint_every=0)
            vals.append(r['loss_auc'])
        refine_results.append((float(np.mean(vals)),c))
    refine_results.sort(key=lambda x:x[0])
    return refine_results[0][1],screen_results,refine_results


def compare_thresholds(curves):
    """Time to fixed shared losses only when both arms start at the same loss."""
    rows=[]; cf=pd.DataFrame(curves)
    if cf.empty:return pd.DataFrame()
    cf=cf[cf.phase=='heldout']
    for (task,seed),group in cf.groupby(['task','seed']):
        initial=group[group.method=='muon'].sort_values('step')
        if initial.empty:continue
        start=float(initial.iloc[0]['full_training_loss'])
        # A shared *absolute* threshold is derived from Muon's *development*
        # baseline when available; evaluation-derived thresholds would bias
        # comparisons. Here we report a fixed fraction of the same initial loss,
        # clearly marked as exploratory, not a tuning statistic.
        for fraction in (.5,.25):
            threshold=start*fraction
            for method,d in group.groupby('method'):
                d=d.sort_values('step');done=d[d.full_training_loss<=threshold]
                rows.append(dict(task=task,seed=seed,method=method,threshold_fraction=fraction,threshold=threshold,
                                 first_step=int(done.iloc[0].step) if len(done) else None,
                                 first_training_seconds=float(done.iloc[0].training_seconds) if len(done) else None))
    return pd.DataFrame(rows)


def summarize(folder):
    folder=Path(folder);rows=[];curves=[]
    for p in sorted((folder/'runs').glob('*.json')):
        x=json.loads(p.read_text());rows.append(x['result']);curves.extend(x['curve'])
    allruns=pd.DataFrame(rows);cf=pd.DataFrame(curves)
    if len(allruns):allruns.to_csv(folder/'all_runs.csv',index=False)
    if len(cf):cf.to_csv(folder/'all_curves.csv',index=False)
    held=allruns[allruns.phase=='heldout'].copy() if len(allruns) else pd.DataFrame()
    if len(held):
        held.to_csv(folder/'heldout.csv',index=False)
        summary=held.groupby(['task','method'],as_index=False).agg({
            'loss_auc':'mean','final':'mean','ms_per_step':'mean','ms_optimizer':'mean',
            'late_fraction':'mean','total_upward_loss':'mean','optimizer_aux_bytes':'mean',
            'master_bytes':'mean','plateau_flag':'mean', 'persistent_total_bytes':'mean'})
        summary.to_csv(folder/'summary.csv',index=False)
        gates=[]
        for task,g in summary.groupby('task'):
            mu=g[g.method=='muon']
            if len(mu)==0:continue
            b=mu.iloc[0]
            for _,v in g[g.method!='muon'].iterrows():
                gates.append(dict(task=task,method=v.method,beats_muon_auc=bool(v.loss_auc<b.loss_auc),
                    beats_muon_step_ms=bool(v.ms_per_step<b.ms_per_step),
                    passes_both=bool(v.loss_auc<b.loss_auc and v.ms_per_step<b.ms_per_step)))
        pd.DataFrame(gates).to_csv(folder/'gates.csv',index=False)
        thresholds=compare_thresholds(curves)
        if len(thresholds):thresholds.to_csv(folder/'thresholds.csv',index=False)
    return held,cf


def make_plots(folder):
    import matplotlib.pyplot as plt
    folder=Path(folder);f=folder/'all_curves.csv'
    if not f.exists():return
    df=pd.read_csv(f);df=df[df.phase=='heldout']
    if not len(df):return
    for task,d in df.groupby('task'):
        for horizontal,label in [('step','updates'),('training_seconds','seconds')]:
            fig,ax=plt.subplots(figsize=(7.2,4.2))
            for method,g in d.groupby('method'):
                piv=g.pivot_table(index='step',columns='seed',values='full_training_loss');mean=piv.mean(axis=1)
                if horizontal=='step':xx=piv.index.values
                else:
                    tim=g.groupby('step').training_seconds.mean().reindex(piv.index)
                    xx=tim.values
                ax.plot(xx,mean.values,label=method)
                if horizontal=='step' and piv.shape[1]>1:
                    ax.fill_between(xx,piv.min(axis=1).values,piv.max(axis=1).values,alpha=.09)
            ax.set(title=f'{task}: full TRAINING loss',xlabel=f'Cumulative training {label}' if horizontal=='seconds' else 'Optimizer updates',ylabel='Full training loss')
            ax.grid(alpha=.2);ax.legend();fig.tight_layout();fig.savefig(folder/f'{task}_{label}.png',dpi=120);plt.close(fig)


def export_zip(folder,zip_path='Stage27_CPU_Long_Horizon_Results.zip',download=True):
    folder=Path(folder);out=Path(zip_path)
    with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
        for path in folder.rglob('*'):
            if path.is_file():z.write(path,arcname=str(Path(folder.name)/path.relative_to(folder)))
    print('Export:',out.resolve(),'bytes',out.stat().st_size,flush=True)
    if download:
        try:
            from google.colab import files
            files.download(str(out))
        except ImportError: print('Not in Colab; ZIP available at',out.resolve())
    return out


def run(tasks=None,methods=('muon','fast4','fast8','fast16'),mode='plateau',output='stage27_results',do_tune=True):
    if mode not in MODES:raise ValueError('Unknown MODE: '+str(mode))
    if 'muon' not in methods:raise ValueError('Muon is a mandatory comparator')
    tasks=tasks or DEFAULT_TASKS
    for task in tasks:
        if task not in s26.TASKS:raise ValueError('Unknown task: '+str(task))
    prepare_numba()
    # The fast smoke/quick presets reduce training-set size but never network
    # width/depth. Balanced and plateau retain original Stage26 dataset size.
    for name,original in ORIGINAL_N.items():
        cap=(128 if mode=='smoke' else
             (640 if name.startswith('transformer') else 720) if mode=='quick' else original)
        s26.TASKS[name]['n']=min(original,cap)
    s26.dataset.cache_clear();s26.tokens_data.cache_clear();s26.old.make_dataset.cache_clear()
    out=Path(output);out.mkdir(exist_ok=True,parents=True)
    spec=MODES[mode];settings={}
    (out/'protocol.json').write_text(json.dumps({'mode':mode,'tasks':tasks,'methods':methods,'seed_dev':spec['devseeds'],
        'seed_eval':spec['evalseeds'],'samples_per_task':{t:s26.TASKS[t]['n'] for t in tasks},'thread_count':THREADS,'metric':'full training loss; time excludes evaluation',
        'note':'CPU reference; Numba compiled before timing; INT4/INT2 bitpacked residuals, INT8 signed bytes, FP16 native tensors; uint8 INT2 weight symbols; all residuals shared support [-row_scale,row_scale] (except legacy INT2 half range)'},indent=2))
    lockfile=out/f'locked_configs_{mode}.json'
    if lockfile.exists():settings=json.loads(lockfile.read_text())
    try:
        for task in tasks:
            if task not in settings:settings[task]={}
            for method in methods:
                if method not in settings[task]:
                    if do_tune:
                        cfg,screen,refine=tune(task,method,mode,out)
                        print('LOCKED',task,method,cfg,'dev AUC',round(refine[0][0],6),flush=True)
                    else:
                        cfg=(LOCKED_STAGE26.get(task,LOCKED_STAGE26['resmlp_quantile']) if method=='muon' else {'lr':.14,'code_eta':.04})
                        print('UN-TUNED CONTROL',task,method,cfg,flush=True)
                    settings[task][method]=cfg;lockfile.write_text(json.dumps(settings,indent=2))
                cfg=settings[task][method]
                steps=spec['transformer_steps'] if task.startswith('transformer') else spec['compact_steps']
                eval_every=max(1,round(steps/spec['evals']))
                for seed in spec['evalseeds']:
                    r,_=train(task,method,cfg,seed,steps,out,'heldout',eval_every=eval_every,
                              checkpoint_every=max(0,steps//8))
                    print('EVAL',task,method,'seed',seed,'AUC',round(r['loss_auc'],5),
                         'final',round(r['final'],5),'ms/step',round(r['ms_per_step'],2),
                         'plateau_ratio',round(r['late_fraction'],4),flush=True)
                summarize(out);make_plots(out)
    finally:
        summarize(out);make_plots(out)
    return out


Overwriting stage30_src/stage27_suite.py


In [ ]:
%%writefile stage30_src/stage28_candidates.py
"""Stage 28 exploratory joint discrete-codebook proximal vs split-codebook.

Mode 0: Stage27 split-codebook with alternative schedule for codebook LR.
Mode 1: A SINGLE JOINT surrogate, exact conditional finite integer/residual
        minimization and 2D constrained codebook magnitude solve.
This is CPU-only and *not* claimed to guarantee descent of true loss.
"""
import math
import numpy as np,torch
from numba import njit
from stage27_fast import FastCodebookBlock,_residual_value,_project_residual,_write_residual
from packed_prox_numba import _fitter
from stage22_optim import activation_diag

@njit(cache=True)
def row_step28(k,a,b,stored,scale,g,v,lr,lam,beta,code_eta,cap,bits,
               mode, code_decay, gamma, passes):
    m,n=k.shape;flips=0
    for i in range(m):
        ai=float(a[i]);bi=float(b[i]);span=max(1.e-12,float(scale[i]));rms=1e-8
        for j in range(n):rms+=float(g[i,j])*float(g[i,j])
        rms=math.sqrt(rms/n)
        T=np.empty(n,dtype=np.float64)
        ix=np.empty(n,dtype=np.uint8)
        er=np.empty(n,dtype=np.float64)
        for j in range(n):
            ko=int(k[i,j]);qold=(-ai if ko==0 else -bi if ko==1 else bi if ko==2 else ai)
            gn=float(g[i,j])/rms/max(1e-6,float(v[j]));gn=max(-cap,min(cap,gn))
            T[j]=qold+beta*_residual_value(stored,i,j,bits,span)-lr*ai*gn
        aa=ai;bb=bi
        for pp in range(passes):
            # exact fixed-codebook index+residual conditional step
            for j in range(n):
                best=1e100; bestk=0;beste=0.
                for z in range(4):
                    q=(-aa if z==0 else -bb if z==1 else bb if z==2 else aa)
                    e=_project_residual((T[j]-q)/(1.+lam),span,bits)
                    cost=(q+e-T[j])**2+lam*e*e
                    if cost<best:best=cost;bestk=z;beste=e
                ix[j]=bestk;er[j]=beste
            if pp==passes-1:break
            if mode==0:
                # Separate gradient-on-codebook surrogate (legacy), optionally scheduled.
                ga=0.;gb=0.;na=0.;nb=0.
                for j in range(n):
                    z=int(ix[j]);gg=float(g[i,j])
                    if z==0:ga-=gg;na+=1.
                    elif z==3:ga+=gg;na+=1.
                    elif z==1:gb-=gg;nb+=1.
                    else:gb+=gg;nb+=1.
                ea=max(code_eta*code_decay,1.e-8)
                ha=rms*max(na,1.)/(ea*max(.006,aa))
                hb=rms*max(nb,1.)/(ea*max(.006,aa))
                aa,bb=_fitter(aa-ga/ha,bb-gb/hb,ha,hb,ai,bi)
            else:
                # Exact constrained 2D codebook minimizer of SAME fixed-T joint surrogate.
                ha=gamma*n;hb=gamma*n;sa=gamma*n*ai;sb=gamma*n*bi
                for j in range(n):
                    z=int(ix[j]);target=T[j]-er[j]; vv=max(1.e-6,float(v[j]))
                    if z==0:ha+=vv;sa-=vv*target
                    elif z==3:ha+=vv;sa+=vv*target
                    elif z==1:hb+=vv;sb-=vv*target
                    else:hb+=vv;sb+=vv*target
                aa,bb=_fitter(sa/ha,sb/hb,ha,hb,ai,bi)
        a[i]=aa;b[i]=bb
        for j in range(n):
            z=int(ix[j]);_write_residual(stored,i,j,bits,er[j],span)
            if z!=int(k[i,j]):flips+=1
            k[i,j]=z
    return flips

class Stage28Optim(FastCodebookBlock):
    def __init__(self,model,mode=0,gamma=.02,code_decay_exponent=0.,passes=2,
                 constant_lr=False,**kwargs):
        super().__init__(model,**kwargs)
        self.mode=int(mode);self.gamma=float(gamma);self.decay_p=float(code_decay_exponent)
        self.passes=int(passes);self.constant_lr=bool(constant_lr)
    @torch.no_grad()
    def step(self):
        x=min(self.calls,self.total_steps-1)/max(1,self.total_steps-1)
        sched=(self.floor+(1.-self.floor)*.5*(1+math.cos(math.pi*x)))
        self.lr=self.base_lr*(1 if self.constant_lr else sched)
        self.calls+=1;self.transitions=0
        for layer in self.model.direct_layers():
            if layer._w is None or layer._w.grad is None:continue
            v=activation_diag(layer,self.activation_power) if self.use_activation else torch.ones(layer.k.shape[1])
            st=self.state[layer];error=st['error'].numpy()
            if self.residual_bits==16:error=error.view(np.uint16)
            self.transitions+=row_step28(layer.k.numpy(),layer.a.numpy(),layer.b.numpy(),error,
               st['res_scale'].numpy(),layer._w.grad.detach().contiguous().numpy(),v.contiguous().numpy(),
               self.lr,self.lam,self.beta,self.code_eta,self.cap,self.residual_bits,self.mode,
               sched**self.decay_p,self.gamma,self.passes)


Overwriting stage30_src/stage28_candidates.py


In [ ]:
%%writefile stage30_src/stage28_suite.py
"""Stage 28: auditable horizon-aware CPU search for a coherent discrete objective.

Inherits Stage27 training instrumentation, model definitions, Muon baseline,
checkpoints, and export. Adds joint0/4/8/16 and fixes the Stage27 search omission
of conservative codebook rates. Fits joint magnitudes and integer/residual block
within ONE surrogate via conditional exact solves. A global optimum is not claimed.
"""
from copy import deepcopy
import numpy as np
import stage27_suite as s27
from stage28_candidates import Stage28Optim,row_step28

ORIGINAL_MODEL=s27.model_and_optimizer

def model_and_optimizer(task,method,cfg,seed,steps):
    if not method.startswith('joint'):
        return ORIGINAL_MODEL(task,method,cfg,seed,steps)
    bits=int(method[5:])
    model=s27.s26.model_for(task,'codebook_block',seed+9000,'cpu')
    fixed=dict(s27.FIXED)
    fixed.update({'lr':float(cfg['lr']),'code_eta':.04,'total_steps':steps})
    opt=Stage28Optim(model,mode=1,gamma=float(cfg.get('gamma',.01)),passes=2,
                     residual_bits=bits,**fixed)
    return model,opt

def configs(task,method):
    if method.startswith('joint'):
        # Gamma fixed as a specified structural trust penalty; ablate separately.
        return [{'lr':float(lr)} for lr in (.035,.07,.10,.14,.20,.28,.40)]
    if method.startswith('fast'):
        # Stage27 omitted code_eta=.005/.01, which made RNN dramatically worse.
        order=[(.14,.04),(.14,.01),(.07,.02),(.20,.01),(.035,.005),
               (.10,.02),(.28,.01),(.14,.005),(.07,.01),(.20,.04),
               (.10,.04),(.28,.005),(.035,.02),(.14,.02),(.07,.005),
               (.20,.02),(.10,.01),(.28,.04),(.035,.01),(.14,.08),
               (.07,.04),(.20,.005),(.10,.005),(.035,.04),
               (.28,.02),(.07,.08),(.10,.08),(.20,.08),(.28,.08),(.035,.08)]
        assert len(order)==len(set(order))
        return [{'lr':float(lr),'code_eta':float(eta)} for lr,eta in order]
    if method=='muon':
        base=s27.LOCKED_STAGE26.get(task,s27.LOCKED_STAGE26['mlp_robust'])
        arr=[]
        def add(**kw):
            c=dict(base);c.update(kw)
            if c not in arr:arr.append(c)
        for f in (.15,.25,.5,1.,1.5,2.):add(lr=round(base['lr']*f,8))
        for f in (.35,3.):add(code_lr=round(base['code_lr']*f,8))
        add(head_mode=('adam' if base['head_mode']=='muon' else 'muon'))
        add(momentum=(.90 if base['momentum']==.95 else .95))
        add(head_lr=base['head_lr']*.5)
        add(head_lr=base['head_lr']*2.)
        add(adjust_lr_fn=('original' if base['adjust_lr_fn']!='original' else 'match_rms_adamw'))
        return arr
    raise ValueError(f'Unsupported method {method}')

def select_screen(task,method,n):
    pool=configs(task,method)
    return pool[:n] if n<len(pool) else pool

def tune(task,method,mode,folder):
    opts=s27.MODES[mode]
    istransformer=task.startswith('transformer')
    steps=opts['transformer_steps'] if istransformer else opts['compact_steps']
    if mode=='smoke':
        n,ss,nref,rs,nseeds=1,2,1,2,1
    elif mode=='quick':
        n,ss,nref,rs,nseeds=(4,25,2,80,1) if not istransformer else (3,12,1,36,1)
    elif mode=='balanced':
        n,ss,nref,rs,nseeds=(9,120,3,260,2) if not istransformer else (6,45,2,130,1)
    else: # plateau
        n,ss,nref,rs,nseeds=(13,150,4,500,2) if not istransformer else (7,50,2,200,1)
    ss=min(ss,steps);rs=min(rs,steps)
    devseeds=opts['devseeds'][:nseeds]
    scr=[]
    for c in select_screen(task,method,n):
        v,_=s27.train(task,method,c,devseeds[0],ss,folder,'screen28',
                      eval_every=max(2,ss//8),checkpoint_every=0)
        scr.append((v['loss_auc'],c))
    scr.sort(key=lambda x:x[0]);ref=[]
    for _,cfg in scr[:nref]:
        results=[]
        for seed in devseeds:
            v,_=s27.train(task,method,cfg,seed,rs,folder,'refine28',
                          eval_every=max(3,rs//16),checkpoint_every=0)
            results.append(v['loss_auc'])
        ref.append((float(np.mean(results)),cfg))
    ref.sort(key=lambda x:x[0]);return ref[0][1],scr,ref

# Stage27 orchestration resolves its globals at call-time. Patch only the
# three discovery operations; the original evaluation/export are unchanged.
s27.model_and_optimizer=model_and_optimizer
s27.configs=configs
s27.select_screen=select_screen
s27.tune=tune
s27.MODES['plateau']['evalseeds']=[901,902]
s27.MODES['balanced']['evalseeds']=[901,902]
# Re-exports preserve Stage27 cell ergonomics.
def prepare_stage28():
    s27.prepare_numba()
    import numpy as np
    K=np.array([[0,1,2,3]],dtype=np.uint8)
    A=np.array([.6],dtype=np.float32)
    B=np.array([.2],dtype=np.float32)
    G=np.array([[.1,-.2,.1,-.3]],dtype=np.float32)
    V=np.ones(4,dtype=np.float32)
    for bits,dt,col in ((0,np.uint8,0),(4,np.uint8,2),(8,np.int8,4),(16,np.uint16,4)):
        E=np.zeros((1,col),dtype=dt)
        row_step28(K.copy(),A.copy(),B.copy(),E,A.copy(),G,V,.14,.05,.9,.04,4.,bits,1,1.,.01,2)

def run(*args,**kwargs):
    prepare_stage28()
    return s27.run(*args,**kwargs)

export_zip=s27.export_zip
MODES=s27.MODES
DEFAULT_TASKS=s27.DEFAULT_TASKS
OPTIONAL_TASKS=s27.OPTIONAL_TASKS
prepare_numba=s27.prepare_numba
LOCKED_STAGE26=s27.LOCKED_STAGE26
summarize=s27.summarize
make_plots=s27.make_plots
THREADS=s27.THREADS


Overwriting stage30_src/stage28_suite.py


In [ ]:
%%writefile stage30_src/stage30_joint.py
"""Stage 30 allocation-lean CPU joint constrained INT2+INT8 proximal solver.

Same Stage28 alternating surrogate, but row scratch arrays are allocated once
per matrix. Optionally expands the *upper* codebook bound (also applied to
Muon in the controlled matching arm). Float32 temporary gradients are not
persistent optimizer state.
"""
import math
import numpy as np
import torch
from numba import njit, prange, get_num_threads, set_num_threads
from stage28_candidates import Stage28Optim
from stage27_fast import _residual_value,_project_residual,_write_residual
from stage22_optim import activation_diag

@njit(cache=True)
def _fitter_bounds(ua,ub,Ha,Hb,aa0,bb0,amax):
    amin=.006; bmin=.002; gap=.004
    wa=max(Ha,1e-12);wb=max(Hb,1e-12)
    best=1e100;ans_a=aa0;ans_b=bb0
    if ua>=amin and ua<=amax and ub>=bmin and ub<=ua-gap:
        best=0.;ans_a=ua;ans_b=ub
    ax=amin;bx=max(bmin,min(amin-gap,ub))
    val=wa*(ax-ua)**2+wb*(bx-ub)**2
    if val<best:best=val;ans_a=ax;ans_b=bx
    ax=amax;bx=max(bmin,min(amax-gap,ub))
    val=wa*(ax-ua)**2+wb*(bx-ub)**2
    if val<best:best=val;ans_a=ax;ans_b=bx
    bx=bmin;ax=max(max(amin,bmin+gap),min(amax,ua))
    val=wa*(ax-ua)**2+wb*(bx-ub)**2
    if val<best:best=val;ans_a=ax;ans_b=bx
    bx=max(max(bmin,amin-gap),min(amax-gap,(wa*(ua-gap)+wb*ub)/(wa+wb)));ax=bx+gap
    val=wa*(ax-ua)**2+wb*(bx-ub)**2
    if val<best:best=val;ans_a=ax;ans_b=bx
    return ans_a,ans_b

@njit(cache=True)
def joint_step30(k,a,b,stored,scale,g,v,lr,lam,beta,cap,bits,gamma,passes,amax):
    m,n=k.shape;flips=0
    # One row scratch allocation per layer, rather than one per row.
    T=np.empty(n,dtype=np.float64)
    ix=np.empty(n,dtype=np.uint8)
    er=np.empty(n,dtype=np.float64)
    for i in range(m):
        ai=float(a[i]);bi=float(b[i]);span=max(1.e-12,float(scale[i]));rms=1e-8
        for j in range(n):rms+=float(g[i,j])*float(g[i,j])
        rms=math.sqrt(rms/n)
        for j in range(n):
            ko=int(k[i,j]);qold=(-ai if ko==0 else -bi if ko==1 else bi if ko==2 else ai)
            gn=float(g[i,j])/rms/max(1e-6,float(v[j]));gn=max(-cap,min(cap,gn))
            T[j]=qold+beta*_residual_value(stored,i,j,bits,span)-lr*ai*gn
        aa=ai;bb=bi
        for pp in range(passes):
            for j in range(n):
                best=1e100;bestk=0;beste=0.
                for z in range(4):
                    q=(-aa if z==0 else -bb if z==1 else bb if z==2 else aa)
                    e=_project_residual((T[j]-q)/(1.+lam),span,bits)
                    cost=(q+e-T[j])**2+lam*e*e
                    if cost<best:best=cost;bestk=z;beste=e
                ix[j]=bestk;er[j]=beste
            if pp==passes-1:break
            ha=gamma*n;hb=gamma*n;sa=gamma*n*ai;sb=gamma*n*bi
            for j in range(n):
                z=int(ix[j]);target=T[j]-er[j]; vv=max(1.e-6,float(v[j]))
                if z==0:ha+=vv;sa-=vv*target
                elif z==3:ha+=vv;sa+=vv*target
                elif z==1:hb+=vv;sb-=vv*target
                else:hb+=vv;sb+=vv*target
            aa,bb=_fitter_bounds(sa/ha,sb/hb,ha,hb,ai,bi,amax)
        a[i]=aa;b[i]=bb
        for j in range(n):
            z=int(ix[j]);_write_residual(stored,i,j,bits,er[j],span)
            if z!=int(k[i,j]):flips+=1
            k[i,j]=z
    return flips

class Stage30Joint(Stage28Optim):
    def __init__(self,model,gamma=.01,passes=2,amax=2.,**kwargs):
        super().__init__(model,mode=1,gamma=gamma,passes=passes,**kwargs)
        self.amax=float(amax)
    @torch.no_grad()
    def step(self):
        x=min(self.calls,self.total_steps-1)/max(1,self.total_steps-1)
        self.lr=self.base_lr*(self.floor+(1.-self.floor)*.5*(1.+math.cos(math.pi*x)))
        self.calls+=1;self.transitions=0
        for layer in self.model.direct_layers():
            if layer._w is None or layer._w.grad is None:continue
            v=activation_diag(layer,self.activation_power) if self.use_activation else torch.ones(layer.k.shape[1])
            st=self.state[layer];error=st['error'].numpy()
            if self.residual_bits==16:error=error.view(np.uint16)
            kernel=(joint_step30_parallel if len(layer.a)>=16 and get_num_threads()>1 else joint_step30)
            self.transitions+=kernel(layer.k.numpy(),layer.a.numpy(),layer.b.numpy(),error,
                st['res_scale'].numpy(),layer._w.grad.detach().contiguous().numpy(),v.contiguous().numpy(),
                self.lr,self.lam,self.beta,self.cap,self.residual_bits,self.gamma,self.passes,self.amax)


# The split-codebook solver uses the same reference coordinate update as Stage 27.
@njit(cache=True)
def split_step30(k,a,b,stored,scale,g,v,lr,lam,beta,code_eta,cap,bits,scale_sqrt,amax):
    m,n=k.shape
    flips=0
    for i in range(m):
        ai=float(a[i]);bi=float(b[i]);span=max(1.e-12,float(scale[i]));rms=1e-8
        for j in range(n):rms+=float(g[i,j])*float(g[i,j])
        rms=math.sqrt(rms/n)
        targets=np.empty(n,dtype=np.float64)
        indices=np.empty(n,dtype=np.uint8)
        ga=0.;gb=0.;na=0.;nb=0.
        for j in range(n):
            ko=int(k[i,j]);old=(-ai if ko==0 else -bi if ko==1 else bi if ko==2 else ai)
            err0=_residual_value(stored,i,j,bits,span)
            gn=float(g[i,j])/rms/max(1e-6,float(v[j]))
            gn=max(-cap,min(cap,gn))
            t=old+beta*err0-lr*ai*gn
            targets[j]=t
            best=1e100;bestk=ko
            for z in range(4):
                q=(-ai if z==0 else -bi if z==1 else bi if z==2 else ai)
                e=_project_residual((t-q)/(1.+lam),span,bits)
                cost=(q+e-t)**2+lam*e*e
                if cost<best:best=cost;bestk=z
            indices[j]=bestk
            if bestk==0:ga-=float(g[i,j]);na+=1.
            elif bestk==3:ga+=float(g[i,j]);na+=1.
            elif bestk==1:gb-=float(g[i,j]);nb+=1.
            else:gb+=float(g[i,j]);nb+=1.
        nn_a=max(1.,math.sqrt(na) if scale_sqrt else na)
        nn_b=max(1.,math.sqrt(nb) if scale_sqrt else nb)
        ha=rms*nn_a/(code_eta*max(.006,ai))
        hb=rms*nn_b/(code_eta*max(.006,ai))
        newa,newb=_fitter_bounds(ai-ga/ha,bi-gb/hb,ha,hb,ai,bi,amax)
        a[i]=newa;b[i]=newb
        for j in range(n):
            z=int(indices[j]);q=(-newa if z==0 else -newb if z==1 else newb if z==2 else newa)
            e=_project_residual((targets[j]-q)/(1.+lam),span,bits)
            _write_residual(stored,i,j,bits,e,span)
            if z!=int(k[i,j]):flips+=1
            k[i,j]=z
    return flips

class Stage30Split(Stage28Optim):
    def __init__(self,model,amax=2.,**kw):
        super().__init__(model,mode=0,passes=2,**kw)
        self.amax=float(amax)
    @torch.no_grad()
    def step(self):
        x=min(self.calls,self.total_steps-1)/max(1,self.total_steps-1)
        self.lr=self.base_lr*(self.floor+(1.-self.floor)*.5*(1.+math.cos(math.pi*x)))
        self.calls+=1;self.transitions=0
        for layer in self.model.direct_layers():
            if layer._w is None or layer._w.grad is None:continue
            v=activation_diag(layer,self.activation_power) if self.use_activation else torch.ones(layer.k.shape[1])
            st=self.state[layer];error=st['error'].numpy()
            if self.residual_bits==16:error=error.view(np.uint16)
            kernel=(split_step30_parallel if len(layer.a)>=16 and get_num_threads()>1 else split_step30)
            self.transitions+=kernel(layer.k.numpy(),layer.a.numpy(),layer.b.numpy(),error,
                st['res_scale'].numpy(),layer._w.grad.detach().contiguous().numpy(),v.contiguous().numpy(),
                self.lr,self.lam,self.beta,self.code_eta,self.cap,self.residual_bits,
                self.scale_mode=='sqrt',self.amax)

@njit(cache=True,parallel=True)
def joint_step30_parallel(k,a,b,stored,scale,g,v,lr,lam,beta,cap,bits,gamma,passes,amax):
    m,n=k.shape;flips=0
    change_counts=np.zeros(m,dtype=np.int64)
    for i in prange(m):
        T=np.empty(n,dtype=np.float64)
        ix=np.empty(n,dtype=np.uint8)
        er=np.empty(n,dtype=np.float64)
        ai=float(a[i]);bi=float(b[i]);span=max(1.e-12,float(scale[i]));rms=1e-8
        for j in range(n):rms+=float(g[i,j])*float(g[i,j])
        rms=math.sqrt(rms/n)
        for j in range(n):
            ko=int(k[i,j]);qold=(-ai if ko==0 else -bi if ko==1 else bi if ko==2 else ai)
            gn=float(g[i,j])/rms/max(1e-6,float(v[j]));gn=max(-cap,min(cap,gn))
            T[j]=qold+beta*_residual_value(stored,i,j,bits,span)-lr*ai*gn
        aa=ai;bb=bi
        for pp in range(passes):
            for j in range(n):
                best=1e100;bestk=0;beste=0.
                for z in range(4):
                    q=(-aa if z==0 else -bb if z==1 else bb if z==2 else aa)
                    e=_project_residual((T[j]-q)/(1.+lam),span,bits)
                    cost=(q+e-T[j])**2+lam*e*e
                    if cost<best:best=cost;bestk=z;beste=e
                ix[j]=bestk;er[j]=beste
            if pp==passes-1:break
            ha=gamma*n;hb=gamma*n;sa=gamma*n*ai;sb=gamma*n*bi
            for j in range(n):
                z=int(ix[j]);target=T[j]-er[j]; vv=max(1.e-6,float(v[j]))
                if z==0:ha+=vv;sa-=vv*target
                elif z==3:ha+=vv;sa+=vv*target
                elif z==1:hb+=vv;sb-=vv*target
                else:hb+=vv;sb+=vv*target
            aa,bb=_fitter_bounds(sa/ha,sb/hb,ha,hb,ai,bi,amax)
        a[i]=aa;b[i]=bb
        for j in range(n):
            z=int(ix[j]);_write_residual(stored,i,j,bits,er[j],span)
            if z!=int(k[i,j]):change_counts[i]+=1
            k[i,j]=z
    return int(change_counts.sum())

@njit(cache=True,parallel=True)
def split_step30_parallel(k,a,b,stored,scale,g,v,lr,lam,beta,code_eta,cap,bits,scale_sqrt,amax):
    m,n=k.shape
    flips=0
    change_counts=np.zeros(m,dtype=np.int64)
    for i in prange(m):
        ai=float(a[i]);bi=float(b[i]);span=max(1.e-12,float(scale[i]));rms=1e-8
        for j in range(n):rms+=float(g[i,j])*float(g[i,j])
        rms=math.sqrt(rms/n)
        targets=np.empty(n,dtype=np.float64)
        indices=np.empty(n,dtype=np.uint8)
        ga=0.;gb=0.;na=0.;nb=0.
        for j in range(n):
            ko=int(k[i,j]);old=(-ai if ko==0 else -bi if ko==1 else bi if ko==2 else ai)
            err0=_residual_value(stored,i,j,bits,span)
            gn=float(g[i,j])/rms/max(1e-6,float(v[j]))
            gn=max(-cap,min(cap,gn))
            t=old+beta*err0-lr*ai*gn
            targets[j]=t
            best=1e100;bestk=ko
            for z in range(4):
                q=(-ai if z==0 else -bi if z==1 else bi if z==2 else ai)
                e=_project_residual((t-q)/(1.+lam),span,bits)
                cost=(q+e-t)**2+lam*e*e
                if cost<best:best=cost;bestk=z
            indices[j]=bestk
            if bestk==0:ga-=float(g[i,j]);na+=1.
            elif bestk==3:ga+=float(g[i,j]);na+=1.
            elif bestk==1:gb-=float(g[i,j]);nb+=1.
            else:gb+=float(g[i,j]);nb+=1.
        nn_a=max(1.,math.sqrt(na) if scale_sqrt else na)
        nn_b=max(1.,math.sqrt(nb) if scale_sqrt else nb)
        ha=rms*nn_a/(code_eta*max(.006,ai))
        hb=rms*nn_b/(code_eta*max(.006,ai))
        newa,newb=_fitter_bounds(ai-ga/ha,bi-gb/hb,ha,hb,ai,bi,amax)
        a[i]=newa;b[i]=newb
        for j in range(n):
            z=int(indices[j]);q=(-newa if z==0 else -newb if z==1 else newb if z==2 else newa)
            e=_project_residual((targets[j]-q)/(1.+lam),span,bits)
            _write_residual(stored,i,j,bits,e,span)
            if z!=int(k[i,j]):change_counts[i]+=1
            k[i,j]=z
    return int(change_counts.sum())


Overwriting stage30_src/stage30_joint.py


In [ ]:
%%writefile stage30_src/stage30_suite.py
"""Stage 30 fair CPU search: log coarse LR -> fine search with 2 dev seeds -> held-out.

Two discrete candidates: coherent joint INT8 and split-codebook INT8.
All optimizers use the same legal INT2 forward codebooks within a comparison.
Dev-only hyperparameter selection, full TRAINING loss; no validation metrics.
"""
import os,math,json,time,zipfile,random,hashlib,types,itertools
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from functools import lru_cache
from advanced_models import QuantCNN
from stage30_joint import Stage30Joint,Stage30Split,joint_step30,split_step30,joint_step30_parallel,split_step30_parallel
from numba import set_num_threads,get_num_threads
import stage28_suite as s28
import stage27_suite as s27
import stage26_suite as s26
from stage27_fast import _row_update
from stage28_candidates import row_step28

METHODS=('muon','joint8','fast8')
VERSION='stage30-v1.1-lrsweep-bounds'
FIXED_BASE=dict(beta=.90,lam=.05,gamma=.01,floor=.20,cap=4.,activation_power=1.,use_activation=True)
INITIAL_TASKS={k:dict(v) for k,v in s26.TASKS.items()}
# Re-use the existing synthetic problem generators and fully trainable models.
# The MSE problem deliberately uses noisy/outlier-containing regression labels.
EXTRA={
 'mlp_mse':dict(arch='mlp',task='robust',n=960,input_dim=48,width=128,depth=3,out=6,batch=96,steps=300),
 'mlp_bce':dict(arch='mlp',task='bce',n=960,input_dim=48,width=128,depth=3,out=6,batch=96,steps=300),
 'mlp_ranking':dict(arch='mlp',task='ranking',n=960,input_dim=48,width=128,depth=3,out=1,batch=96,steps=300),
 'rnn_multiclass':dict(arch='rnn',task='multiclass',n=960,input_dim=12,width=72,depth=1,out=6,batch=64,steps=300,sequence_length=14),
 'rescnn_shapes':dict(arch='rescnn',task='multiclass',n=960,channels=1,width=16,out=4,batch=64,steps=300),
 'transformer_96_deep':dict(arch='transformer',task='nexttoken',n=768,vocab=40,width=96,depth=6,heads=4,seq=24,batch=24,steps=300),
}
s26.TASKS.update(EXTRA)
s26.old.PRESETS.update({k:v for k,v in EXTRA.items() if v['arch'] not in ('rescnn','transformer')})

class ResidualShapeCNN(QuantCNN):
    """Convolutional skip pathway; all trainable matrices are legal INT2."""
    def forward(self,x):
        x=F.relu(self.conv1(x))
        h=F.relu(self.conv2(x))
        y=F.relu(self.conv3(h))
        y=(y+F.avg_pool2d(h,2))/math.sqrt(2.)
        return self.head(y.mean((-2,-1)))

_ORIGINAL_MODEL=s26.model_for
_ORIGINAL_DATASET=s26.dataset

def extended_model_for(task,method,seed,device):
    if task=='rescnn_shapes':
        c=s26.TASKS[task]
        return ResidualShapeCNN(c['channels'],c['out'],c['width'],seed=seed,
            method='direct' if method!='muon' else 'ste',device=device).to(device)
    return _ORIGINAL_MODEL(task,method,seed,device)

@lru_cache(maxsize=40)
def extended_dataset(task,seed):
    if task=='rescnn_shapes':
        return s26.old.make_dataset('cnn_shapes',seed,n=s26.TASKS[task]['n'])
    return _ORIGINAL_DATASET(task,seed)

s26.model_for=extended_model_for
s26.dataset=extended_dataset
_ORIGINAL_LOSS=s26.loss_for

def custom_loss(model,x,y,task):
    if task=='mlp_mse':return (model(x)-y).square().mean()
    return _ORIGINAL_LOSS(model,x,y,task)
s26.loss_for=custom_loss

DEFAULT_TASKS=['mlp_mse','mlp_robust','resmlp_quantile','resmlp_poisson',
               'rnn_bce','cnn_shapes','transformer_128','transformer_192']
EXPANDED_TASKS=['mlp_mse','mlp_robust','mlp_bce','mlp_ranking','resmlp_quantile',
 'resmlp_focal','resmlp_poisson','rnn_bce','rnn_multiclass','cnn_shapes',
 'rescnn_shapes','transformer_128','transformer_192','transformer_96_deep']

MODES={
 'smoke':dict(compact_steps=3,transformer_steps=2,coarse_steps=2,fine_steps=2,coarse_lrs=2,
              fine_trials=2,muon_fine_trials=2,devseeds=[21],evalseeds=[901],evals=2,samples=32),
 'survey':dict(compact_steps=360,transformer_steps=150,coarse_steps=95,fine_steps=245,
               coarse_lrs=6,fine_trials=14,muon_fine_trials=17,devseeds=[21,31],
               evalseeds=[901,902],evals=17,samples=768),
 'balanced':dict(compact_steps=600,transformer_steps=230,coarse_steps=140,fine_steps=440,
                 coarse_lrs=6,fine_trials=17,muon_fine_trials=21,devseeds=[21,31],
                 evalseeds=[901,902,903],evals=25,samples=1200),
 'plateau':dict(compact_steps=900,transformer_steps=340,coarse_steps=220,fine_steps=680,
                coarse_lrs=7,fine_trials=20,muon_fine_trials=25,devseeds=[21,31],
                evalseeds=[901,902,903],evals=31,samples=10000000),
}

AMAX=2.
CODEBOOK_MIN_A=.006

def set_bounds(model,amax):
    # Muon STE codebooks use exactly the same feasible set as both hard-2bit arms.
    if amax<=2.:return
    @torch.no_grad()
    def widened(self):
        self.a.clamp_(min=.006,max=amax)
        self.b.clamp_(min=.002,max=amax-.004)
        self.a.copy_(torch.maximum(self.a,self.b+.004))
        self.a.clamp_(max=amax)
        self.b.copy_(torch.minimum(self.b,self.a-.004))
    for layer in model.layers:
        if hasattr(layer,'clamp_codebook'):
            layer.clamp_codebook=types.MethodType(widened,layer)

def model_and_optimizer(task,method,cfg,seed,steps):
    if method not in METHODS:raise ValueError('Unknown optimizer '+method)
    model=s26.model_for(task,'muon' if method=='muon' else 'codebook_block',seed+9000,'cpu')
    if method=='muon':
        set_bounds(model,AMAX)
        opt=s26.MuonSTE(model,cfg,'cpu')
    else:
        base=dict(s27.FIXED)
        base.update(beta=float(cfg.get('beta',FIXED_BASE['beta'])),
            lam=float(cfg.get('lam',FIXED_BASE['lam'])),
            floor=float(cfg.get('floor',FIXED_BASE['floor'])),
            activation_power=float(cfg.get('activation_power',FIXED_BASE['activation_power'])),
            cap=float(cfg.get('cap',FIXED_BASE['cap'])),
            use_activation=bool(cfg.get('use_activation',True)),
            lr=float(cfg['lr']),total_steps=int(steps),code_eta=float(cfg.get('code_eta',.02)))
        if method=='joint8':
            opt=Stage30Joint(model,residual_bits=8,amax=AMAX,
                    gamma=float(cfg.get('gamma',FIXED_BASE['gamma'])),passes=2,**base)
        else:
            opt=Stage30Split(model,residual_bits=8,amax=AMAX,**base)
    return model,opt

s27.model_and_optimizer=model_and_optimizer

def stable_key(task,method,cfg,seed,steps,phase):
    payload=dict(version=VERSION,task=task,method=method,cfg=cfg,seed=seed,steps=steps,
                 phase=phase,threads=s27.THREADS,task_config=s26.TASKS[task],amax=AMAX)
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()[:24]
s27.stable_key=stable_key

MUON_BASE={k:dict(v) for k,v in s27.LOCKED_STAGE26.items()}

def baseline_cfg(task):
    # Prior configurations are *anchors*, not the bounds of the new search.
    base=dict(MUON_BASE.get(task,MUON_BASE['mlp_robust']))
    if s26.TASKS[task]['arch']=='transformer':
        base=dict(MUON_BASE['transformer_128'])
    return base

def base_cfg(task,method):
    if method=='muon':return baseline_cfg(task)
    if method=='joint8':return dict(lr=.1,beta=.9,lam=.05,gamma=.01,floor=.2)
    return dict(lr=.1,code_eta=.02,beta=.9,lam=.05,floor=.2)

def coarse_configs(task,method,limit):
    # ~2.5 orders of magnitude, independent of the earlier locked configuration.
    grid=[.003,.01,.03,.10,.30,.70,1.30]
    if limit<=2:grid=[.03,.10]
    elif limit==6:grid=grid[:6]
    base=base_cfg(task,method)
    return [dict(base,lr=x) for x in grid]

def _config_id(cfg):return json.dumps(cfg,sort_keys=True)

def fine_configs(task,method,coarse_ranked,budget):
    # Coarse logarithmic search determines a promising LR interval; a separate
    # *structured one-factor* screen tests each remaining significant knob.
    # Not a giant factorial search, and never claims fixed knobs are insensitive.
    base=base_cfg(task,method)
    found=[];used=set()
    def add(c):
        c=dict(c);key=_config_id(c)
        if key not in used:used.add(key);found.append(c)
    top=[cfg for _,cfg in coarse_ranked[:2]]
    for c in top:add(c)
    c=top[0]
    for ratio in (.65,.82,1.2,1.5):
        x=dict(c);x['lr']=round(float(c['lr'])*ratio,8);add(x)
    # One-factor perturbations at the leading coarse LR; each item specifies
    # a *real* candidate evaluated on all refinement seeds.
    if method=='muon':
        variables=[('code_lr',[.1,3.16]),('head_lr',[.316,3.16]),
                   ('momentum',[.80,.90,.98]),
                   ('adjust_lr_fn',['original','match_rms_adamw']),
                   ('head_mode',['adam','muon']),
                   ('head_lr_factor',[.5,1.5])]
        for key,values in variables:
            for value in values:
                x=dict(c)
                x[key]=(round(max(1e-6,c[key]*value),8) if key in ('code_lr','head_lr') else value)
                add(x)
    else:
        variables=[('beta',[.75,1.]),('lam',[.005,.2]),
                   ('floor',[.05,.5]),('activation_power',[0.,.5])]
        if method=='joint8':variables.insert(2,('gamma',[.001,.1]))
        else:variables.insert(2,('code_eta',[.003,.01,.04,.10]))
        # round robin over categories, so a limited budget still covers each knob
        for round_i in range(max(len(x[1]) for x in variables)):
            for key,vals in variables:
                if round_i>=len(vals):continue
                x=dict(c);x[key]=vals[round_i];add(x)
    add(base)  # historical anchor is a check, not a search restriction
    return found[:budget]


def evaluate_tuning_trial(task,method,cfg,steps,seeds,output,phase):
    vals=[];error=''
    for seed in seeds:
        try:
            r,_=s27.train(task,method,cfg,seed,steps,output,phase,
                          eval_every=max(1,steps//9),checkpoint_every=0)
            vals.append(r)
        except (FloatingPointError,OverflowError,ValueError,RuntimeError) as exc:
            # High LR coarse trials can diverge. Preserve a failure record rather
            # than silently dropping the trial or aborting the entire sweep.
            error=f'{type(exc).__name__}: {str(exc)[:200]}'
            break
    return dict(task=task,method=method,phase=phase,cfg=_config_id(cfg),steps=steps,
        seeds=len(vals),loss_auc=(float(np.mean([v['loss_auc'] for v in vals]))
                if not error and vals else float('inf')),
        final=(float(np.mean([v['final'] for v in vals])) if not error and vals else float('inf')),
        ms_per_step=(float(np.mean([v['ms_per_step'] for v in vals])) if vals else float('nan')),
        failure=error)


def tune(task,method,mode,folder):
    spec=MODES[mode];ist=s26.TASKS[task]['arch']=='transformer'
    horizon=spec['transformer_steps'] if ist else spec['compact_steps']
    coarse_steps=min(spec['coarse_steps'],max(2,horizon))
    fine_steps=min(spec['fine_steps'],max(2,horizon))
    coarse=coarse_configs(task,method,spec['coarse_lrs'])
    coarse_rows=[]
    for cfg in coarse:
        row=evaluate_tuning_trial(task,method,cfg,coarse_steps,spec['devseeds'][:1],folder,'coarse30')
        coarse_rows.append(row)
    ranked=sorted([(x['loss_auc'],json.loads(x['cfg'])) for x in coarse_rows if np.isfinite(x['loss_auc'])],key=lambda x:x[0])
    if not ranked:raise RuntimeError(f'All coarse trials failed for {task}/{method}; inspect coarse_search.csv')
    nb=spec['muon_fine_trials'] if method=='muon' else spec['fine_trials']
    fine=fine_configs(task,method,ranked,nb)
    fine_rows=[]
    for cfg in fine:
        row=evaluate_tuning_trial(task,method,cfg,fine_steps,spec['devseeds'],folder,'fine30')
        fine_rows.append(row)
    eligible=[x for x in fine_rows if np.isfinite(x['loss_auc'])]
    if not eligible:raise RuntimeError(f'All fine trials failed for {task}/{method}; inspect fine_search.csv')
    best=min(eligible,key=lambda x:(x['loss_auc'],x['final']))
    return json.loads(best['cfg']),coarse_rows,fine_rows

def _write_csv_rows(folder,name,rows):
    path=Path(folder)/name
    incoming=pd.DataFrame(rows)
    if path.exists():
        older=pd.read_csv(path)
        incoming=pd.concat((older,incoming),ignore_index=True).drop_duplicates(
            subset=['task','method','phase','cfg','steps'],keep='last')
    incoming.to_csv(path,index=False)


def setup(mode,amax=2.,tasks=None):
    global AMAX
    if mode not in MODES:raise ValueError(mode)
    if amax<2.:raise ValueError('Codebook maximum must be >=2 for this control')
    AMAX=float(amax)
    desired=max(1,int(os.getenv('STAGE30_NUMBA_THREADS',min(2,os.cpu_count() or 1))))
    set_num_threads(desired)
    for task,cfg in list(s26.TASKS.items()):
        original=INITIAL_TASKS.get(task,EXTRA.get(task,cfg))
        cap=MODES[mode]['samples']
        limit=(max(32,min(cap,original['n'])) if mode!='plateau' else original['n'])
        s26.TASKS[task]['n']=limit
        if task in s26.old.PRESETS:
            s26.old.PRESETS[task]['n']=limit
    s26.dataset.cache_clear();s26.tokens_data.cache_clear();s26.old.make_dataset.cache_clear()
    # compile kernels in preflight, not in timed trials
    k=np.array([[0,1,2,3]],dtype=np.uint8)
    a=np.array([.6],dtype=np.float32);b=np.array([.2],dtype=np.float32)
    r=np.zeros((1,4),dtype=np.int8);g=np.array([[.1,-.2,.1,-.3]],dtype=np.float32)
    v=np.ones(4,dtype=np.float32)
    joint_step30(k.copy(),a.copy(),b.copy(),r.copy(),a.copy(),g,v,.1,.05,.9,4.,8,.01,2,AMAX)
    split_step30(k.copy(),a.copy(),b.copy(),r.copy(),a.copy(),g,v,.1,.05,.9,.02,4.,8,False,AMAX)
    if get_num_threads()>1:
        joint_step30_parallel(k.copy(),a.copy(),b.copy(),r.copy(),a.copy(),g,v,.1,.05,.9,4.,8,.01,2,AMAX)
        split_step30_parallel(k.copy(),a.copy(),b.copy(),r.copy(),a.copy(),g,v,.1,.05,.9,.02,4.,8,False,AMAX)


def run(tasks=None,methods=METHODS,mode='survey',out='stage30_cpu_results',amax=2.,do_tune=True,max_minutes=None):
    tasks=list(DEFAULT_TASKS if tasks is None else tasks)
    if set(methods)!=set(METHODS):raise ValueError('Must compare muon, joint8 and fast8 for a complete protocol')
    setup(mode,amax,tasks)
    folder=Path(out);folder.mkdir(parents=True,exist_ok=True)
    session_started=time.monotonic()
    spec=MODES[mode]; protocol=dict(version=VERSION,mode=mode,tasks=tasks,methods=list(methods),
        codebook_amax=AMAX,codebook_bmin=.002,codebook_gap=.004,
        numba_threads=get_num_threads(),dev_seeds=spec['devseeds'],heldout_seeds=spec['evalseeds'],
        budgets={k:spec[k] for k in ('coarse_steps','fine_steps','coarse_lrs','fine_trials','muon_fine_trials')},
        datasets={t:s26.TASKS[t] for t in tasks},
        note='same initial legal weights, same batches, first-order CPU, full TRAINING loss; heldout seeds never tune; wall time excludes diagnostic evaluation. Muon CPU FP32 reference.')
    proto=folder/'protocol.json'
    if proto.exists():
        old=json.loads(proto.read_text())
        if old!=protocol:raise ValueError('Protocol differs from previous run: choose a new OUT directory')
    proto.write_text(json.dumps(protocol,indent=2))
    lock=folder/'locked_configs.json'
    settings=json.loads(lock.read_text()) if lock.exists() else {}
    try:
        for task in tasks:
            if task not in s26.TASKS:raise ValueError(task)
            settings.setdefault(task,{})
            for method in methods:
                if max_minutes is not None and (time.monotonic()-session_started)/60>max_minutes:
                    note=(f'CPU session budget reached before {task}/{method}; '+
                          'all previous runs saved. Re-run the notebook with the same output directory to continue.')
                    (folder/'INCOMPLETE_SESSION.txt').write_text(note)
                    print(note,flush=True)
                    return folder
                if method not in settings[task]:
                    if do_tune:
                        config,coarse,fine=tune(task,method,mode,folder)
                        _write_csv_rows(folder,'coarse_search.csv',coarse)
                        _write_csv_rows(folder,'fine_search.csv',fine)
                    else:config=base_cfg(task,method)
                    settings[task][method]=config
                    lock.write_text(json.dumps(settings,indent=2))
                    print('LOCKED',task,method,config,flush=True)
                config=settings[task][method]
                steps=spec['transformer_steps'] if s26.TASKS[task]['arch']=='transformer' else spec['compact_steps']
                for seed in spec['evalseeds']:
                    result,_=s27.train(task,method,config,seed,steps,folder,'heldout',
                        eval_every=max(1,round(steps/spec['evals'])),checkpoint_every=max(0,steps//6))
                    print('EVAL',task,method,'seed',seed,'AUC',round(result['loss_auc'],6),
                          'final',round(result['final'],6),'ms/step',round(result['ms_per_step'],2),flush=True)
                s27.summarize(folder);s27.make_plots(folder)
    finally:
        s27.summarize(folder);s27.make_plots(folder)
    return folder

def audit_bounds(task='cnn_shapes',seed=21,steps=100,mode='survey',amax=2.):
    """Development-only check: do row levels approach codebook constraints?"""
    setup(mode,amax,[task])
    rows=[]
    for meth in METHODS:
        cfg=base_cfg(task,meth)
        model,opt=model_and_optimizer(task,meth,cfg,seed,steps)
        X,Y=s26.dataset(task,seed)
        order=torch.randint(len(X),(steps,s26.TASKS[task]['batch']),generator=torch.Generator().manual_seed(seed+9999))
        for it in range(steps):
            ids=order[it]
            if meth=='muon':opt.zero_grad()
            else:model.zero_grad(set_to_none=True)
            s26.loss_for(model,X[ids],Y[ids],task).backward();opt.step()
        a=torch.cat([l.a.detach().flatten() for l in model.layers]);b=torch.cat([l.b.detach().flatten() for l in model.layers])
        rows.append(dict(method=meth,amax=amax,max_observed_a=float(a.max()),mean_a=float(a.mean()),
            fraction_at_upper=float((a>amax-1e-5).float().mean()),
            fraction_at_lower=float((a<.006+1e-5).float().mean()),
            fraction_gap_active=float(((a-b)-.004<1e-5).float().mean())))
    return pd.DataFrame(rows)

def export_zip(folder,zip_path='Stage30_CPU_BroadSweep_Results.zip',download=True):
    return s27.export_zip(folder,zip_path=zip_path,download=download)


Overwriting stage30_src/stage30_suite.py


## Controls — edit these before Run All

`MODE='survey'` performs a meaningful CPU coarse-to-fine search and runs all selected tasks. `balanced` adds longer horizons and more evaluation seeds. `plateau` runs 900 compact / 340 transformer updates and can take substantially longer. The default 8-task menu includes both medium transformers; use `TASK_SELECTION` to select a subset when CPU time is limited. `expanded` adds BCE, focal, ranking, recurrent multiclass, a skip-connection CNN, and a deeper six-layer transformer.

**Important:** A codebook bound is a *feasible-set constraint*. A wide-bound study must re-run **Muon and both discrete methods** under exactly the same bound. `RUN_WIDE_FULL` performs that expensive, separate experiment; the default inexpensive bound audit only checks whether levels hit the bound.


In [ ]:
# EDIT THIS CELL to select experiment scope.
MODE = 'smoke' if os.environ.get('STAGE30_SMOKE')=='1' else 'balanced'  # smoke | survey | balanced | plateau
from pathlib import Path
sys.path.insert(0,str(Path('stage30_src').resolve()))
import stage30_suite as suite
TASK_SELECTION = (['mlp_mse','rnn_bce','cnn_shapes','transformer_128'] if MODE=='smoke'
                  else list(suite.DEFAULT_TASKS))
# For the full 14-task diversity screen: TASK_SELECTION = list(suite.EXPANDED_TASKS)
METHOD_SELECTION = ('muon','joint8','fast8')
CODEBOOK_AMAX = 2.0       # Current bound; use 8.0 for a controlled wider codebook re-run.
DO_TUNE = True           # Keep True for scientific comparisons.
RUN_EXPERIMENTS = True
RUN_BOUND_AUDIT = (MODE!='smoke')   # Small, development-only diagnostic, no held-out tuning
RUN_WIDE_FULL = False              # True repeats EVERY method under amax=8.0 (expensive)
AUTO_DOWNLOAD = True
MAX_MINUTES = None if MODE=='smoke' else 150  # stop at a method boundary; Run All again to resume
IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
ROOT = Path('/content') if IS_COLAB else Path('.')
OUT = ROOT / f'stage30_cpu_{MODE}_amax{CODEBOOK_AMAX:g}'
print('Mode:',MODE,'tasks:',TASK_SELECTION,'bounds:',CODEBOOK_AMAX,'output:',OUT)


Mode: plateau tasks: ['mlp_mse', 'mlp_robust', 'resmlp_quantile', 'resmlp_poisson', 'rnn_bce', 'cnn_shapes', 'transformer_128', 'transformer_192'] bounds: 2.0 output: /content/stage30_cpu_plateau_amax2


## Preflight — exact numerical parity and one-step legality

This test compiles Numba before timed experiments, compares updated indices, codebook values, and INT8 residuals against the original Stage 28 kernel, and verifies the codebook ordering constraints. It is a **correctness check**, not a claim of improved training convergence.


In [ ]:
from stage30_joint import joint_step30,split_step30,joint_step30_parallel,split_step30_parallel
from stage28_candidates import row_step28
from stage27_fast import _row_update
from numba import set_num_threads, get_num_threads
suite.setup(MODE,CODEBOOK_AMAX,TASK_SELECTION)
rng=np.random.default_rng(17)
m,n=24,64
k=rng.integers(0,4,size=(m,n),dtype=np.uint8)
a=np.full(m,.20,np.float32);b=a/3;er=np.zeros((m,n),dtype=np.int8)
scale=a.copy();g=rng.normal(0,.02,(m,n)).astype('float32');v=rng.uniform(.8,1.2,size=n).astype('float32')
args=[k,a,b,er,scale,g,v]
reference=[x.copy() for x in args];serial=[x.copy() for x in args];parallel=[x.copy() for x in args]
row_step28(*reference,.10,.05,.90,.04,4.,8,1,1.,.01,2)
joint_step30(*serial,.10,.05,.90,4.,8,.01,2,CODEBOOK_AMAX)
joint_step30_parallel(*parallel,.10,.05,.90,4.,8,.01,2,CODEBOOK_AMAX)
for orig,fast,par in zip(reference[:4],serial[:4],parallel[:4]):
    if CODEBOOK_AMAX==2.0:assert np.array_equal(orig,fast), 'Stage28 regression mismatch'
    assert np.array_equal(fast,par), 'Parallel changes numerical results'
assert np.all(serial[1]>=serial[2]+.004-1e-7) and np.all(serial[2]>=.002-1e-7)
print('Passed: integer transitions, residuals, codebook fit, and serial/parallel parity. Numba threads:',get_num_threads())


Passed: integer transitions, residuals, codebook fit, and serial/parallel parity. Numba threads: 2


## Fair logarithmic → fine-grained → held-out search

Each model/method is **independently tuned** on development seeds only. Coarse screens test roughly logarithmic learning-rate values, then fine search tests nearby learning rates and material algorithm constants. The saved `coarse_search.csv`, `fine_search.csv` and `locked_configs.json` show exactly which trials ran. Muon receives a larger fine-trial budget to avoid handicapping the reference optimizer. The reference also searches codebook/head rates, momentum, matrix scaling and head mode.

The `finally` block exports accumulated results even after errors. If a longer run is interrupted, re-running with the same `OUT` and protocol reuses completed runs and checkpoints.


In [ ]:
# MAIN EXPERIMENT — Run All proceeds through this cell.
from stage30_suite import run,export_zip
try:
    if RUN_EXPERIMENTS:
        run(TASK_SELECTION, METHOD_SELECTION, MODE, out=str(OUT), amax=CODEBOOK_AMAX, do_tune=DO_TUNE, max_minutes=MAX_MINUTES)
finally:
    if OUT.exists() and (OUT/'runs').exists() and any((OUT/'runs').glob('*.json')):
        suite.s27.summarize(OUT);suite.s27.make_plots(OUT)
        archive=ROOT / f'Stage30_CPU_FairSweep_{MODE}_amax{CODEBOOK_AMAX:g}.zip'
        export_zip(OUT,str(archive),download=AUTO_DOWNLOAD)


LOCKED mlp_mse muon {'adjust_lr_fn': 'match_rms_adamw', 'code_lr': 0.003, 'head_lr': 0.003, 'head_lr_factor': 0.5, 'head_mode': 'muon', 'lr': 0.03, 'momentum': 0.8}
EVAL mlp_mse muon seed 901 AUC 0.533616 final 0.269965 ms/step 10.93
EVAL mlp_mse muon seed 902 AUC 0.542731 final 0.263718 ms/step 11.05
EVAL mlp_mse muon seed 903 AUC 0.548103 final 0.280704 ms/step 11.15
LOCKED mlp_mse joint8 {'beta': 0.9, 'floor': 0.2, 'gamma': 0.01, 'lam': 0.05, 'lr': 0.12}
EVAL mlp_mse joint8 seed 901 AUC 0.406208 final 0.035724 ms/step 11.05
EVAL mlp_mse joint8 seed 902 AUC 0.40523 final 0.03648 ms/step 10.89
EVAL mlp_mse joint8 seed 903 AUC 0.409897 final 0.036459 ms/step 9.67
LOCKED mlp_mse fast8 {'beta': 0.9, 'code_eta': 0.04, 'floor': 0.2, 'lam': 0.05, 'lr': 0.1}
EVAL mlp_mse fast8 seed 901 AUC 0.367616 final 0.042096 ms/step 8.21
EVAL mlp_mse fast8 seed 902 AUC 0.364298 final 0.040285 ms/step 8.63
EVAL mlp_mse fast8 seed 903 AUC 0.373562 final 0.039236 ms/step 9.22
LOCKED mlp_robust muon {'adjus

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Codebook range diagnostics (optional)

Check the fraction of learned outer magnitudes at `amax` and the fraction of rows at the lower/ordering boundaries. If almost no rows approach `amax=2`, widening it to 8 is unlikely to affect training except under more aggressive hyperparameters. The audit uses development seeds **only**. Full wide-bound training requires the optional separate matched control below.


In [ ]:
if RUN_BOUND_AUDIT:
    audits=[]
    for task in ['cnn_shapes','transformer_128']:
        for bound in (2.,8.):
            df=suite.audit_bounds(task,seed=21,steps=80,mode=MODE,amax=bound)
            df['task']=task;audits.append(df)
    bounds=pd.concat(audits,ignore_index=True)
    display(bounds)
    OUT.mkdir(exist_ok=True,parents=True)
    bounds.to_csv(OUT/'codebook_bounds_development_audit.csv',index=False)
    # Put this additional file into an up-to-date ZIP.
    export_zip(OUT,str(ROOT / f'Stage30_CPU_FairSweep_{MODE}_amax{CODEBOOK_AMAX:g}.zip'),download=AUTO_DOWNLOAD)


### Optional full wide-codebook comparison

`RUN_WIDE_FULL=True` repeats **both** the broad Muon search and both discrete searches with the same bound `a_i ≤ 8`, a separate output directory, and fresh locked configurations. Do not compare a wide-bound discrete result against the narrow-bound Muon result without this control.


In [ ]:
if RUN_WIDE_FULL and MODE!='smoke':
    WIDE_OUT=ROOT/f'stage30_cpu_{MODE}_amax8'
    try:
        run(TASK_SELECTION,METHOD_SELECTION,MODE,out=str(WIDE_OUT),amax=8.,do_tune=True,max_minutes=MAX_MINUTES)
    finally:
        if WIDE_OUT.exists() and (WIDE_OUT/'runs').exists():
            suite.s27.summarize(WIDE_OUT);suite.s27.make_plots(WIDE_OUT)
            export_zip(WIDE_OUT,str(ROOT/f'Stage30_CPU_FairSweep_{MODE}_amax8.zip'),download=AUTO_DOWNLOAD)


## Read the generated results

`summary.csv` and `heldout.csv` describe full **training** AUC, final training loss, optimizer-only and full-step CPU time, memory, and plateau markers. `gates.csv` flags whether a candidate beats Muon **on both AUC and per-step time** within a task. These descriptive flags are not statistical significance tests. `all_curves.csv` and the plots show training loss versus updates and *training seconds*.

High-quality research claims require multiple untouched seeds, a sufficiently long horizon, and a repeatable advantage across tasks. A few fixed constants remain sensitive; the one-factor search makes them visible rather than calling them mathematically eliminated.


In [ ]:
from IPython.display import display
if (OUT/'summary.csv').exists():
    summary=pd.read_csv(OUT/'summary.csv')
    display(summary[['task','method','loss_auc','final','ms_per_step','ms_optimizer','optimizer_aux_bytes','persistent_total_bytes']].sort_values(['task','loss_auc']))
if (OUT/'gates.csv').exists():display(pd.read_csv(OUT/'gates.csv'))
if (OUT/'fine_search.csv').exists():
    fine=pd.read_csv(OUT/'fine_search.csv')
    print('Fine-trial counts:')
    display(fine.groupby(['task','method']).size().rename('trials').reset_index())
print('All files in:',OUT)


## Known scope and reproducibility caveats

- Both arms always use **exactly four legal codewords per row in the forward pass**. Muon's FP32 shadow/momentum states are distinct from a temporary realized FP32 tensor used for discrete autograd.
- Persistent state is logically **2-bit weight indices + 8-bit residual per weight** for discrete candidates, plus row codebooks/scales; physical indices are currently `uint8`, so actual peak memory is larger. The residual is true `torch.int8` and incurs `O(P)` optimizer memory.
- Activation RMS is approximate for convolutions and falls back to ones for incompatible recurrent inputs. Current small synthetic models and datasets are not a substitute for real-data scaling.
- The finer search explores parameter axes but not every interaction. Muon's CPU Newton–Schulz path is a FP32 reference, not a validation of CUDA efficiency.
- Changing bounds changes feasible sets; it must be matched across optimizers. In Stage 30's 100-step CNN development check no method approached `a=2`, so wider bounds are exploratory rather than an evidence-backed fix.
